# Suite DIA — The Python3, TypeScript5, MATLAB, Excel, LaTeX, C, Cpp and SystemVerilog dialects

Each dialect models a language's expressions: its kinds, its vocabulary and the domains of its values, `render`
to its source text, an evaluator with the language's own rules, and a scope that resolves names, imports and
references as the language does. Python evaluates by Python's rules, with NumPy as one more module to import;
MATLAB is two-valued over doubles, logicals and strings, with functions from the path and packages; Excel coerces,
computes with error values, and reads cells from workbooks. LaTeX writes notation, and has no evaluator. C evaluates by C's
rules over its arithmetic types (and Cpp, which shares its kinds, by C++'s), and SystemVerilog by IEEE 1800's over
4-state vectors and reals. Each dialect's install adds its union of its kinds, named plainly, to the framework's
`Term`, as the branch named after it, and its terms are written in a store through its writers; it reads only its own
terms. What every dialect shares is suite FRM; the Patterns dialect is suites EXP and EVL.

In [ ]:
import math

import numpy as np

from mbse.Expressions import Framework
from mbse.Expressions.IEEE1164 import Expressions as IEEE1164
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Unicode import Expressions as Unicode
from mbse.Expressions.C import Expressions as C
from mbse.Expressions.Cpp import Expressions as Cpp
from mbse.Expressions.Excel import Expressions as Excel
from mbse.Expressions.LaTeX import Expressions as Latex
from mbse.Expressions.MATLAB import Expressions as Matlab
from mbse.Expressions.Patterns import Expressions as Patterns
from mbse.Expressions.Python import Python3
from mbse.Expressions.Arm import AAPCS64
from mbse.Expressions.SystemV import AMD64
from mbse.Expressions.SystemVerilog import Assertions as SVAssertions, Constraints as SVConstraints, \
    Expressions as SystemVerilog, Patterns as SVPatterns, Preprocessor as SVPreprocessor, Types as SVTypes
from mbse.Expressions.TypeScript import TypeScript5
from mbse.Expressions.C import Preprocessor as CPreprocessor, Types as CTypes  # the retrofit's new dialects
from mbse.Expressions.Cpp import RegExp as CppRegExp, Types as CppTypes
from mbse.Expressions.Python import Patterns as PythonPatterns, RegExp as PythonRegExp, Types as PythonTypes
from mbse.Expressions.TypeScript import Patterns as TypeScriptPatterns, RegExp as TypeScriptRegExp, Types as TypeScriptTypes
from mbse.Expressions.C.Expressions import Domains as CD, Evaluators as CV, Expressions as C, Text as CText
from mbse.Expressions.Excel.Expressions import Domains as XD, Evaluators as XV, Expressions as X, Text as XText
from mbse.Expressions.LaTeX.Expressions import Domains as LD, Expressions as L, Text as LText
from mbse.Expressions.MATLAB.Expressions import Domains as MD, Evaluators as MV, Expressions as M, Text as MText
from mbse.Expressions.Python.Python3 import Domains as PD, Evaluators as PV, Expressions as P, Text as PText
from mbse.Expressions.SystemVerilog.Expressions import Domains as SVD, Evaluators as SVV, Expressions as SV, Text as SVText
from mbse.Expressions.SystemVerilog.Types import Expressions as SVT
from mbse.Expressions.TypeScript.TypeScript5 import Evaluators as TV, Expressions as TS, Text as TSText
from mbse.Expressions.TypeScript.Patterns import Expressions as TSP
from mbse.Expressions.TypeScript.Types import Expressions as TST
from mbse.Expressions.Framework import Domains as D, Errors as FE, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import Builders, JSON, YAML
from support import raises

store = Builders.Store()  # every dialect's kinds, in one store
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, IEEE1164.install, Unicode.install,  # the standard
                Patterns.install, Python3.install, Matlab.install, Excel.install, Latex.install,  # dialects first
                C.install, Cpp.install, AMD64.install, AAPCS64.install, SystemVerilog.install, SVTypes.install,
                SVPatterns.install, SVAssertions.install, SVConstraints.install, SVPreprocessor.install, TypeScript5.install,
                PythonTypes.install, PythonPatterns.install, TypeScriptTypes.install, TypeScriptPatterns.install,  # each
                CTypes.install, CppTypes.install, CPreprocessor.install,  # after its family's dialect
                PythonRegExp.install, TypeScriptRegExp.install, CppRegExp.install]:  # the regular expressions last
    store.transaction(install)
store.transaction(Patterns.define)  # Patterns' operations, which its validation reads
for define in [C.define, Cpp.define, AMD64.define, AAPCS64.define]:  # C's, C++'s and the ABIs' types, which their
    store.transaction(define)  # evaluation, inference and validation read through a term's holder
for define in [CTypes.define, CppTypes.define, CPreprocessor.define]:  # their families' new dialects' data, none yet
    store.transaction(define)
store.transaction(SystemVerilog.define)  # SystemVerilog's types, which its evaluation, inference and validation read
Term = store.registered("Expressions.Framework.Term")
AMD64_, AAPCS64_ = store.Instances.Expressions.SystemV.AMD64, store.Instances.Expressions.Arm.AAPCS64  # the ABIs


def term(spec, dialect, abi=AMD64_):
    """The term value `spec` writes, a native value as `dialect`'s literal, the root of a holder written in a transaction;
    a C or C++ term's holder names the ABI its terms are read under, `abi` (x86-64's System V ABI unless given; None,
    none)."""
    with store.transaction():
        if dialect.tag in ("c", "cpp") and abi is not None:
            return F.term(store.Expressions.Framework.Expression(lambda e: e.root(dialect.spec(spec)).abi([abi])))
        return F.term(store.Expressions.Framework.Expression(lambda e: e.root(dialect.spec(spec))))


def round_trips(value):
    """Whether a term reads back the same from JSON and from YAML."""
    return all(F.same(decode(Term, encode(Term, value)), value) for encode, decode in [
        (JSON.ToJSON(store), JSON.FromJSON(store)), (YAML.ToYAML(store), YAML.FromYAML(store))])


def data(txn):
    """The data the expressions read: people, and objects with lists."""
    S = txn.Schemas
    ints = lambda: S.OfContainer(lambda c: c.index([S.OfInt]).element([S.OfInt]))  # noqa: E731
    return txn.uid("dia.data/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Person": S.OfObject, "Squad": S.OfObject, "Board": S.OfObject, "Bus": S.OfObject})
        .spec([
            lambda _: _.Person.name("Person").ref(True).properties({"name": S.OfStr, "age": S.OfInt, "email": S.OfStr}),
            lambda _: _.Squad.name("Squad").ref(True).properties({"ranks": ints()}),
            lambda _: _.Board.name("Board").ref(True).properties({"name": S.OfStr, "grid": S.OfContainer(
                lambda c: c.index([S.OfInt]).element([S.OfContainer(lambda d: d.index([S.OfInt]).element([S.OfInt]))]))}),
            lambda _: _.Bus.name("Bus").ref(True).properties({"name": S.OfStr, "ports": ints()}),
        ])]))])


store.transaction(data)
with store.transaction():
    ann = store.Person(lambda p: p.name("Ann").age(30).email("ann@example.com"))
    bob = store.Person(lambda p: p.name("Bob").age(70))  # no email
    squad = store.Squad(lambda s: s.ranks([1, 2, 3]))  # a list property is a list of its values
    board = store.Board(lambda b: b.name("b").grid([lambda e: e.as_container([lambda x: x.as_scalar(1), lambda x: x.as_scalar(2)]),
                                                    lambda e: e.as_container([lambda x: x.as_scalar(3)])]))
    bus = store.Bus(lambda b: b.name("b").ports([4, 5]))

<a id="DIA-01"></a>
## DIA-01 · Python: expressions as `ast` has them, with their imports; rendered and parsed as source

In [ ]:
def py(spec):
    return term(spec, P.DIALECT)


def text(spec):
    return PText.ToText(py(spec))


def parsed(source):
    return py(PText.FromText(source))


def bare(node):
    """A term as the writers write it: its literals without the spelling `FromText` keeps."""
    def rebuild(node):
        form = F.form(node)
        kind = P.DIALECT.kinds()[form.kind]
        attributes = {k: v for k, v in form.attributes.items() if kind.role != F.LITERAL or k not in kind.spelling}
        return P.DIALECT.make(F.Form(form.kind, attributes, tuple(None if a is None else rebuild(a) for a in form.arguments)))

    return py(rebuild(node))


t, a, b, c, d, x, y, z = (P.name(n) for n in "tabcdxyz")
for built, source in [  # written with the writers, as source
    (P.import_("numpy", P.importfrom("numpy.ma", "getmaskarray", P.call(
        "np.logical_and", P.call("np.greater_equal", P.subscript(t, "age"), 18),
        P.call("np.logical_not", P.call("getmaskarray", P.subscript(t, "email"))))), alias="np"),
     "import numpy as np\nfrom numpy.ma import getmaskarray\n"
     "np.logical_and(np.greater_equal(t['age'], 18), np.logical_not(getmaskarray(t['email'])))"),
    (P.let_("a", t, P.boolop("and", P.compare(">", P.attribute(a, "b"), 1), P.unaryop("not", P.attribute(a, "c")))),
     "(lambda a: a.b > 1 and not a.c)(t)"),
    (P.ifexp(P.compare("<", a, b), x, P.unaryop("-", P.binop("**", y, 2))), "x if a < b else -y ** 2"),
    (P.ifexp(b, P.ifexp(a, x, y), z), "(x if a else y) if b else z"),
    (P.ifexp(a, x, P.ifexp(b, y, z)), "x if a else y if b else z"),
    (P.binop("-", P.binop("-", P.binop("**", 2, P.unaryop("-", 1)), P.binop("-", a, b)), c), "2 ** -1 - (a - b) - c"),
    (P.boolop("or", P.boolop("and", a, b), P.boolop("and", c, P.unaryop("not", d))), "a and b or c and not d"),
    (P.boolop("and", P.boolop("or", a, b), P.boolop("or", c, d)), "(a or b) and (c or d)"),
    (P.boolop("and", a, b, c), "a and b and c"), (P.boolop("and", P.boolop("and", a, b), c), "(a and b) and c"),  # n values
    (P.compare("!=", P.unaryop("not", P.compare("==", a, b)), P.compare("<=", c, d)), "(not a == b) != (c <= d)"),
    (P.compare("<", a, b, ("<=", c), ("is not", d)), "a < b <= c is not d"),  # a chain
    (P.call(P.call("f", P.name("g")), P.subscript(P.attribute(x, "y"), "z"), P.unaryop("+", 1), b"\x00", True, "it's"),
     "f(g)(x.y['z'], +1, b'\\x00', True, \"it's\")"),
    (P.call("f", a, P.starred(b), P.keyword("k", 1), P.keyword(None, c)), "f(a, *b, k=1, **c)"),  # keywords, unpacked
    (P.import_("a", P.import_("b", P.importfrom("d", "e", P.importfrom("d", "f", 1, alias="g")), alias="c")),
     "import a\nimport b as c\nfrom d import e\nfrom d import f as g\n1"),
    (P.call("all", P.generatorexp(P.compare(">", P.attribute(P.name("p"), "pin"), 0), P.comprehension("p", P.attribute(t, "ports")))),
     "all(p.pin > 0 for p in t.ports)"),
    (P.call("sum", P.generatorexp(1, P.comprehension("p", x, (P.compare("in", P.name("p"), y), P.compare("not in", P.name("p"), z))))),
     "sum(1 for p in x if p in y if p not in z)"),
    (P.binop("+", P.subscript(x, P.binop("-", a, 1)), P.call("len", P.call("set", x))), "x[a - 1] + len(set(x))"),
    (P.call("max", P.generatorexp(P.ifexp(b, P.name("v"), z), P.comprehension("v", P.ifexp(a, x, y), (P.boolop("or", a, b),))), 1),
     "max((v if b else z for v in (x if a else y) if a or b), 1)"),
    (P.subscript(t, P.tuple_(P.slice_(1, None, 2), P.slice_(), P.constant(...), P.constant(None), P.starred(x))),
     "t[1::2, :, ..., None, *x]"),  # a tensor's indices: slices, ..., a new axis, unpacked
    # every other kind: displays, comprehensions, lambdas, named expressions, await and yield, `@`, constants
    (P.listcomp(x, P.comprehension("x", y, (x,)), P.comprehension("z", x)), "[x for x in y if x for z in x]"),
    (P.dictcomp(P.name("k"), P.name("v"), P.comprehension(P.tuple_(P.name("k"), P.name("v")), P.call(P.attribute(d, "items")))),
     "{k: v for k, v in d.items()}"),
    (P.setcomp(x, P.comprehension("x", y)), "{x for x in y}"), (P.listcomp(x, P.comprehension("x", y, (), True)), "[x async for x in y]"),
    (P.set_(1, P.starred(a)), "{1, *a}"),
    (P.dict_((None, a), ("k", 1)), "{**a, 'k': 1}"), (P.tuple_(), "()"), (P.tuple_(1), "(1,)"), (P.tuple_(1, 2), "(1, 2)"),
    (P.subscript(x, P.tuple_()), "x[()]"), (P.subscript(x, P.tuple_(1)), "x[1,]"), (P.subscript(x, P.slice_()), "x[:]"),
    (P.lambda_(0), "lambda: 0"), (P.lambda_(P.name("k"), kwonlyargs=("k",)), "lambda *, k: k"),
    (P.lambda_(a, ("b", P.arg("c", 1)), posonlyargs=("a",), vararg="d", kwonlyargs=("e", P.arg("f", 2)), kwarg="g"),
     "lambda a, /, b, c=1, *d, e, f=2, **g: a"),
    (P.namedexpr("y", 10), "(y := 10)"), (P.await_(x), "await x"), (P.yield_(), "(yield)"), (P.yield_(x), "(yield x)"),
    (P.yieldfrom(x), "(yield from x)"), (P.binop("@", a, b), "a @ b"), (P.compare("is", a, b), "a is b"),
    (P.unaryop("-", P.binop("**", x, 2)), "-x ** 2"), (P.attribute(1, "real"), "(1).real"),
    (P.constant(2j), "2j"), (P.constant(None), "None"), (P.constant(...), "..."), (P.concatenatedstring("a", "b"), "'a' 'b'"),
    (P.call("f", P.generatorexp(x, P.comprehension("x", y))), "f(x for x in y)"),
    (P.call("f", P.generatorexp(x, P.comprehension("x", y)), 1), "f((x for x in y), 1)"),
]:
    written = py(built)
    assert PText.ToText(written) == source, (PText.ToText(written), source)
    assert F.same(bare(parsed(source)), written) and round_trips(written), source  # Python reads it back (parse is Python's only)
for built, source in [  # printed as written, what a parser keeps beside its terms: an f-string's text and fields, parentheses
    (P.joinedstr(("a{{b}}", P.formattedvalue(x, "r", (">", P.formattedvalue(P.name("w")))), " ", P.formattedvalue(y, debug=True))),
     "f'a{{b}}{x!r:>{w}} {y=}'"),
    (P.templatestr(("x", P.interpolation(y), "z")), "t'x{y}z'"), (P.joinedstr((P.formattedvalue(P.set_(1)),), "rf", '"'), 'rf"{ {1}}"'),
    (P.concatenatedstring(P.constant("a", "'a'"), P.constant("b", '"b"')), "'a' \"b\""),
    (P.parenthesized(P.binop("+", a, b)), "(a + b)"), (P.binop("*", P.parenthesized(a), b), "(a) * b"),
    (P.dictcomp(x, None, P.comprehension("x", y)), "{**x for x in y}"),  # Python 3.15's, which 3.14 does not parse
    (P.listcomp(P.starred(x), P.comprehension("x", y)), "[*x for x in y]"),
]:
    assert PText.ToText(py(built)) == source and round_trips(py(built)), (PText.ToText(py(built)), source)
for source in [
    "import numpy as np\nfrom numpy.ma import getmaskarray\nnp.logical_and(np.greater_equal(t['age'], 18), "
    "np.logical_not(getmaskarray(t['email'])))",
    "import numpy.ma\nnumpy.ma.masked",
    "(lambda a: a.b > 1 and not a.c)(t)",
    "x if a < b else -y ** 2",
    "(x if a else y) if b else z",
    "(-1) ** 2 + 3 * (4 - 5) // 2 % 7 / 1.5",
    "2 ** -1 - (a - b) - c",
    "a and b or c and not d",
    "(a or b) and (c or d)",
    "not (a == b) != (c <= d)",
    "f(g)(x.y['z'], +1, b'\\x00', True, 'it''s')",
]:
    read = parsed(source)
    assert PText.ToText(read) == PText.ToText(parsed(PText.ToText(read))), source  # render and parse are inverses
    assert F.same(parsed(PText.ToText(read)), read), source
# Every kind of Python 3's expressions, printed as Python writes it and read back by Python's own parser, each literal
# keeping its spelling: what the narrower kinds could not hold (keywords, slices, several `for` clauses, `None`, `@`,
# `is`, sets, a dict's `**`) among them.
for source in [
    "a < b <= c", "a and b and c", "f(x, *y, k=1, **z)", "f(x=1)", "(lambda a, b: a)(1, 2)", "x[0:1]", "x[a:b]", "x[:]",
    "x[a, b:c]", "x[()]", "x[1,]", "(a for b in c for d in e)", "(a for b, c in d)", "[x for x in y if x for z in x]",
    "{k: v for k, v in d.items()}", "{x for x in y}", "[x async for x in y]", "None", "...", "1_0j", "a @ b", "a is b",
    "a is not b", "{1}", "{*a}", "{**a}", "{**a, 'k': 1}", "{}", "()", "(1,)", "(1, 2)", "[*a, b]", "[]", "a.b.c", "(1).real",
    "f(x for x in y)", "f((x for x in y), 1)", "lambda: 0", "lambda a, /, b=1, *c, d, e=2, **f: a", "lambda *, k: k",
    "lambda x=lambda: 1: x", "(y := 10)", "await x", "(yield)", "(yield x)", "(yield from x)", "-x ** 2", "(-x) ** 2",
    "not a", "x if a else y if b else z", "f'a{{b}}{x!r:>{w}} \\n {y = }'", "f'{x:{y}.{z}}'", "f'{\"a\"}'", "f'{x}' 'y'",
    "t'x{y}z'", "(lambda a=1: a)(2)", "f(*a)(b)", "'a' \"b\"", "rb'\\d' b'x'", "u'x'", "R'x'", "'''tri'''", "1_000_000", "0x_FF", "0o17", "0b1", "1e3", "True",
]:
    read = parsed(source)
    assert PText.ToText(read) == source and F.same(parsed(PText.ToText(read)), read) and round_trips(read), source
assert PText.ToText(parsed("import numpy.ma\nnumpy.ma.masked")) == "import numpy.ma\nnumpy.ma.masked"
assert PText.ToText(parsed("x if a else y if b else z")) == "x if a else y if b else z"
assert PText.ToText(parsed("(x if a else y) if b else z")) == "(x if a else y) if b else z"
assert PText.ToText(parsed("import a, b as c\nfrom d import e, f as g\n1")) == (
    "import a\nimport b as c\nfrom d import e\nfrom d import f as g\n1")
assert PText.ToText(parsed("(a for (b, c) in d)")) == "(a for b, c in d)"  # a tuple target, without its parentheses
assert PText.ToText(parsed("(a)")) == "a" and PText.ToText(parsed("x[(1, 2)]")) == "x[1, 2]"  # `ast` keeps no parentheses
assert [text(P.constant(v)) for v in [math.nan, math.inf, -math.inf, -2]] == ["float('nan')", "float('inf')", "-float('inf')", "-2"]
assert [text(P.constant(v)) for v in [None, ..., 2j, complex(1, -2), complex(1, 2.5), complex(0, -2.5)]] == [
    "None", "...", "2j", "(1-2j)", "(1+2.5j)", "-2.5j"]  # Python3's own
assert text(P.binop("**", P.constant(complex(0, -2.5)), 2)) == "(-2.5j) ** 2"  # its precedence, as its spelling's
assert text(P.binop("**", -2, 2)) == "(-2) ** 2" and text(P.unaryop("-", P.binop("**", 2, 2))) == "-2 ** 2"
assert text(P.call("np.ma.getmaskarray", P.name("x"))) == "np.ma.getmaskarray(x)"  # a dotted function
assert text(P.parenthesized(P.binop("+", a, b))) == "(a + b)" and text(P.binop("*", P.parenthesized(a), b)) == "(a) * b"
assert text(P.slice_(a, b)) == "a:b" and text(P.tuple_(a)) == "(a,)" and text(P.yield_(a)) == "(yield a)"  # alone
assert text(P.call("f", P.namedexpr("y", 1))) == "f((y := 1))" and text(P.starred(P.binop("+", a, b))) == "*a + b"
assert text(P.starred(P.compare("<", a, b))) == "*(a < b)"  # unpacked: a bitwise or, or what binds tighter
built = py(P.importfrom("m", "f", P.let_("a", P.subscript(P.name("t"), "k"), P.ifexp(
    P.boolop("and", P.compare("<", P.name("a"), 1), P.unaryop("not", P.attribute(P.name("a"), "x"))),
    P.binop("+", P.call("f", P.name("a")), 2.5), P.constant(b""))), alias="g"))
assert F.same(built, bare(parsed("from m import f as g\n(lambda a: f(a) + 2.5 if a < 1 and not a.x else b'')(t['k'])")))
assert text(P.attribute(P.binop("+", 1, 2), "real")) == "(1 + 2).real"
counted = py(P.generatorexp(P.name("p"), P.comprehension("p", P.name("ps"), (P.name("q"),))))  # p is bound in its clause and the element
assert SY.free(counted, P.DIALECT) == {"ps", "q"} and P.DIALECT.validate(counted, {"ps", "q"}) == []
assert P.DIALECT.validate(py(P.generatorexp(1, P.comprehension("1p", P.name("ps")))), {"ps"}) == [
    "generators 0 target: a name must be an identifier, got '1p'"]
assert P.DIALECT.validate(py(P.generatorexp(1, P.comprehension("p", P.name("p"))))) == ["generators 0 iter: variable 'p' is not bound"]
chain = parsed("a < b <= c")  # one comparison chain: b written once, the operators of its links
assert PText.ToText(chain) == "a < b <= c" and [F.name_of(operand) for operand in F.form(chain).arguments] == ["a", "b", "c"]
assert [part.attributes["op"] for part in F.form(chain).attributes["comparisons"]] == ["<", "<="]
with raises(ValueError, match="an import can only enclose the whole expression"):
    text(P.binop("+", 1, P.import_("math", 2)))
for source, message in [
    ("x = 1\nx", "cannot parse 'x = 1': only imports may precede the expression"),
    ("from . import a\na", "cannot parse 'from . import a': only imports may precede the expression"),
    ("from m import *\na", "cannot parse 'from m import *': import * binds names that cannot be known"),
]:
    with raises(ValueError, match=message):
        PText.FromText(source)
for source in ["", "import math"]:
    with raises(ValueError, match="the source must end with an expression"):
        PText.FromText(source)
# Displays: a list display (the framework's container constructor), and a dict display, evaluated as Python's.
assert text(P.list_(1, x)) == "[1, x]" and text(P.dict_(("a", 1), (x, P.list_()))) == "{'a': 1, x: []}" and text(P.dict_()) == "{}"
assert F.same(bare(parsed("[1, {'a': [x]}]")), py(P.list_(1, P.dict_(("a", P.list_(x))))))
assert PV.OfAny(parsed("{'a': 1, 'a': 2, 'b': [x]}"), {"x": 3}) == {"a": 2, "b": [3]}  # a later key replaces an earlier's
kinds = P.DIALECT.kinds()  # an import binds the names its kind reads from its attributes
assert kinds[P.IMPORT].names(F.form(py(P.import_("numpy.ma", 1))).attributes) == ("numpy",)
assert kinds[P.IMPORTFROM].names(F.form(py(P.importfrom("m", "n", 1, alias="k"))).attributes) == ("k",)
assert [F.form(i).attributes for i in SY.imports(parsed("import numpy as np\nfrom m import n\n1"), P.DIALECT)] == [
    {"module": "numpy", "alias": "np"}, {"module": "m", "name": "n"}]
for spec, problems in [
    (P.import_("not a module", 1), ["an import's module must be a dotted name, got 'not a module'"]),
    (P.import_("m", 1, alias="a b"), ["an import's alias must be an identifier, got 'a b'"]),
    (P.importfrom("m.", "x y", 1), ["an importfrom's module must be a dotted name, got 'm.'",
                                    "an importfrom's name must be an identifier, got 'x y'"]),
    (P.importfrom("m", "n", 1, alias=""), ["an importfrom needs an alias"]),  # empty, not absent
    (P.let_("a b", 1, 2), ["a let's name must be an identifier, got 'a b'"]),
    (P.importfrom("m", "n", P.name("n")), []),
    (lambda u: getattr(u, P.IMPORT)(lambda r: r.module(7).body(P.constant(1))), ["an import's module must be a str, got int"]),
    (lambda u: getattr(u, P.IMPORT)(lambda r: r.module("numpy")), ["an import needs a body"]),
    (lambda u: getattr(u, P.LITERAL)(lambda r: r), ["a literal needs a value"]),
    (lambda u: getattr(u, P.COMPARE)(lambda r: r.left(P.constant(1))), ["a compare needs a comparison"]),
]:
    assert P.DIALECT.validate(py(spec)) == problems, (spec, P.DIALECT.validate(py(spec)))
assert P.DIALECT.validate(py(P.attribute(P.name("x"), "1")), {"x"}) == ["an attribute must be an identifier, got '1'"]
assert P.DIALECT.validate(py(P.binop("@", 1, 2)), core=True) == [] and P.DIALECT.validate(py(P.compare("is", 1, 2)), core=True) == []
assert P.DIALECT.validate(py(P.binop("@@", 1, 2)), core=True) == ["'@@' is not a core operation"]
assert P.DIALECT.validate(py(P.compare("<", 1, 2, ("===", 3))), core=True) == ["'===' is not a core operation"]
assert P.DIALECT.validate(py(P.compare("<", 1, 2, ("===", 3)))) == []  # an extension, unless the core is asked for
assert P.DIALECT.validate(py(P.binop("+", 1, 2)), core=True) == []
assert P.DIALECT.validate(py(P.call("hasattr", P.name("x"), "k")), {"x"}) == []  # builtins are ambient


def infer(source, environment=None):
    return P.DIALECT.infer(parsed(source), environment)


assert infer("1 + 2.5") is PD.Float and infer("True + 1") is PD.Int and infer("'a' + 'b'") is PD.Str and infer("-2.0") is PD.Float
assert infer("2 ** 3").name() == "int | float" and infer("not 1") is PD.Bool and infer("1 < 2.5") is PD.Bool
assert infer("x.a", {"x": D.Anything}) is D.Anything and infer("1 and 'a'").name() == "int | str"  # Python's and gives an operand
assert infer("(lambda a: a * 2.5)(1)") is PD.Float and infer("f(1)", {"f": D.Anything}) is D.Anything
assert infer("(p for p in v)", {"v": D.Anything}) is D.Anything and infer("1 < 2 <= 3.5") is PD.Bool  # a chain
assert infer("1 and 'a' and 2.5").name() == "int | str | float"
assert P.DIALECT.infer(py(P.compare("<", 1, 2, ("===", 3)))) is D.Anything  # an extension's operator: not known
assert infer("None") is PD.NoneType and infer("...") is PD.Ellipsis and infer("2j") is PD.Complex  # Python3's constants
assert infer("f'{x}'", {"x": D.Anything}) is PD.Str and infer("x[1:]", {"x": D.Anything}) is D.Anything
with raises(TypeError, match="< cannot take (str, int)"):
    infer("'a' < 1")
with raises(TypeError, match="<= cannot take (int, str)"):
    infer("1 < 2 <= 'a'")
assert PD.of(b"") is PD.Bytes and PD.Numeric.includes(PD.Bool)
with raises(TypeError, match="expected a term, a writer or a native value, got [1]"):
    P.DIALECT.spec([1])  # no Python3 constant holds a list
with raises(ValueError, match="a Patterns call is not a Python3 term"):  # a term of another dialect: Python3 reads its own
    PV.OfAny(term(Patterns.Expressions.operation("add", 1, 2), Patterns.Expressions.DIALECT))

<a id="DIA-02"></a>
## DIA-02 · Python evaluates by Python's rules, in a scope that allowlists what an expression may reach

In [ ]:
def ev(source, variables=None, **scope):
    return PV.OfAny(parsed(source), PV.Scope(variables, **scope))


assert ev("0 or 'x'") == "x" and ev("'' and 1") == "" and ev("1 and 2") == 2  # an operand, not a bool
assert ev("1 == 1.0") is True and ev("True + 1") == 2 and ev("7 // 2 % 3 - 2 ** 2 / 4") == -1.0
assert ev("a if a > 1 else missing", {"a": 2}) == 2 and ev("(lambda a: a * a)(3)") == 9
assert ev("t.age", {"t": ann}) == 30 and ev("hasattr(t, 'email')", {"t": bob}) is False  # properties are attributes
assert ev("getattr(t, 'name')", {"t": ann}) == "Ann" and ev("t['x']", {"t": {"x": 1}}) == 1
assert ev("hasattr(t, '_secret')", {"t": ann}) is False and ev("hasattr(t, 'email')", {"t": ann}) is True
assert ev("abs(-2) + len('abc') + max(1, 2)") == 7 and ev("f(2)", {"f": lambda x: x + 1}) == 3
with raises(AttributeError, match="'Person' object has no attribute 'email'"):
    ev("t.email", {"t": bob})
with raises(AttributeError, match="attribute '__class__' is private"):
    ev("t.__class__", {"t": ann})
with raises(NameError, match="name 'x' is not defined"):
    ev("x")
with raises(NameError, match="name 'len' is not defined"):
    ev("len('a')", builtins={})


def shout(text):
    return text.upper()


with raises(TypeError, match="calling 'shout' is not allowed by the scope"):
    ev("t['f']('a')", {"t": {"f": shout}})  # only what the scope provides, not what values hold
with raises(TypeError, match="calling 'int' is not allowed by the scope"):
    ev("t()", {"t": 3})

numpy = {"numpy": np}
columns = {"age": np.array([30, 10, 70]), "email": np.ma.array(["a", "b", "c"], mask=[False, False, True])}
constraint = ("import numpy as np\nfrom numpy.ma import getmaskarray\n"
              "np.logical_and(np.greater_equal(t['age'], 18), np.logical_not(getmaskarray(t['email'])))")
assert ev(constraint, {"t": columns}, modules=numpy).tolist() == [True, False, False]  # vectorized, with masks
assert ev("import numpy.ma\nnumpy.ma.masked", modules=numpy) is np.ma.masked  # binds numpy
assert ev("import numpy.ma as ma\nma.masked", modules=numpy) is np.ma.masked and ev("import numpy\nnumpy.pi",
                                                                                    modules=numpy) == math.pi
with raises(ImportError, match="import of 'numpy' is not allowed by the scope"):
    ev(constraint, {"t": columns})  # nothing is imported unless the scope allows it
with raises(ImportError, match="import of 'os' is not allowed by the scope"):
    ev("import os\nos.getcwd()", modules=numpy)
with raises(ImportError, match="No module named 'numpy.nope'"):
    ev("import numpy.nope\n1", modules=numpy)
with raises(ImportError, match="No module named 'numpy._core'"):
    ev("import numpy._core\n1", modules=numpy)
with raises(ImportError, match="cannot import name 'nope' from 'numpy'"):
    ev("from numpy import nope\n1", modules=numpy)
with raises(ImportError, match="cannot import name '_core' from 'numpy'"):
    ev("from numpy import _core\n1", modules=numpy)
assert PV.OfAny(parsed("x + 1"), {"x": 1}) == 2  # a mapping is the variables of a scope
with store.transaction():  # an expression, its root written by a writer
    adult = store.Expressions.Framework.Expression(lambda e: e.root(P.compare(">=", P.attribute(P.name("t"), "age"), 18)))
assert PV.OfAny(adult, {"t": ann}) is True and PText.ToText(adult) == "t.age >= 18"

# Python's rules, case by case. Here they run against Python itself; the TypeScript suite runs the same table against
# its model of them.
import math as math_module

n, NAN, INF = P.name, float("nan"), float("inf")
modules = {"math": math_module, "numpy": np}
VALUES = [  # (expression, variables, value): the value and its type
    (P.unaryop("not", n("v")), {"v": None}, True), (P.unaryop("not", 0.0), {}, True), (P.unaryop("not", ""), {}, True),
    (P.unaryop("not", b""), {}, True), (P.unaryop("not", n("v")), {"v": []}, True), (P.unaryop("not", n("v")), {"v": {}}, True),
    (P.unaryop("not", NAN), {}, False), (P.unaryop("not", n("v")), {"v": {"k": 1}}, False),
    (P.compare("<", 1, 1.5), {}, True), (P.compare(">", 2, 1.5), {}, True), (P.compare("==", 1, 1.5), {}, False),
    (P.compare("==", P.binop("+", P.binop("**", 2, 60), 1), P.call("float", P.binop("**", 2, 60))), {}, False),
    (P.compare("<", 1, INF), {}, True), (P.compare(">", 1, -INF), {}, True), (P.compare("<", 1, NAN), {}, False),
    (P.compare("<", 1.5, 2), {}, True), (P.compare("==", NAN, NAN), {}, False), (P.compare("==", 3, 3.0), {}, True),
    (P.compare("<", 3, 3.5), {}, True), (P.compare("<", 2.5, 1.5), {}, False), (P.compare("<", True, 2), {}, True),
    (P.compare("<", "a", "b"), {}, True), (P.compare("<", b"a", b"b"), {}, True), (P.compare("==", b"a", b"a"), {}, True),
    (P.compare("!=", "a", 1), {}, True),
    (P.binop("+", "ab", "c"), {}, "abc"), (P.binop("+", b"a", b"b"), {}, b"ab"), (P.binop("*", "ab", 2), {}, "abab"),
    (P.binop("*", 2, b"x"), {}, b"xx"), (P.binop("*", "ab", -1), {}, ""), (P.binop("//", -7, 2), {}, -4),
    (P.binop("%", -7, 2), {}, 1), (P.binop("%", 7, -2), {}, -1), (P.binop("%", -7.5, 2), {}, 0.5), (P.binop("%", 7.5, 2), {}, 1.5),
    (P.binop("//", 7.5, 2), {}, 3.0), (P.binop("**", 2, -1), {}, 0.5), (P.binop("**", 2.0, 3), {}, 8.0),
    (P.binop("+", 1.5, 1), {}, 2.5), (P.binop("*", 1.5, 2), {}, 3.0), (P.binop("-", 3, 1.5), {}, 1.5), (P.binop("/", 1, 2), {}, 0.5),
    (P.unaryop("-", True), {}, -1), (P.unaryop("+", True), {}, 1), (P.unaryop("-", 1.5), {}, -1.5), (P.unaryop("+", 1.5), {}, 1.5),
    (P.call("abs", True), {}, 1), (P.call("abs", -1.5), {}, 1.5), (P.call("bool", ""), {}, False), (P.call("float", 1), {}, 1.0),
    (P.call("float", " 1e3 "), {}, 1000.0), (P.call("float", "inf"), {}, INF), (P.call("float", "-inf"), {}, -INF),
    (P.call("str", P.call("float", "nan")), {}, "nan"), (P.call("int", " 42 "), {}, 42), (P.call("int", 2.7), {}, 2),
    (P.call("int", -2.7), {}, -2), (P.call("int", True), {}, 1), (P.call("len", b"ab"), {}, 2),
    (P.call("len", n("v")), {"v": [1, 2]}, 2), (P.call("len", n("v")), {"v": {"a": 1}}, 1), (P.call("len", "héllo"), {}, 5),
    (P.call("max", 1, 2.5), {}, 2.5), (P.call("min", 3, 1, 2), {}, 1), (P.call("round", 2.5), {}, 2),
    (P.call("round", 3.5), {}, 4), (P.call("round", -0.5), {}, 0), (P.call("round", 7), {}, 7), (P.call("str", 1.0), {}, "1.0"),
    (P.call("str", b"x"), {}, "b'x'"), (P.call("str", n("v")), {"v": None}, "None"), (P.call("str", True), {}, "True"),
    (P.call("str", 7), {}, "7"), (P.call("str", "s"), {}, "s"), (P.import_("math", P.call("math.floor", 2.5)), {}, 2),
    (P.unaryop("-", False), {}, 0), (P.unaryop("not", n("v")), {"v": ann}, False), (P.compare("==", 1.5, 1.5), {}, True),
    (P.binop("/", 1.5, 2), {}, 0.75), (P.binop("&", 12, 10), {}, 8), (P.binop("|", 12, True), {}, 13),
    (P.binop("^", True, True), {}, False), (P.binop("&", True, False), {}, False), (P.binop("|", False, True), {}, True),
    (P.binop("<<", True, 3), {}, 8), (P.binop(">>", -7, 1), {}, -4), (P.unaryop("~", 5), {}, -6),
    (P.binop("|", P.binop("**", 2, 70), 1), {}, 2 ** 70 + 1),
    (P.compare("<", 1, 2, ("<", 3)), {}, True), (P.compare("<", 1, 3, ("<", 2)), {}, False),  # a chain, link by link
    (P.compare("==", 1, 1.0, ("in", n("v"))), {"v": [1.0]}, True), (P.compare(">", 1, 2, ("<", "a")), {}, False),  # the first false link decides, its later operands not compared
    (P.boolop("and", 1, 2, 3), {}, 3), (P.boolop("and", 1, 0, n("missing")), {}, 0), (P.boolop("or", 0, "", 5), {}, 5),
    (P.boolop("or", 0, ""), {}, ""),  # n values: the first that decides, else the last
    (P.call("sum", P.generatorexp(P.binop("*", n("p"), n("q")), P.comprehension("p", n("v")),
                                  P.comprehension("q", n("v"), (P.compare(">", n("q"), n("p")),)))), {"v": [1, 2, 3]}, 11),
    (P.call("all", P.generatorexp(P.compare(">", n("p"), 0), P.comprehension("p", n("v")))), {"v": [1, 2]}, True),
    (P.call("all", P.generatorexp(n("p"), P.comprehension("p", n("v")))), {"v": []}, True), (P.call("any", n("v")), {"v": [0, 3]}, True),
    (P.call("any", P.generatorexp(n("p"), P.comprehension("p", n("v")))), {"v": [0, ""]}, False),
    (P.call("sum", P.generatorexp(1, P.comprehension("p", n("v"), (P.compare(">", n("p"), 0),)))), {"v": [1, -1, 2]}, 2),
    (P.call("sum", P.generatorexp(1, P.comprehension("c", "abc", (P.compare("!=", n("c"), "b"), P.compare("!=", n("c"), "c"))))), {}, 1),
    (P.call("sum", n("v")), {"v": [0.1] * 10}, 1.0), (P.call("sum", n("v")), {"v": [1, 0.1, 0.2, 0.3]}, 1.6),
    (P.call("sum", n("v")), {"v": [1e16, 1, 1]}, 1.0000000000000002e16), (P.call("sum", n("v"), True), {"v": [0.1] * 10}, 2.000000000000001),
    (P.call("sum", n("v")), {"v": [True, True]}, 2), (P.call("sum", n("v")), {"v": []}, 0), (P.call("sum", n("v"), 1.5), {"v": []}, 1.5),
    (P.call("sum", n("v")), {"v": [2 ** 62, 2 ** 62, 0.5]}, 9.223372036854776e18), (P.call("sum", n("v")), {"v": [2 ** 63 - 1, 1]}, 2 ** 63),
    (P.call("sum", n("v")), {"v": [-654.25, -2 ** 63 - 1, -9, 2 ** 63 + 1]}, -663.25),
    (P.call("sum", n("v"), 2 ** 64), {"v": [1, 0.5]}, 1.8446744073709552e19),
    (P.call("min", n("v")), {"v": [3, 1, 2]}, 1), (P.call("max", P.generatorexp(P.binop("*", n("p"), 2), P.comprehension("p", n("v")))), {"v": [1, 2.5]}, 5.0),
    (P.call("max", "abc"), {}, "c"), (P.call("min", P.call("set", n("v"))), {"v": [2, 1]}, 1),
    (P.call("len", P.call("set", n("v"))), {"v": [1, 1.0, True, 2]}, 2), (P.call("len", P.call("set")), {}, 0),
    (P.unaryop("not", P.call("set")), {}, True), (P.call("all", n("v")), {"v": [1, 0]}, False), (P.ifexp(0, 1, 2), {}, 2),
    (P.binop("-", 7, 10), {}, -3),
    (P.compare("in", 2, n("v")), {"v": [1, 2]}, True), (P.compare("in", "b", "abc"), {}, True),
    (P.compare("in", "k", n("v")), {"v": {"k": 1}}, True), (P.compare("not in", 3, n("v")), {"v": [1, 2]}, True),
    (P.compare("in", 1.0, n("v")), {"v": [1]}, True), (P.compare("in", 98, b"ab"), {}, True), (P.compare("in", b"b", b"ab"), {}, True),
    (P.compare("in", b"ba", b"ab"), {}, False), (P.compare("in", n("w"), n("v")), {"v": [[1]], "w": [1]}, True),
    (P.compare("in", 1, P.call("set", n("v"))), {"v": [1]}, True), (P.compare("in", 3, P.generatorexp(n("p"), P.comprehension("p", n("v")))), {"v": [1, 2]}, False),
    (P.subscript(n("v"), 1), {"v": [10, 20]}, 20), (P.subscript(n("v"), -1), {"v": [10, 20]}, 20), (P.subscript("abc", 1), {}, "b"),
    (P.subscript(b"ab", 0), {}, 97), (P.subscript(n("v"), True), {"v": [10, 20]}, 20), (P.subscript(n("v"), 1.0), {"v": {1: "x"}}, "x"),
    (P.subscript(n("v"), "k"), {"v": {"k": 1}}, 1), (P.call("all", P.subscript(n("v"), 0)), {"v": [[1, 1]]}, True),
    (P.call("sum", b"\x01\x02"), {}, 3), (P.call("max", b"ab"), {}, 98), (P.call("len", P.call("set", n("v"))), {"v": {"a": 1, "b": 2}}, 2),
    (P.subscript(n("v"), "b"), {"v": {"a": 1, "b": 2}}, 2),
    (P.call("sum", P.attribute(n("v"), "ranks")), {"v": squad}, 6), (P.subscript(P.attribute(n("v"), "ranks"), -1), {"v": squad}, 3),
]
for expression, variables, expected in VALUES:
    value = PV.OfAny(py(expression), PV.Scope(variables, modules=modules))
    assert value == expected and type(value) is type(expected), (text(expression), value, expected)
ERRORS = [  # (expression, variables, error, message)
    (P.compare("<", "a", 1), {}, TypeError, "'<' not supported between instances of 'str' and 'int'"),
    (P.binop("*", "a", 1.5), {}, TypeError, "can't multiply sequence by non-int of type 'float'"),
    (P.binop("-", "a", 1), {}, TypeError, "unsupported operand type(s) for -: 'str' and 'int'"),
    (P.binop("&", 1.5, 1), {}, TypeError, "unsupported operand type(s) for &: 'float' and 'int'"),
    (P.binop("<<", 1, -1), {}, ValueError, "negative shift count"), (P.binop(">>", 1, -1), {}, ValueError, "negative shift count"),
    (P.unaryop("~", 1.5), {}, TypeError, "bad operand type for unary ~: 'float'"),
    (P.binop("/", 1, 0), {}, ZeroDivisionError, "division by zero"),
    (P.binop("//", 1.5, 0), {}, ZeroDivisionError, "division by zero"),
    (P.binop("%", 1, 0), {}, ZeroDivisionError, "division by zero"),
    (P.binop("**", 0.0, -1), {}, ZeroDivisionError, "zero to a negative power"),
    (P.unaryop("-", "a"), {}, TypeError, "bad operand type for unary -: 'str'"),
    (P.call("abs", "a"), {}, TypeError, "bad operand type for abs(): 'str'"),
    (P.call("float", "x"), {}, ValueError, "could not convert string to float: 'x'"),
    (P.call("int", "x"), {}, ValueError, "invalid literal for int() with base 10: 'x'"),
    (P.call("int", NAN), {}, ValueError, "cannot convert float NaN to integer"),
    (P.call("int", INF), {}, OverflowError, "cannot convert float infinity to integer"),
    (P.call("len", 1), {}, TypeError, "object of type 'int' has no len()"),
    (P.call("max", 1, "a"), {}, TypeError, "'>' not supported between instances of 'str' and 'int'"),
    (P.call("round", "x"), {}, TypeError, "type str doesn't define __round__ method"),
    (P.call("float", n("v")), {"v": None}, TypeError, "float() argument must be a string or a real number, not 'NoneType'"),
    (P.call("int", n("v")), {"v": None}, TypeError,
     "int() argument must be a string, a bytes-like object or a real number, not 'NoneType'"),
    (P.import_("numpy", P.attribute(n("numpy"), "nope")), {}, AttributeError, "module 'numpy' has no attribute 'nope'"),
    (P.subscript(n("t"), "y"), {"t": {"x": 1}}, KeyError, "'y'"),
    (P.subscript(n("t"), "x"), {"t": 3}, TypeError, "'int' object is not subscriptable"),
    (P.call(P.subscript(n("t"), "f")), {"t": {"f": lambda: 1}}, TypeError, "calling '<lambda>' is not allowed by the scope"),
    (P.call("len", P.generatorexp(n("p"), P.comprehension("p", n("v")))), {"v": [1]}, TypeError, "object of type 'generator' has no len()"),
    (P.call("sum", n("v")), {"v": ["a"]}, TypeError, "unsupported operand type(s) for +: 'int' and 'str'"),
    (P.call("sum", n("v")), {"v": [0.5, "a"]}, TypeError, "unsupported operand type(s) for +: 'float' and 'str'"),
    (P.call("sum", n("v"), "a"), {"v": [1]}, TypeError, "sum() can't sum strings [use ''.join(seq) instead]"),
    (P.call("sum", n("v"), b""), {"v": [1]}, TypeError, "sum() can't sum bytes [use b''.join(seq) instead]"),
    (P.call("sum", n("v")), {"v": [0.5, 10 ** 400]}, OverflowError, "int too large to convert to float"),
    (P.call("min", n("v")), {"v": []}, ValueError, "min() iterable argument is empty"),
    (P.call("max", P.generatorexp(n("p"), P.comprehension("p", n("v")))), {"v": []}, ValueError, "max() iterable argument is empty"),
    (P.call("max"), {}, TypeError, "max expected at least 1 argument, got 0"), (P.call("min", 1), {}, TypeError, "'int' object is not iterable"),
    (P.call("set", n("v")), {"v": [[1]]}, TypeError, "cannot use 'list' as a set element (unhashable type: 'list')"),
    (P.call("all", 5), {}, TypeError, "'int' object is not iterable"), (P.generatorexp(n("p"), P.comprehension("p", 5)), {}, TypeError, "'int' object is not iterable"),
    (P.compare("in", 1, 5), {}, TypeError, "argument of type 'int' is not a container or iterable"),
    (P.compare("in", 1, "abc"), {}, TypeError, "'in <string>' requires string as left operand, not int"),
    (P.compare("in", "a", b"ab"), {}, TypeError, "a bytes-like object is required, not 'str'"),
    (P.compare("in", 256, b"ab"), {}, ValueError, "byte must be in range(0, 256)"),
    (P.compare("in", n("w"), n("v")), {"v": {1: 2}, "w": [1]}, TypeError, "cannot use 'list' as a dict key (unhashable type: 'list')"),
    (P.compare("in", n("w"), P.call("set", n("v"))), {"v": [1], "w": [1]}, TypeError, "cannot use 'list' as a set element (unhashable type: 'list')"),
    (P.subscript(n("v"), 5), {"v": [1]}, IndexError, "list index out of range"), (P.subscript(n("v"), -3), {"v": [1, 2]}, IndexError, "list index out of range"),
    (P.subscript("abc", 5), {}, IndexError, "string index out of range"), (P.subscript(b"a", 3), {}, IndexError, "index out of range"),
    (P.subscript(n("v"), P.binop("**", 2, 70)), {"v": [1]}, IndexError, "cannot fit 'int' into an index-sized integer"),
    (P.subscript(n("v"), 1.5), {"v": [1]}, TypeError, "list indices must be integers or slices, not float"),
    (P.subscript("abc", 1.5), {}, TypeError, "string indices must be integers, not 'float'"),
    (P.subscript(b"ab", 1.5), {}, TypeError, "byte indices must be integers or slices, not float"),
    (P.subscript(5, 0), {}, TypeError, "'int' object is not subscriptable"), (P.subscript(n("v"), 5), {"v": {"k": 1}}, KeyError, "5"),
    (P.subscript(n("v"), n("w")), {"v": {"k": 1}, "w": ["k"]}, TypeError, "cannot use 'list' as a dict key (unhashable type: 'list')"),
    (P.binop("@@", 1, 2), {}, NotImplementedError, "'@@' is not a core operation"),
    (lambda u: getattr(u, P.LITERAL)(lambda r: r), {}, ValueError, "a literal needs a value"),
    # The evaluator is shallow: what it does not evaluate it refuses by name, before evaluating anything
    (P.binop("@", 1, 2), {}, NotImplementedError, "Python3 binop '@' is not evaluated"),
    (P.compare("<", 1, 2, ("is", 3)), {}, NotImplementedError, "Python3 compare 'is' is not evaluated"),
    (P.compare("<", 1, 2, ("===", 3)), {}, NotImplementedError, "'===' is not a core operation"),
    (P.lambda_(1), {}, NotImplementedError, "Python3 lambda is not evaluated"),
    (P.constant(None), {}, NotImplementedError, "Python3 constant is not evaluated"),
    (P.tuple_(1, 2), {}, NotImplementedError, "Python3 tuple is not evaluated"),
    (P.subscript(n("v"), P.slice_(1)), {"v": [1]}, NotImplementedError, "Python3 slice is not evaluated"),
    (P.call("abs", P.keyword("x", 1)), {}, NotImplementedError, "Python3 keyword is not evaluated"),
    (P.call("abs", P.starred(n("v"))), {"v": [1]}, NotImplementedError, "Python3 starred is not evaluated"),
    (P.dict_((None, n("v"))), {"v": {}}, NotImplementedError, "Python3 dict's ** item is not evaluated"),
    (P.generatorexp(1, P.comprehension("p", n("v"), is_async=True)), {"v": [1]}, NotImplementedError,
     "Python3 async comprehension is not evaluated"),
    (P.generatorexp(1, P.comprehension(P.tuple_(n("p")), n("v"))), {"v": [1]}, NotImplementedError,
     "Python3 tuple target is not evaluated"),
    (lambda u: getattr(u, P.GENERATOREXP)(lambda r: r.body(P.constant(1)).generators([lambda a: a.as_object(lambda o: o.iter(n("v")))])),
     {"v": [1]}, NotImplementedError, "Python3 missing target is not evaluated"),
    (lambda u: getattr(u, P.GENERATOREXP)(lambda r: r.body(P.constant(1))), {}, ValueError, "a generatorexp needs a for clause"),
    (lambda u: getattr(u, P.IMPORT)(lambda r: r.module("m")), {}, ValueError, "an import needs a body"),
]
for expression, variables, error, message in ERRORS:
    with raises(error, match=message):
        PV.OfAny(py(expression), PV.Scope(variables, modules=modules))
assert PV.OfAny(py(P.importfrom("numpy.ma", "masked", n("masked"))), PV.Scope(modules=modules)) is np.ma.masked

<a id="DIA-03"></a>
## DIA-03 · MATLAB: operators, fields and `isfield`, rendered as MATLAB source

In [ ]:
def mat(spec):
    return term(spec, M.DIALECT)


def mtext(spec):
    return MText.ToText(mat(spec))


s = M.identifier("s")
constraint = mat(M.binary("&&", M.binary(">=", M.field(s, "age"), 18), M.call("isfield", s, "email")))
assert MText.ToText(constraint) == 's.age >= 18 && isfield(s, "email")' and round_trips(constraint)
assert mtext(M.binary("-", 1, M.binary("-", 2, 3))) == "1 - (2 - 3)" and mtext(M.binary("-", M.binary("-", 1, 2), 3)) == "1 - 2 - 3"
assert mtext(M.binary(".*", M.binary("+", 1, 2), M.unary("-", M.unary("~", s)))) == "(1 + 2) .* -~s"
assert mtext(M.unary("-", M.binary("+", 1, 2))) == "-(1 + 2)" and mtext(M.unary("~", -1)) == "~-1"
assert mtext(M.binary("||", M.binary("&&", True, False), M.binary("&&", 1, 0))) == "true && false || 1 && 0"
assert [mtext(M.constant(v)) for v in [math.nan, math.inf, -math.inf, -2.5, 'say "hi"', 7]] == [
    "NaN", "Inf", "-Inf", "-2.5", '"say ""hi"""', "7"]
assert mtext(M.field(M.field(s, "a"), "b")) == "s.a.b"
assert M.DIALECT.infer(constraint, {"s": MD.Struct}) is MD.Logical
assert M.DIALECT.infer(mat(M.binary("+", "a", 1))) is MD.String and M.DIALECT.infer(mat(M.binary("+", True, 1))) is MD.Double
assert MD.Double.contains(1) and MD.Struct.contains({}) and not MD.Struct.contains("s") and MD.of(1) is MD.Double
assert MD.Struct.contains(ann) and not MD.Struct.contains([1])  # a data object is a struct
assert M.DIALECT.validate(mat(M.binary("^", 1, 2)), core=True) == ["'^' is not a core operation"]
assert M.DIALECT.validate(mat(lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(b"x"))))) == ["a literal cannot hold a bytes"]
assert M.DIALECT.validate(mat(M.binary("+", None, 1))) == ["a binary needs a left"]
assert F.form(mat(M.call("f", 1))).attributes == {"callable": "f"}  # MATLAB's call is the framework's
with raises(TypeError, match="a literal cannot hold a bytes"):
    M.constant(b"x")
with raises(TypeError, match="expected a term, a writer or a native value, got b'x'"):
    M.binary("+", b"x", 1)
with raises(Exception, match="'let' is not a member of a union value"):  # MATLAB's union has no let
    term(lambda u: u.let(lambda r: r.name("x")), M.DIALECT)
not_matlab = term(Patterns.Expressions.let_("x", 1, 2), Patterns.Expressions.DIALECT)
assert M.DIALECT.validate(not_matlab) == ["a Patterns let is not a MATLAB term"]
with raises(ValueError, match="a Patterns let is not a MATLAB term"):
    M.DIALECT.kind(not_matlab)

<a id="DIA-04"></a>
## DIA-04 · MATLAB evaluates two-valued, converts numbers and logicals, and has no unknown

In [ ]:
def ev(spec, scope=None):
    return MV.OfAny(mat(spec), scope)


assert MV.OfAny(constraint, {"s": ann}) is True and MV.OfAny(constraint, {"s": bob}) is False
assert MV.OfAny(constraint, {"s": {"age": 20, "email": "x"}}) is True  # a mapping is a struct
assert ev(M.field(s, "age"), {"s": ann}) == 30.0 and type(ev(18)) is float  # numbers are doubles
assert ev(M.binary("==", 1, True)) is True and ev(M.binary("+", True, 1)) == 2.0
assert ev(M.binary("<", "apple", "banana")) is True and ev(M.binary("==", "5", 5)) is True
assert ev(M.binary("==", "true", True)) is True and ev(M.binary("~=", "1.5", 1.5)) is False
assert ev(M.binary("+", "a", 1)) == "a1" and ev(M.binary("+", "x", 0.125)) == "x0.125"
assert [ev(M.binary("+", "x", v)) for v in [math.nan, math.inf, -math.inf, 12345.6, 1.5e-7, 123456.7, True, False]] == [
    "xNaN", "xInf", "x-Inf", "x12346", "x1.5e-07", "x1.2346e+05", "xtrue", "xfalse"]  # numbers as MATLAB writes them
assert [ev(M.binary(op, 2, 3)) for op in ["~=", "<=", ">", ">=", "-", ".*"]] == [True, True, False, False, -1.0, 6.0]
assert ev(M.binary("&&", False, M.identifier("missing"))) is False  # short-circuits
assert ev(M.binary("||", 2, M.identifier("missing"))) is True and ev(M.binary("||", 0, 0)) is False
assert ev(M.unary("~", 0)) is True and ev(M.unary("-", True)) == -1.0
assert ev(M.call("isfield", 3, "x")) is False and ev(M.call("isfield", s, 1), {"s": ann}) is False
with raises(KeyError, match='Unrecognized field name "email".'):
    ev(M.field(s, "email"), {"s": bob})
with raises(TypeError, match="Dot indexing is not supported for variables of this type."):
    ev(M.field(s, "age"), {"s": 3})
with raises(NameError, match="Unrecognized function or variable 'missing'."):
    ev(M.identifier("missing"))
with raises(TypeError, match="Operands to the && operator must be convertible to logical scalar values."):
    ev(M.binary("&&", "yes", True))
with raises(ValueError, match="NaN's cannot be converted to logicals."):
    ev(M.unary("~", math.nan))
with raises(TypeError, match="Operator '-' is not supported for operands of type 'string' and 'double'."):
    ev(M.binary("-", "a", 1))
with raises(TypeError, match="Operator '.*' is not supported for operands of type 'struct' and 'logical'."):
    ev(M.binary(".*", s, True), {"s": ann})
with raises(TypeError, match="Operator '-' is not supported for operands of type 'string'."):
    ev(M.unary("-", "a"))
with store.transaction():  # an expression, its root written by a writer
    adult = store.Expressions.Framework.Expression(lambda e: e.root(M.binary(">", M.field(s, "age"), 18)))
assert MText.ToText(adult) == "s.age > 18" and MV.OfAny(adult, {"s": ann}) is True

<a id="DIA-05"></a>
## DIA-05 · MATLAB resolves functions from imports, the path and packages that the scope provides

In [ ]:
geo = {"dist": lambda a, b: (a * a + b * b) ** 0.5, "area": lambda r: 3 * r * r}
scope = MV.Scope({"r": 2}, functions={"twice": lambda x: 2 * x}, packages={"geo": geo, "geo.more": {"area": lambda r: 0}})
wild = mat(M.import_("geo.*", M.binary("+", M.call("dist", 3, 4), M.call("area", M.identifier("r")))))
assert MText.ToText(wild) == "import geo.*\ndist(3, 4) + area(r)" and M.DIALECT.validate(wild, {"r"}) == []
assert M.DIALECT.validate(wild, {"r"}, core=True) == ["body: left: 'dist' is not a core operation",
                                                     "body: right: 'area' is not a core operation"]
assert [F.form(i).attributes for i in SY.imports(wild, M.DIALECT)] == [{"name": "geo.*"}]
assert MV.OfAny(wild, scope) == 17.0  # an int result is a double
assert ev(M.import_("geo.dist", M.call("dist", 6, 8)), scope) == 10.0
assert ev(M.call("geo.more.area", 1), scope) == 0.0 and ev(M.call("twice", 4), scope) == 8.0
assert ev(M.import_("geo.*", M.import_("geo.more.area", M.call("area", 5))), scope) == 0.0  # innermost wins
with raises(NameError, match="Unrecognized function or variable 'dist'."):
    ev(M.call("dist", 3, 4), scope)  # not imported
with raises(ImportError, match="Import argument 'os.system' cannot be found or cannot be imported."):
    ev(M.import_("os.system", 1), scope)
with raises(ImportError, match="Import argument 'geo.nope' cannot be found or cannot be imported."):
    ev(M.import_("geo.nope", 1), scope)
with raises(ImportError, match="Import argument 'nope.*' cannot be found or cannot be imported."):
    ev(M.import_("nope.*", 1), scope)
with raises(ValueError, match="an import can only enclose the whole expression"):
    mtext(M.binary("+", 1, M.import_("geo.*", 2)))
assert M.DIALECT.validate(mat(M.import_("geo", 1))) == ["an import's name must be pkg.name or pkg.*, got 'geo'"]
assert M.DIALECT.validate(mat(M.import_("geo.*.x", 1))) == ["an import's name must be pkg.name or pkg.*, got 'geo.*.x'"]
assert M.DIALECT.validate(mat(M.import_(7, M.constant(1)))) == ["an import's name must be a str, got int"]

<a id="DIA-06"></a>
## DIA-06 · Excel: formulas with `LET`, functions, operators and fields

In [ ]:
def xl(spec):
    return term(spec, X.DIALECT)


def xtext(spec):
    return XText.ToText(xl(spec))


r = X.name("r")
constraint = xl(X.let_("age", X.field(r, "age"), X.function("AND", X.infix(">=", X.name("age"), 18),
                                                             X.function("NOT", X.function("ISERROR", X.field(r, "email"))))))
assert XText.ToText(constraint) == "=LET(age, r.age, AND(age >= 18, NOT(ISERROR(r.email))))" and round_trips(constraint)
assert xtext(X.infix("*", X.infix("+", 1, 2), X.prefix("-", X.infix("-", 3, 4)))) == "=(1 + 2) * -(3 - 4)"
assert xtext(X.infix("=", X.infix("-", 1, X.infix("-", 2, 3)), -1)) == "=1 - (2 - 3) = -1"
assert xtext(X.prefix("-", X.field(r, "first name"))) == "=-r.[first name]"
assert [xtext(X.constant(v)) for v in [True, False, 'say "hi"', math.inf, 2.5, 3]] == [
    "=TRUE", "=FALSE", '="say ""hi"""', "=#NUM!", "=2.5", "=3"]
assert all(round_trips(xl(X.constant(v))) for v in [True, 'say "hi"', 2.5, 3, -0.0, math.nan])
assert X.DIALECT.validate(constraint, {"r"}, core=True) == [] and SY.free(constraint, X.DIALECT) == {"r"}
assert X.DIALECT.validate(xl(X.function("VLOOKUP", 1, 2)), core=True) == ["'VLOOKUP' is not a core operation"]
assert X.DIALECT.infer(constraint, {"r": XD.Record}) is XD.Logical and X.DIALECT.infer(xl(X.infix("+", "1", True))) is XD.Number
assert XD.of(1) is XD.Number and XD.Errors.contains(XD.Error("#N/A")) and repr(XD.Error("#N/A")) == "#N/A"
assert XD.Record.contains(ann) and not XD.Record.contains(1)
with raises(TypeError, match="an Excel literal cannot hold a bytes"):
    X.constant(b"\x00")
for spec, problems in [
    (lambda u: u.literal(lambda c: c.value(lambda v: v.as_scalar(b"\x01"))), ["an Excel literal cannot hold a bytes"]),
    (lambda u: u.cell(lambda c: c.sheet("S")), ["a cell needs an address"]),
    (lambda u: u.call(lambda c: c.arguments([])), ["a call needs a callable"]),
    (lambda u: u.infix(lambda c: c.operator("+").left(X.constant(1))), ["an infix needs a right"]),
    (X.infix("+", None, 1), ["an infix needs a left"]),  # a writer leaves out a slot given None
    (lambda u: u.literal(lambda c: c), ["a literal needs a value"]),
    (lambda u: u.map(lambda c: c.name("p")), ["a map needs a value", "a map needs a body"]),
    (X.name("u"), ["variable 'u' is not bound"]),
    (X.function("NOT", 1, 2), ["NOT takes 1 arguments, got 2"]),
]:
    assert X.DIALECT.validate(xl(spec)) == problems, X.DIALECT.validate(xl(spec))
assert X.DIALECT.validate(xl(X.infix("^", 1, 2)), core=True) == ["'^' is not a core operation"]
quantified = term(Patterns.Expressions.variable("xs").all("p", True), Patterns.Expressions.DIALECT)  # a Patterns kind Excel does not read
assert X.DIALECT.validate(quantified) == ["a Patterns quantifier is not an Excel term"]
with raises(ValueError, match="a Patterns variable is not an Excel term"):  # printing reads its terms first
    XText.ToText(quantified)
with raises(ValueError, match="a Patterns variable is not an Excel term"):  # the same kinds, of another dialect
    xtext(Patterns.Expressions.operation("SUM", Patterns.Expressions.variable("x"), 1))

<a id="DIA-07"></a>
## DIA-07 · Excel coerces, orders values of every type, and computes with errors

In [ ]:
def ev(spec, scope=None):
    return XV.OfAny(xl(spec), scope)


Error = XV.Error
assert XV.OfAny(constraint, {"r": ann}) is True and XV.OfAny(constraint, {"r": bob}) is False  # ISERROR finds the #FIELD!
assert ev(X.field(r, "email"), {"r": bob}) == Error("#FIELD!") and ev(X.name("nope")) == Error("#NAME?")
assert ev(X.field(r, "age"), {"r": {"age": 5}}) == 5.0 and ev(X.field(r, "x"), {"r": 3}) == Error("#VALUE!")
assert ev(X.field(r, "age"), {"r": ann}) == 30.0
assert ev(X.field(X.field(r, "a"), "b"), {"r": {}}) == Error("#FIELD!")  # errors propagate
assert ev(X.infix("+", "1", True)) == 2.0 and ev(X.infix("*", "x", 1)) == Error("#VALUE!")  # coercion
assert ev(X.infix("-", 1, X.infix("+", X.name("u"), 1))) == Error("#NAME?")
assert ev(X.prefix("-", "2")) == -2.0 and ev(X.prefix("-", X.name("u"))) == Error("#NAME?")
assert ev(X.infix(">", "a", 1)) is True and ev(X.infix(">", True, "z")) is True  # number < text < logical
assert ev(X.infix("=", "a", "A")) is True and ev(X.infix("<>", 1, "1")) is True  # case; no coercion
assert [ev(X.infix(op, 2, 3)) for op in ["<", "<=", ">=", "-"]] == [True, True, False, -1.0]
assert ev(X.infix("=", 2, 2)) is True and ev(X.infix("+", False, 1)) == 1.0
assert ev(X.infix("+", "inf", 1)) == Error("#VALUE!") and ev(X.infix("+", " 1e3 ", 1)) == 1001.0  # decimal text
assert ev(X.infix("+", r, 1), {"r": {}}) == Error("#VALUE!")
assert ev(X.infix("=", X.name("u"), 1)) == Error("#NAME?") and ev(X.infix("=", r, 1), {"r": {}}) == Error("#VALUE!")
assert ev(X.function("AND", 1, True)) is True and ev(X.function("OR", 0, False)) is False
assert ev(X.function("AND", False, X.name("u"))) == Error("#NAME?")  # AND evaluates every argument
assert ev(X.function("OR", "yes", True)) == Error("#VALUE!") and ev(X.function("NOT", 0)) is True
assert ev(X.function("NOT", X.name("u"))) == Error("#NAME?")
assert ev(X.function("IF", True, 1, X.name("u"))) == 1.0 and ev(X.function("IF", 0, 1, "no")) == "no"
assert ev(X.function("IF", X.name("u"), 1, 2)) == Error("#NAME?")  # IF evaluates only the branch it takes
assert ev(X.function("ISERROR", 1)) is False and ev(X.let_("x", 2, X.infix("*", X.name("x"), 3))) == 6.0
with store.transaction():  # an expression, its root written by a writer
    adult = store.Expressions.Framework.Expression(lambda e: e.root(X.function("AND", X.infix(">=", X.field(r, "age"), 18))))
assert XV.OfAny(adult, {"r": ann}) is True and XText.ToText(adult) == "=AND(r.age >= 18)"

<a id="DIA-08"></a>
## DIA-08 · Excel reads cells from workbooks, and add-in functions from the workbook's add-ins

In [ ]:
rates = XV.Workbook(sheets={"Rates": {"B2": 0.5}}, name="Rates.xlsx")
book = XV.Workbook({"base": 10}, {"Sheet1": {"a1": 2, "$A$2": 3}, "My Sheet": {"C3": "x"}}, books={"Rates.xlsx": rates},
                   add_ins={"double": lambda v: v * 2})
total = xl(X.infix("+", X.infix("*", X.cell("A1"), X.cell("B2", "Rates", "Rates.xlsx")), X.function("DOUBLE", X.name("base"))))
assert XText.ToText(total) == "=A1 * [Rates.xlsx]Rates!B2 + DOUBLE(base)" and XV.OfAny(total, book) == 21.0 and round_trips(total)
assert X.DIALECT.validate(total, {"base"}) == [] and X.DIALECT.validate(total, {"base"}, core=True) == [
    "right: 'DOUBLE' is not a core operation"]
assert xtext(X.cell("C3", "My Sheet")) == "='My Sheet'!C3" and ev(X.cell("C3", "My Sheet"), book) == "x"
assert xtext(X.cell("A1", "Bob's", "My Book.xlsx")) == "='[My Book.xlsx]Bob''s'!A1"
assert ev(X.cell("$a$2"), book) == 3.0 and ev(X.cell("Z9"), book) == 0.0  # an empty cell is 0
assert ev(X.cell("A1", "Sheet1", "Book1"), book) == 2.0  # this book, named
assert ev(X.cell("A1", "Nope"), book) == Error("#REF!") and ev(X.cell("A1", "S", "Other.xlsx"), book) == Error("#REF!")
assert ev(X.function("NOPE", 1), book) == Error("#NAME?") and ev(X.function("double", X.name("u")), book) == Error(
    "#NAME?")  # add-ins ignore case; errors in their arguments propagate
assert ev(X.cell("A1"), XV.Workbook()) == Error("#REF!")  # a workbook has no sheet until given one
assert ev(X.name("x"), {"x": 1}) == 1.0 and XV.Workbook().sheet == "Sheet1"
with raises(ValueError, match="not a cell address: 'total'"):
    XV.Workbook(sheets={"Sheet1": {"total": 1}})
assert X.DIALECT.validate(xl(X.cell("A0"))) == ["a cell's address must be a column and a row, like A1, got 'A0'"]
assert X.DIALECT.validate(xl(X.cell("A1", None, "B.xlsx"))) == ["a cell in another book needs a sheet"]
assert X.DIALECT.infer(xl(X.cell("A1"))) is D.Anything and X.address("$b$12") == "B12" and X.address("B") is None

<a id="DIA-09"></a>
## DIA-09 · LaTeX: notation in math mode, rendered and checked, with no evaluator

In [ ]:
def tex(spec):
    return term(spec, L.DIALECT)


def ltext(spec):
    return LText.ToText(tex(spec))


a, person = L.symbol("a"), L.symbol("person")
notation = tex(L.where("a", L.member(person, "age"), L.binary("\\land", L.binary("\\geq", a, 18), L.binary(
    "\\lor", L.unary("\\lnot", L.function("has", person, "email")), L.binary(">", L.frac(a, 2), 1.5)))))
assert LText.ToText(notation) == (
    "a \\geq 18 \\land (\\lnot \\operatorname{has}(\\mathit{person}, \\text{email}) \\lor \\frac{a}{2} > 1.5) "
    "\\quad \\text{where } a = \\mathit{person}.\\mathit{age}") and round_trips(notation)
assert ltext(L.binary("\\cdot", L.binary("+", 1, 2), L.unary("-", L.binary("-", 3, -4)))) == "(1 + 2) \\cdot -(3 - -4)"
assert ltext(L.binary("\\implies", L.binary("\\implies", 1, 2), L.binary("\\implies", 3, 4))) == (
    "(1 \\implies 2) \\implies 3 \\implies 4")  # right-associative
assert ltext(L.binary("<", L.binary("<", 1, 2), 3)) == "(1 < 2) < 3"  # comparisons do not chain
assert ltext(L.member(L.binary("+", person, 1), "x_y")) == "(\\mathit{person} + 1).\\mathit{x\\_y}"
assert ltext(L.where("b", 1, L.where("a", 2, a))) == "(a \\quad \\text{where } a = 2) \\quad \\text{where } b = 1"
assert [ltext(L.constant(v)) for v in [math.nan, math.inf, -math.inf, 1e-7, 1e16, -2, True, False, "a_b {c}"]] == [
    "\\mathrm{NaN}", "\\infty", "-\\infty", "1 \\times 10^{-7}", "1 \\times 10^{16}", "-2", "\\mathrm{true}",
    "\\mathrm{false}", "\\text{a\\_b \\{c\\}}"]
assert ltext(L.constant("\\$&#%~^")) == "\\text{\\textbackslash{}\\$\\&\\#\\%\\textasciitilde{}\\textasciicircum{}}"
assert ltext(L.binary("\\cdot", L.constant(1e16), 2)) == "1 \\times 10^{16} \\cdot 2"
assert L.DIALECT.validate(notation, {"person"}, core=True) == [] and L.DIALECT.validate(notation) == [
    "value: value: variable 'person' is not bound", "body: right: left: operand: argument 0: variable 'person' is not bound"]
assert L.DIALECT.validate(tex(L.binary("\\times", 1, 2)), core=True) == ["'\\\\times' is not a core operation"]
assert L.DIALECT.validate(tex(L.function("max", 1, 2))) == [] and L.DIALECT.validate(tex(L.function("has", 1))) == [
    "has takes 2 arguments, got 1"]
assert L.DIALECT.validate(tex(lambda u: u.binary(lambda r: r.operator("+").left(L.DIALECT.spec(1))))) == [
    "a binary needs a right"]
assert L.DIALECT.validate(tex(lambda u: u.variable(lambda r: r))) == ["a variable needs a name"]
assert L.DIALECT.validate(tex(L.constant(b"\x00"))) == ["a LaTeX literal cannot hold a bytes"]  # the framework's literal, LaTeX's check
assert L.DIALECT.infer(notation, {"person": D.Anything}) is LD.Truth and L.DIALECT.infer(tex(L.frac(1, 2.5))) is LD.Number
with raises(TypeError, match="\\land cannot take (number, truth)"):
    L.DIALECT.infer(tex(L.binary("\\land", 1, True)))
with raises(TypeError, match="expected a term, a writer or a native value, got b'x'"):
    L.DIALECT.spec(b"x")
assert F.same(F.form(tex(L.frac(notation, 1))).arguments[0], notation)  # a term written into another is copied
assert LD.of("x") is LD.Text and not hasattr(Latex, "Evaluators")  # notation is not evaluated

<a id="DIA-10"></a>
## DIA-10 · Bitwise operations: Python's operators, and MATLAB's and Excel's bit functions

In [ ]:
# Python: rendered and parsed with Python's precedence, and evaluated by its rules (DIA-02's tables have more).
for source in ["a | b ^ c & d << 1 >> 2", "(a | b) & c", "~x + 1", "(a << b) + c", "x & 1 == 1", "-~x"]:
    assert PText.ToText(parsed(source)) == source
assert PV.OfAny(parsed("(x & 12 | x >> 2) ^ x << 1"), {"x": 13}) == 21
assert P.DIALECT.infer(parsed("True & False")) is PD.Bool and P.DIALECT.infer(parsed("1 << True")) is PD.Int
assert P.DIALECT.infer(parsed("~x"), {"x": PD.Bool}) is PD.Int
# MATLAB: the bit functions take doubles holding integers of 53 bits.
assert MV.OfAny(mat(M.call("bitand", 12, 10))) == 8.0 and MV.OfAny(mat(M.call("bitor", 12, True))) == 13.0
assert MV.OfAny(mat(M.call("bitxor", 12, 10))) == 6.0 and MV.OfAny(mat(M.call("bitshift", 1, 3))) == 8.0
assert MV.OfAny(mat(M.call("bitshift", 16, M.unary("-", 2)))) == 4.0
assert MV.OfAny(mat(M.call("bitshift", 2.0 ** 52, 1))) == 0.0  # beyond 53 bits
assert mtext(M.call("bitshift", M.identifier("x"), M.unary("-", 2))) == "bitshift(x, -2)"
assert M.DIALECT.infer(mat(M.call("bitand", 1, True))) is MD.Double
for call in [M.call("bitand", -1, 1), M.call("bitand", 1.5, 1), M.call("bitor", 2.0 ** 54, 1), M.call("bitshift", 1, 0.5)]:
    with raises(ValueError, match="Double inputs must have integer values in the range of ASSUMEDTYPE."):
        MV.OfAny(mat(call))
with raises(TypeError, match="Undefined function 'bitor' for input arguments of type 'string'."):
    MV.OfAny(mat(M.call("bitor", "a", 1)))
# Excel: the bit functions take integers below 2^48, converted as arithmetic converts; otherwise #NUM!.
assert XV.OfAny(xl(X.function("BITAND", 12, 10))) == 8.0 and XV.OfAny(xl(X.function("BITOR", "12", True))) == 13.0
assert XV.OfAny(xl(X.function("BITXOR", 12, 10))) == 6.0 and XV.OfAny(xl(X.function("BITLSHIFT", 1, 3))) == 8.0
assert XV.OfAny(xl(X.function("BITRSHIFT", 16, 2))) == 4.0 and XV.OfAny(xl(X.function("BITRSHIFT", 1, -3))) == 8.0
assert XV.OfAny(xl(X.function("BITLSHIFT", 16, -2))) == 4.0 and xtext(X.function("BITXOR", X.name("x"), 1)) == "=BITXOR(x, 1)"
for formula in [X.function("BITAND", -1, 1), X.function("BITAND", 1.5, 1), X.function("BITOR", 2 ** 48, 1),
                X.function("BITLSHIFT", 1, 54), X.function("BITLSHIFT", 1, 48), X.function("BITRSHIFT", 1, 0.5)]:
    assert XV.OfAny(xl(formula)) == XV.Error("#NUM!"), xtext(formula)
assert XV.OfAny(xl(X.function("BITAND", "a", 1))) == XV.Error("#VALUE!")
assert XV.OfAny(xl(X.function("BITXOR", 1, X.name("nope")))) == XV.Error("#NAME?")
assert X.DIALECT.infer(xl(X.function("BITAND", 1, 2))) is XD.Number

<a id="DIA-11"></a>
## DIA-11 · C: C and C++ expressions over their arithmetic types, with C's precedence, typed literals and casts

In [ ]:
def cc(spec):
    return term(spec, C.DIALECT)


def keywords(*words):
    """A type name of keywords (`unsigned int`), written through the builders: a spec of C's or C++'s `Type`."""
    def keyword(word):
        return lambda s: s.primitivetypespecifier(lambda p: p.keyword(word))
    return lambda t: t.typeid(lambda i: i.specifiers([keyword(word) for word in words]))


def named_type(name):
    """A type name that is a type's name (`uint8_t`), written through the builders: a spec of C's or C++'s `Type`."""
    return lambda t: t.typeid(lambda i: i.specifiers([lambda s: s.namedtypespecifier(lambda n: n.name(
        lambda m: m.identifier(lambda d: d.spelling(name))))]))


x, y, device = C.identifier("x"), C.identifier("y"), C.identifier("device")
cases = [
    (C.binary("+", x, C.binary("*", y, 2)), "x + y * 2"), (C.binary("*", C.binary("+", x, y), 2), "(x + y) * 2"),
    (C.binary("-", x, C.binary("-", y, 1)), "x - (y - 1)"), (C.binary("<<", C.binary("+", x, 1), 2), "x + 1 << 2"),
    (C.binary("&", C.binary("==", x, 1), y), "x == 1 & y"), (C.binary("==", C.binary("&", x, 1), y), "(x & 1) == y"),
    (C.binary("||", C.binary("&&", x, y), C.unary("!", x)), "x && y || !x"),
    (C.binary("|", C.binary("^", x, y), C.binary("%", x, y)), "x ^ y | x % y"),
    (C.binary("<", C.binary(">>", x, 1), C.binary("/", y, 2)), "x >> 1 < y / 2"),
    (C.conditional(x, y, C.conditional(y, 1, 2)), "x ? y : y ? 1 : 2"), (C.conditional(C.conditional(x, y, 1), 2, 3), "(x ? y : 1) ? 2 : 3"),
    (C.unary("-", C.unary("-", x)), "- -x"), (C.unary("-", -1), "- -1"), (C.unary("~", C.unary("!", x)), "~!x"),
    (C.cast(named_type("uint8_t"), C.binary("+", x, 1)), "(uint8_t)(x + 1)"), (C.unary("-", C.cast(keywords("int"), x)), "-(int)x"),
    (C.member(C.member(device, "a"), "b", "->"), "device.a->b"), (C.subscript(C.member(device, "ports"), C.binary("+", x, 1)), "device.ports[x + 1]"),
    (C.call("f", x, C.binary("+", y, 1)), "f(x, y + 1)"), (C.member(C.binary("+", x, y), "a"), "(x + y).a"),
    (C.literal(5, "u"), "5u"), (C.literal(5, "ul"), "5ul"), (C.literal(5, "ll"), "5ll"), (C.literal(5, "ULL"), "5ULL"),
    (C.literal(5, "l"), "5l"), (C.literal(5), "5"), (C.literal(1.5, "f"), "1.5f"), (C.literal(1.5, "L"), "1.5L"),
    (C.literal(1.0, "L"), "1.0L"), (C.literal(math.inf, "L"), "INFINITY"), (C.cast(named_type("uint8_t"), 5), "(uint8_t)5"),
    (C.cast(keywords("unsigned", "long", "long"), 5), "(unsigned long long)5"), (C.literal(2.5), "2.5"),
    (C.literal(1.0), "1.0"), (C.literal(1e300), "1e+300"), (C.literal(math.nan), "NAN"), (C.literal(-math.inf), "-INFINITY"), (C.literal(math.inf), "INFINITY"),
    (C.literal(True), "true"), (C.literal(False), "false"), (C.literal('a"\\\n\t\r'), '"a\\"\\\\\\n\\t\\r"'),
    (C.literal(-5), "-5"), (C.binary("-", x, -5), "x - -5"),
]
for expression, source in cases:
    assert CText.ToText(cc(expression)) == source, (source, CText.ToText(cc(expression)))
sample = cc(C.conditional(C.binary("&&", C.unary("!", C.member(device, "a")), C.binary(
    ">", C.subscript(C.member(device, "b", "->"), 0), C.literal(1.5, "f"))), C.cast(named_type("uint8_t"), C.call("f", 1)), C.literal(5, "u")))
assert round_trips(sample) and CText.ToText(sample) == "!device.a && device->b[0] > 1.5f ? (uint8_t)f(1) : 5u"
# Validation: what a literal's text encodes, each its own vocabulary, and its value, of a type its suffix and base allow;
# a member's operator; a callee is a term, its name bound as a variable's is.
assert C.DIALECT.validate(cc(C.literal(5, "q"))) == ["a literal's suffix cannot be 'q'"]
assert C.DIALECT.validate(cc(C.literal(2 ** 70))) == ["integer literal 1180591620717411303424 does not fit any type"]
assert C.DIALECT.validate(cc(C.literal(2 ** 70, "u"))) == ["integer literal 1180591620717411303424 does not fit 'unsigned long long'"]
assert C.DIALECT.validate(cc(C.literal(-5, "u"))) == ["integer literal -5 does not fit 'unsigned long long'"]
assert C.DIALECT.validate(cc(C.literal(2 ** 64 - 1, "u"))) == [] and C.DIALECT.validate(cc(C.literal(2 ** 63))) == []  # the last that holds it
assert C.DIALECT.validate(cc(lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(300)).quote("'").prefix("u8")))) == [
    "integer literal 300 does not fit 'unsigned char'"]
assert C.DIALECT.validate(cc(C.literal("a", "u"))) == ["a literal's suffix cannot be 'u'"]  # a string has none
assert C.DIALECT.validate(cc(C.literal(1.5, "u"))) == ["a literal's suffix cannot be 'u'"]  # nor a float an integer's
assert [C.DIALECT.validate(cc(C.literal(5, suffix))) for suffix in ("U", "lu", "LLU", "wb", "uwb")] == [[]] * 5
assert [C.DIALECT.validate(cc(C.literal(1.5, suffix))) for suffix in ("F", "l", "df", "DL", "f32", "F128x")] == [[]] * 6
assert C.DIALECT.validate(cc(C.member(x, "a", "=>")), {"x"}) == ["a memberexpression's operator must be '.' or '->', got '=>'"]
assert C.DIALECT.validate(cc(C.binary("**", x, 1)), {"x"}, core=True) == ["'**' is not a core operation"]
assert C.DIALECT.validate(cc(C.call("f", x)), {"x", "f"}, core=True) == []  # the scope's functions, bound by name
assert C.DIALECT.validate(cc(C.call("f", x)), {"x"}) == ["function: variable 'f' is not bound"]
assert C.DIALECT.validate(cc(lambda u: u.castexpression(lambda r: r))) == ["a castexpression needs a type", "a castexpression needs an operand"]
assert C.DIALECT.validate(cc(C.literal(None))) == ["a literal needs a value"]
with raises(ValueError, match="C literal has no value to print"):  # an invalid term has no source
    CText.ToText(cc(C.literal(None)))
with raises(TypeError, match="expected a term, a writer or a native value, got b'x'"):
    C.DIALECT.spec(b"x")  # no C literal holds bytes
assert C.DIALECT.validate(cc(C.binary("+", x, 1))) == ["left: variable 'x' is not bound"]
# A literal's range is its stored type's, read through its holder: C's own types, then its holder's ABI's; a holder
# naming no ABI defines no `int`, and a lone term reaches no store
assert C.DIALECT.validate(term(C.literal(5), C.DIALECT, None)) == ["'int' is not a type C defines, and its expression names no ABI"]
assert C.DIALECT.validate(term(C.literal(5, "wb"), C.DIALECT, None)) == [] and C.DIALECT.validate(term(C.literal(1.5), C.DIALECT, None)) == []  # read none
assert C.DIALECT.validate(term(C.literal(300), C.DIALECT, AAPCS64_)) == [] and C.DIALECT.validate(term(C.literal(2 ** 70), C.DIALECT, AAPCS64_)) == [
    "integer literal 1180591620717411303424 does not fit any type"]
with store.transaction():
    lone = store.Expressions.Framework.Term(C.literal(5))
with raises(TypeError, match="the value does not belong to a store"):
    C.DIALECT.validate(lone)
# Types: each a stored domain, read through a term's holder, among C's own (`int32_t`) and then among the ABI's its holder
# names (`int`, `size_t`): here x86-64's System V ABI, LP64. The integer promotions and the usual arithmetic conversions.
T = CD.Types(cc(0))
assert T.named("int32_t") == T.named("int") and T.named("int32_t").name() == "int32_t" and repr(T.named("size_t")) == "size_t"
assert T.named("size_t") == T.named("unsigned long") and T.named("int").types is T and T.named("int") is T.named("int")  # read once
assert T.named("int").domain is D.domains(AMD64_)["int"] and T.named("int32_t").domain is D.domains(store.Instances.Expressions.C.Expressions)["int32_t"]
assert CD.promoted(T.named("uint8_t")) is T.named("int") and CD.promoted(T.named("bool")) is T.named("int") and CD.promoted(T.named("long")) is T.named("long")
assert CD.common(T.named("int"), T.named("unsigned int")).name() == "unsigned int" and CD.common(T.named("long"), T.named("unsigned int")).name() == "long"
assert CD.common(T.named("float"), T.named("long")).name() == "float" and CD.common(T.named("double"), T.named("long double")).name() == "long double"
assert CD.common(T.named("unsigned long"), T.named("long long")) == T.named("unsigned long") and CD.common(T.named("short"), T.named("char")) is T.named("int")
assert CD.common(T.named("int"), T.named("long long")).name() == "long long" and CD.common(T.named("long"), T.named("int")).name() == "long"
assert CD.common(T.named("_Decimal32"), T.named("double")) is None and CD.common(T.named("_Decimal64"), T.named("int")).name() == "_Decimal64"  # decimal
assert CD.common(T.named("_Decimal32"), T.named("_Decimal128")).name() == "_Decimal128" and T.named("_Decimal32").decimal and not T.named("double").decimal
assert [CD.of(v, T) for v in [1, 2 ** 40, 2 ** 63, True, 1.5, "a"]] == ["int", "long", "unsigned long", "bool", "double", None]
assert CD.of(2 ** 64, T) is None and CD.of(None, T) is None
assert T.named("uint8_t").contains(255) and not T.named("uint8_t").contains(256) and T.named("int8_t").contains(-128) and not T.named("int8_t").contains(-129)
assert T.named("float").contains(1.5) and T.named("long double").contains("1.5") and T.named("bool").contains(True) and not T.named("int").contains(True)
assert T.named("_Float16").contains(1.5) and T.named("_Float128").contains("1.5") and T.named("_Decimal32").contains("1.5")  # text, where not binary16 to 64
assert not T.named("int").contains(1.5) and T.named("int").includes(T.named("int32_t")) and T.named("int").includes(D.OfUnion(T.named("int")))
assert not T.named("int").includes(T.named("long")) and T.named("long double").format == (2, 64, 16383) and T.named("long double").width == 80  # x87's
assert T.label(T.named("int32_t")).name() == "int" and T.label(T.named("_Float32x")).name() == "double" and T.label(T.named("_Float16")).name() == "_Float16"
assert T.named("long double") == T.named("_Float64x") and T.named("long double") != T.named("double")  # by format and width
assert T.find("my_t") is None and T.find("nullptr_t") is None  # a program's own type; a type that is not arithmetic
assert C.untyped(2 ** 64) is D.Anything  # a literal no type holds is not known (validation refuses it before inference)
with raises(ValueError, match="'my_t' is not a type C or AMD64 defines"):
    T.named("my_t")
# A type name's arithmetic type, its keywords in any order, cv-qualifiers aside, by its name; a type's name is itself
for names, named in [(["unsigned"], "unsigned int"), (["long", "unsigned", "int"], "unsigned long"), (["const", "char"], "char"),
                     (["signed", "char"], "signed char"), (["unsigned", "char"], "unsigned char"), (["short", "int"], "short"),
                     (["long", "long"], "long long"), (["signed"], "int"), (["long", "double"], "long double"), (["_Bool"], "bool"),
                     (["double"], "double"), (["float"], "float"), (["uint8_t"], "uint8_t"), (["volatile", "size_t"], "size_t"),
                     (["bool"], "bool"), (["my_t"], "my_t")]:
    assert CD.arithmetic(names) == named, names
for names in [[], ["void"], ["signed", "unsigned"], ["int", "int"], ["long", "long", "long"], ["short", "long"], ["long", "float"],
              ["unsigned", "double"], ["long", "char"], ["char", "int"], ["unsigned", "unsigned"], ["uint8_t", "int"],
              ["long", "long", "double"], ["unsigned", "float"], ["long", "bool"], [None]]:
    assert CD.arithmetic(names) is None, names
# A literal's type: what its text encodes, by C's rules, a stored domain found by its name
typed = lambda **attributes: CD.literal(attributes, "C", T)  # noqa: E731
CPT = CD.Types(term(0, Cpp.Expressions.DIALECT), "Cpp.Expressions")  # a C++ term's
assert typed(value=5) is T.named("int") and typed(value=1.5) is T.named("double") and typed(value=True) is T.named("bool")  # its value's
assert typed(value=None) is None and typed(value=2 ** 64) is None  # no value, and an int no type holds
assert typed(value=5, suffix="u") is T.named("unsigned int") and typed(value=2 ** 32, suffix="U") is T.named("unsigned long") and typed(value=5, suffix="LL") is T.named("long long")
assert typed(value=0xFFFFFFFF, base=16) is T.named("unsigned int") and typed(value=2 ** 31, base=10, suffix="l") is T.named("long") and typed(value=5, base=8) is T.named("int")
assert typed(value=2 ** 64, suffix="u") is T.named("unsigned long long")  # none holds it: the last, which evaluation refuses
assert typed(value=1.5, suffix="f") is T.named("float") and typed(value=1.5, suffix="L") is T.named("long double") and typed(value=1.5, suffix="f16").name() == "_Float16"
assert typed(value=1.5, suffix="DF").name() == "_Decimal32" and typed(value=1.5, suffix="f64x").name() == "_Float64x" and typed(value=1.5, suffix="q") is D.Anything
assert CD.literal({"value": 1.5, "suffix": "f16"}, "C++", CPT).name() == "std::float16_t" and CD.literal({"value": 1.5, "suffix": "bf16"}, "C++", CPT).name() == "std::bfloat16_t"
assert CD.literal({"value": 1.5, "suffix": "df"}, "C++", CPT) is D.Anything  # C++ writes no decimal floating literal: unknown
assert typed(value=5, suffix="wb").name() == "_BitInt" and typed(value=5, suffix="uwb").name() == "unsigned _BitInt"
assert typed(value=5, suffix="z").name() == "long" and typed(value=5, suffix="q") is D.Anything  # a suffix no language writes: unknown
assert typed(value=97, quote="'") is T.named("int") and CD.literal({"value": 97, "quote": "'"}, "C++", CPT).name() == "char"  # a character: its code
assert typed(value=97, quote="'", prefix="u8") is T.named("unsigned char") and CD.literal({"value": 97, "quote": "'", "prefix": "u8"}, "C++", CPT).name() == "char8_t"
assert typed(value=97, quote="'", prefix="L") is T.named("wchar_t") and typed(value="a") is CD.String and typed(value="a", prefix="u").name() == "char16_t string"
assert typed(value="a", prefix="Q").name() == "Q string" and typed(value=97, quote="'", prefix="Q").name() == "Q"  # a prefix no language has
with raises(ValueError, match="'_Float128x' is not a type C or AMD64 defines"):  # neither C nor this ABI provides it
    typed(value=1.5, suffix="f128x")


def infer(spec, environment=None):
    return C.DIALECT.infer(cc(spec), environment)


assert infer(C.binary("+", C.cast(named_type("uint8_t"), 1), 1)).name() == "int" and infer(C.binary("+", 1, C.literal(1, "u"))).name() == "unsigned int"
assert infer(C.binary("<<", C.cast(named_type("uint8_t"), 1), C.literal(1, "l"))).name() == "int" and infer(C.binary("<", 1, 2.0)).name() == "bool"
assert infer(C.binary("&&", x, 1), {"x": D.Anything}).name() == "bool" and infer(C.binary("+", x, 1), {"x": D.Anything}) is D.Anything
assert infer(C.binary("==", x, y), {"x": D.Anything, "y": D.Anything}).name() == "bool"  # a comparison's type, whatever its operands'
assert infer(C.cast(named_type("uint16_t"), x), {"x": D.Anything}) == T.named("uint16_t") and infer(C.cast(named_type("uint8_t"), 5)) == T.named("uint8_t")
assert infer(C.unary("~", C.cast(named_type("uint8_t"), 1))).name() == "int" and infer(C.unary("-", 1.5)).name() == "double"
assert infer(C.unary("!", 1.5)).name() == "bool" and infer(C.unary("!", x), {"x": D.Anything}).name() == "bool"
assert infer(C.binary("%", C.cast(named_type("uint32_t"), 7), C.cast(named_type("uint32_t"), 2))).name() == "unsigned int"
assert infer(C.binary("+", C.cast(named_type("int32_t"), 7), C.cast(named_type("int64_t"), 2))).name() == "long"  # named as C writes it
assert infer(C.binary("*", C.literal(1.5, "f16"), C.literal(2.5, "f16"))).name() == "_Float16" and infer(C.literal(1.5, "dd")).name() == "_Decimal64"
assert infer(C.cast(named_type("my_t"), 1)) is D.Anything  # a type a program declares: not arithmetic
assert infer(C.conditional(x, 1, 2), {"x": D.Anything}) is D.Anything and infer(C.member(x, "a"), {"x": D.Anything}) is D.Anything
assert infer(C.call("f", 1), {"f": D.Anything}) is D.Anything and infer(C.literal("a")) is CD.String
for expression, message in [(C.binary("%", 1.5, 1), "% cannot take (double, int); it takes (integer, integer) -> their common type"),
                            (C.binary("+", "a", 1), "+ cannot take (string, int)"), (C.binary("<<", 1.5, 1), "<< cannot take (double, int)"),
                            (C.unary("~", 1.5), "~ cannot take (double); it takes (integer) -> its promoted type"),
                            (C.binary("+", C.literal(1.5, "df"), 1.5), "+ cannot take (_Decimal32, double)")]:  # decimal and binary
    with raises(TypeError, match=message):
        infer(expression)

<a id="DIA-12"></a>
## DIA-12 · C evaluates by C's rules: promotions and conversions, wrapping unsigned arithmetic, and undefined behavior raising

In [ ]:
def ev(expression, abi=AMD64_, **variables):
    """The value of a C term, read under `abi` (x86-64's System V ABI unless given)."""
    return CV.OfAny(term(expression, C.DIALECT, abi), variables)


x = C.identifier("x")
U = lambda value: C.literal(value, "u")  # noqa: E731
assert ev(C.binary("+", U(4294967295), 1)) == 0 and ev(C.binary("-", U(0), 1)) == 4294967295 and ev(C.binary("*", U(65536), U(65536))) == 0
assert ev(C.binary("<", -1, U(0))) is False and ev(C.binary("<", -1, C.cast(keywords("unsigned", "short"), 0))) is True  # promoted to int
assert ev(C.binary("+", C.cast(named_type("uint8_t"), 200), C.cast(named_type("uint8_t"), 100))) == 300  # in int
assert ev(C.binary("/", -7, 2)) == -3 and ev(C.binary("%", -7, 2)) == -1 and ev(C.binary("/", 7, -2)) == -3 and ev(C.binary("%", 7, -2)) == 1
assert ev(C.binary("/", 7.0, 2)) == 3.5 and ev(C.binary("/", 1.0, 0.0)) == math.inf and math.isnan(ev(C.binary("/", 0.0, 0.0)))
assert ev(C.binary("+", C.cast(keywords("float"), 1.5), C.cast(keywords("float"), 0.1))) == 1.600000023841858 and ev(C.binary("-", 1.5, 1)) == 0.5
assert ev(C.binary("*", C.cast(keywords("long", "double"), 1.5), 2)) == "3"
assert ev(C.binary("&", 12, 10)) == 8 and ev(C.binary("^", 12, 10)) == 6 and ev(C.binary("|", -1, U(0))) == 4294967295
assert ev(C.binary("<<", U(1), 31)) == 2147483648 and ev(C.binary(">>", -8, 1)) == -4 and ev(C.binary("<<", C.cast(named_type("uint8_t"), 1), 8)) == 256
assert ev(C.unary("-", U(1))) == 4294967295 and ev(C.unary("~", C.cast(named_type("uint8_t"), 0))) == -1 and ev(C.unary("~", U(0))) == 4294967295
assert ev(C.unary("+", True)) == 1 and ev(C.unary("-", 1.5)) == -1.5 and ev(C.unary("+", 1.5)) == 1.5
assert ev(C.unary("!", 0.0)) is True and ev(C.unary("!", -0.0)) is True and ev(C.unary("!", math.nan)) is False and ev(C.unary("!", 3)) is False
assert ev(C.binary("==", math.nan, math.nan)) is False and ev(C.binary("!=", math.nan, 1.0)) is True and ev(C.binary("==", -0.0, 0.0)) is True
assert ev(C.binary("<", 1, 1.5)) is True and ev(C.binary(">=", C.cast(keywords("long", "double"), 2), 1)) is True and ev(C.binary(">", 2, 1)) is True
assert ev(C.binary("<=", math.inf, math.inf)) is True and ev(C.binary("<", C.cast(keywords("long", "double"), math.nan), 1)) is False
assert ev(C.binary("&&", 0, C.binary("/", 1, 0))) is False and ev(C.binary("||", 2, C.binary("/", 1, 0))) is True  # decided early
assert ev(C.binary("&&", 1, 2.5)) is True and ev(C.binary("||", 0, 0.0)) is False
assert ev(C.conditional(C.binary(">", x, 0), C.binary("/", 10, x), 0), x=0) == 0 and ev(C.conditional(1, 2, 3.5)) == 2  # the chosen one
assert ev(C.cast(named_type("uint8_t"), 300)) == 44 and ev(C.cast(named_type("int8_t"), 200)) == -56 and ev(C.cast(keywords("int"), 2.9)) == 2 and ev(C.cast(keywords("int"), -2.9)) == -2
assert ev(C.cast(keywords("bool"), 0.5)) is True and ev(C.cast(keywords("bool"), 0)) is False and ev(C.cast(keywords("bool"), -0.0)) is False
assert ev(C.cast(keywords("double"), 2 ** 60 + 1)) == 2.0 ** 60 and ev(C.cast(keywords("float"), 0.1)) == 0.10000000149011612 and ev(C.cast(keywords("int"), True)) == 1
assert ev(C.cast(keywords("long", "double"), 0.1)) == "0.10000000000000000555" and ev(C.cast(keywords("double"), C.cast(keywords("float"), 0.1))) == 0.10000000149011612
assert ev(C.cast(named_type("uint8_t"), 5)) == 5 and ev(C.literal("hi")) == "hi" and ev(C.literal(True)) is True
# A literal's type is what its text encodes: a float rounded to its type, a long double's as its text
assert ev(C.literal(0.1, "f")) == 0.10000000149011612 and ev(C.literal(1.5, "L")) == "1.5" and ev(C.literal(0xFFFFFFFF, "u")) == 4294967295
assert ev(term(lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(97)).quote("'")), C.DIALECT)) == 97  # 'a', an int in C
# What C leaves undefined raises.
for expression, error, message in [
    (C.binary("+", 2147483647, 1), OverflowError, "signed overflow in +: 2147483648 does not fit 'int'"),
    (C.unary("-", -2147483648), OverflowError, "signed overflow in -: 2147483648 does not fit 'int'"),
    (C.binary("/", -2147483648, -1), OverflowError, "signed overflow in /: 2147483648 does not fit 'int'"),
    (C.binary("%", 1, 0), ZeroDivisionError, "integer division by zero"),
    (C.binary("<<", 1, 32), ValueError, "shift by 32 is undefined for 'int'"), (C.binary(">>", 1, -1), ValueError, "shift by -1 is undefined for 'int'"),
    (C.binary("<<", -1, 1), ValueError, "left shift of a negative value is undefined"),
    (C.binary("<<", 1, 31), OverflowError, "signed overflow in <<: 2147483648 does not fit 'int'"),
    (C.cast(named_type("uint8_t"), 300.0), OverflowError, "300.0 is out of range of 'uint8_t'"),
    (C.cast(keywords("int"), math.inf), OverflowError, "inf is out of range of 'int'"),
    (C.cast(keywords("int"), C.cast(keywords("long", "double"), 1e300)), OverflowError, " is out of range of 'int'"),
    (C.cast(keywords("int"), math.nan), ValueError, "NaN has no integer value"),
    (C.binary("%", 1.5, 1), TypeError, "invalid operands to binary % ('double' and 'double')"),
    (C.binary("<<", 1.5, 1), TypeError, "invalid operands to binary << ('double' and 'int')"),
    (C.unary("~", 1.5), TypeError, "invalid argument type 'double' to unary expression"),
    (C.binary("+", "a", 1), TypeError, "invalid operand to +: 'str'"), (C.unary("!", "a"), TypeError, "invalid operand to !: 'str'"),
]:
    with raises(error, match=message):
        ev(expression)
# A variable's type is its value's (a value carries no domain: one of another type waits for domain inference).
with raises(OverflowError, match="integer 18446744073709551616 is too large for any type"):
    ev(x, x=2 ** 64)
assert ev(C.cast(named_type("uint8_t"), C.binary("+", x, 1)), x=255) == 0 and ev(C.binary("+", x, x), x=1.5) == 3.0
assert ev(C.binary("+", x, 1), x=2 ** 40) == 2 ** 40 + 1
assert ev(C.binary("*", x, 2), x=-5) == -10 and ev(C.literal(5)) == 5  # no suffix: its value's
# Members, elements and calls: data objects in the store are read as structures.
assert ev(C.member(x, "age"), x=ann) == 30 and ev(C.member(x, "age", "->"), x={"age": 5}) == 5 and ev(C.member(x, "age"), x={"age": 7}) == 7
assert ev(C.subscript(x, 1), x=[10, 20]) == 20 and ev(C.subscript(C.subscript(C.member(x, "grid"), 1), 0), x=board) == 3
assert ev(C.member(x, "name"), x=board) == "b"
for expression, variables, error, message in [
    (C.member(x, "phone"), ann, KeyError, "no member named 'phone'"),
    (C.member(x, "a"), 1, TypeError, "member reference base type 'int' is not a structure"),
    (C.subscript(x, 2), [10, 20], ValueError, "index 2 is out of bounds of an array of 2"),
    (C.subscript(x, 0), 1, TypeError, "subscripted value is not an array: 'int'"),
    (C.subscript(x, 0.5), [1], TypeError, "array subscript is not an integer: 'double'"),
    (C.call("f", 1), None, FE.NameError, "use of undeclared identifier 'f'"), (C.identifier("z"), None, FE.NameError, "use of undeclared identifier 'z'"),
    (C.literal(2 ** 64, "u"), None, OverflowError, "integer literal 18446744073709551616 does not fit 'unsigned long long'"),
]:
    with raises(error, match=message):
        ev(expression, x=variables)
assert CV.OfAny(cc(C.call("max", x, 3)), CV.Scope({"x": 5}, functions={"max": max})) == 5
with store.transaction():  # an expression, its root written by a writer, C's holder naming the ABI its terms are read under
    expression = store.Expressions.C.Expressions.Expression(lambda e: e.root(C.binary("+", x, 1)).abi([AMD64_]))
assert CText.ToText(expression) == "x + 1" and CV.OfAny(expression, {"x": 41}) == 42
# What C evaluated before it was complete keeps evaluating; any other kind, and what an evaluated kind holds beyond its
# former shape, is refused by name.
star = lambda u: u.castexpression(lambda r: r.type(lambda t: t.typeid(lambda i: i.specifiers([  # noqa: E731: (char *)x
    lambda s: s.primitivetypespecifier(lambda p: p.keyword("char"))]).declarator(lambda d: d.pointerdeclarator(lambda p: p)))).operand(x))
for expression, message in [
    (lambda u: u.sizeofexpression(lambda r: r.operand(x)), "C sizeofexpression is not evaluated"),
    (lambda u: u.parenthesizedexpression(lambda r: r.expression(x)), "C parenthesizedexpression is not evaluated"),
    (C.unary("*", x), "C unaryexpression '*' is not evaluated"), (C.binary(",", x, 1), "C binaryexpression ',' is not evaluated"),
    (C.literal(5, "wb"), "C literal of _BitInt is not evaluated"), (star, "C castexpression to a type that is not arithmetic is not evaluated"),
    (C.cast(named_type("my_t"), 1), "C castexpression to a type that is not arithmetic is not evaluated"),  # a program's own type
    (C.subscript(x, 0, 1), "C subscriptexpression of several indices is not evaluated"),
    (C.call(C.member(x, "f"), 1), "C callexpression of a callee that is not a function's name is not evaluated"),
    (lambda u: u.conditionalexpression(lambda r: r.condition(x).alternative(C.literal(1))), "C conditionalexpression without a consequence is not evaluated"),
]:
    with raises(NotImplementedError, match=message):
        ev(expression, x=[1])
assert CV.EVALUATED == {"literal", "variable", "unaryexpression", "binaryexpression", "conditionalexpression", "castexpression",
                        "memberexpression", "subscriptexpression", "callexpression"}
# Every type is a stored domain, read through the term's holder: the same term under each ABI its holder names
# evaluates as that ABI says, where they differ: plain char signed or not, wchar_t an int or an unsigned int, long double
# x87's 80-bit format or binary128
for expression, amd64, aapcs64 in [
    (C.cast(keywords("char"), 200), -56, 200), (C.cast(named_type("wchar_t"), -1), -1, 4294967295),
    (C.binary("/", C.literal(1.0, "L"), 3), "0.33333333333333333334", "0.3333333333333333333333333333333333"),
    (C.binary("/", C.literal(1.0, "f64x"), 3), "0.33333333333333333334", "0.3333333333333333333333333333333333"),
    (C.cast(keywords("long", "double"), 0.1), "0.10000000000000000555", "0.1000000000000000055511151231257827"),
    (C.binary("<", C.cast(keywords("char"), 200), 0), True, False),
]:
    assert ev(expression) == amd64 and ev(expression, AAPCS64_) == aapcs64, expression
assert ev(C.binary("+", C.cast(keywords("int"), 1), 2), AAPCS64_) == 3 and ev(C.binary("/", C.literal(1.0, "f32x"), 3)) == 1 / 3  # alike
# The types C's standard fixes are C's own: _FloatN and the decimal types evaluate in their formats, a decimal one with a
# binary one refused, as C's usual arithmetic conversions do not apply
assert ev(C.binary("+", C.literal(0.1, "f16"), C.literal(0.2, "f16"))) == 0.2998046875 and ev(C.literal(0.1, "df")) == "0.1000000"
assert ev(C.binary("*", C.literal(0.1, "dd"), 3)) == "0.3000000000000000" and ev(C.binary("/", C.literal(1.0, "f128"), 3), AAPCS64_) == "0.3333333333333333333333333333333333"
assert ev(C.binary("+", C.literal(1.5, "DL"), 1)) == "2.5" and ev(C.binary("==", C.literal(0.5, "df"), C.literal(0.5, "dd"))) is True
assert ev(term(lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(97)).quote("'").prefix("L")), C.DIALECT)) == 97  # L'a', a wchar_t
for expression, error, message in [
    (C.binary("+", C.literal(0.1, "df"), 1.0), TypeError, "invalid operands to binary + ('_Decimal32' and 'double')"),
    (C.literal(1.0, "f128x"), ValueError, "'_Float128x' is not a type C or AMD64 defines"),  # neither provides it
]:
    with raises(error, match=message):
        ev(expression)
# A holder naming no ABI defines none of the types C leaves to the implementation, and a lone term reaches no store:
# each refused where it reads a type; what reads none evaluates
assert ev(C.literal(True), None) is True and ev(C.literal(1.5, "f16"), None) == 1.5 and ev(C.literal("hi"), None) == "hi"
for expression, message in [(C.literal(5), "'int' is not a type C defines, and its expression names no ABI"),
                            (C.literal(1.5), "'double' is not a type C defines, and its expression names no ABI"),
                            (C.cast(keywords("long"), 1), "'long' is not a type C defines, and its expression names no ABI")]:
    with raises(ValueError, match=message):
        ev(expression, None)
with store.transaction():
    lone, lone_string = store.Expressions.Framework.Term(C.literal(5)), store.Expressions.Framework.Term(C.literal("hi"))
with raises(TypeError, match="the value does not belong to a store"):
    CV.OfAny(lone)
assert CV.OfAny(lone_string) == "hi"

<a id="DIA-13"></a>
## DIA-13 · SystemVerilog: expressions over 4-state vectors and reals, with IEEE 1800's precedence, sized literals and casts

In [ ]:
def sv(spec):
    return term(spec, SV.DIALECT)


def svtext(spec):
    return SVText.ToText(sv(spec))


def validate(spec, bound=(), core=False):
    return SV.DIALECT.validate(sv(spec), bound, core)


def infer(spec, environment=None):
    return SV.DIALECT.infer(sv(spec), environment)


x, y, v = SV.identifier("x"), SV.identifier("y"), SV.vector
atom = lambda keyword: lambda t: t.integeratomtype(lambda a: a.keyword(keyword))  # noqa: E731: a type, `int`, written through the builders
signing = lambda signing: lambda t: t.implicittype(lambda i: i.signing(signing))  # noqa: E731: `signed`, a type of a signing alone
cases = [
    (SV.binary("+", x, SV.binary("*", y, 2)), "x + y * 2"), (SV.binary("*", SV.binary("+", x, y), 2), "(x + y) * 2"),
    (SV.binary("-", x, SV.binary("-", y, 1)), "x - (y - 1)"), (SV.binary("**", SV.binary("**", x, y), 2), "x ** y ** 2"),
    (SV.binary("**", x, SV.binary("**", y, 2)), "x ** (y ** 2)"), (SV.binary("<<", SV.binary("+", x, 1), 2), "x + 1 << 2"),
    (SV.binary("&&", SV.binary("||", x, y), x), "(x || y) && x"), (SV.binary("||", SV.binary("&&", x, y), x), "x && y || x"),
    (SV.binary("^~", SV.binary("|", x, y), SV.binary("&", x, y)), "(x | y) ^~ x & y"),
    (SV.binary("==", SV.binary("&", x, 1), y), "(x & 1) == y"), (SV.binary("<", SV.binary(">>>", x, 1), SV.binary("%", y, 2)), "x >>> 1 < y % 2"),
    (SV.binary("->", SV.binary("->", x, y), x), "(x -> y) -> x"), (SV.binary("<->", x, SV.binary("->", y, x)), "x <-> y -> x"),
    (SV.binary("->", SV.conditional(x, y, 1), y), "x ? y : 1 -> y"), (SV.conditional(SV.binary("->", x, y), 1, 2), "(x -> y) ? 1 : 2"),
    (SV.conditional(x, y, SV.conditional(y, 1, 2)), "x ? y : y ? 1 : 2"), (SV.conditional(SV.conditional(x, y, 1), 2, 3), "(x ? y : 1) ? 2 : 3"),
    (SV.unary("-", SV.unary("-", x)), "-(-x)"), (SV.unary("&", SV.unary("&", x)), "&(&x)"), (SV.unary("~", SV.unary("!", x)), "~(!x)"),
    (SV.unary("-", -1), "-(-1)"), (SV.binary("-", x, -1), "x - -1"), (SV.unary("~&", SV.binary("+", x, 1)), "~&(x + 1)"),
    (SV.inside(SV.binary("+", x, 1), 1, SV.span(2, 5)), "x + 1 inside {1, [2:5]}"), (SV.binary("<", SV.inside(x, 1), 0), "x inside {1} < 0"),
    (SV.binary("+", SV.inside(x, 1), 1), "(x inside {1}) + 1"), (SV.binary("==?", x, v("1x")), "x ==? 2'b1x"),
    (v("11111111", base="h"), "8'hff"), (v("10x1", base="h"), "4'b10x1"), (v("1xxxx", base="h"), "5'h1x"), (v("1zzz", base="h"), "4'b1zzz"),
    (v("00000101", True, "d"), "8'sd5"), (v("xxxx", base="d"), "4'dx"), (v("0x", base="d"), "2'b0x"), (v("111", base="o"), "3'o7"),
    (v("10z1"), "4'b10z1"), (v("1", True), "1'sb1"),
    (SV.literal(1.5), "1.5"), (SV.literal(1.0), "1.0"), (SV.literal(1e300), "1e+300"), (SV.literal(math.nan), "0.0 / 0.0"),
    (SV.literal(math.inf), "1.0 / 0.0"), (SV.literal(-math.inf), "-1.0 / 0.0"), (SV.binary("*", x, math.inf), "x * (1.0 / 0.0)"),
    (SV.unary("-", math.inf), "-(1.0 / 0.0)"), (SV.literal('a"\\\n\t'), '"a\\"\\\\\\n\\t"'), (SV.literal(-5), "-5"),
    (SV.cast(signing("signed"), x), "signed'(x)"), (SV.cast(8, SV.binary("+", x, 1)), "8'(x + 1)"), (SV.cast(atom("byte"), -1), "byte'(-1)"),
    (SV.member(SV.member(x, "a"), "b"), "x.a.b"), (SV.select(SV.binary("+", x, y), 0), "(x + y)[0]"), (SV.range_(x, 7, 4), "x[7:4]"),
    (SV.concatenation(x, SV.replication(3, y)), "{x, {3{y}}}"), (SV.call("$clog2", x), "$clog2(x)"), (SV.call("f", x, 1), "f(x, 1)"),
]
for expression, source in cases:
    assert svtext(expression) == source, (source, svtext(expression))
sample = sv(SV.binary("->", SV.inside(SV.member(x, "a"), 1, SV.span(2, 5)), SV.call("$clog2", SV.iterate(
    SV.method(x, "unique"), "sum", "q", SV.identifier("q")))))
assert round_trips(sample) and svtext(sample) == "x.a inside {1, [2:5]} -> $clog2(x.unique().sum(q) with (q))"
assert SV.DIALECT.validate(sample, SY.free(sample, SV.DIALECT)) == [] and F.form(sample).attributes == {"operator": "->"}
# Validation: a constant's bits and base, and a cast's type or width.
assert validate(v("12")) == ["a constant's bits must be 0, 1, x and z, at least one, got '12'"]
assert validate(v("")) == ["a constant's bits must be 0, 1, x and z, at least one, got ''"]
assert validate(v("1", base="q")) == ["a constant's base must be b, o, d or h, got 'q'"]
assert validate(SV.cast(lambda t: t.namedtype(lambda n: n.name(lambda i: i.identifier(lambda d: d.spelling("word")))), x),
                bound={"x"}) == []  # a type a program declares: no domain is read
assert validate(SV.cast(0, x), bound={"x"}) == ["a cast's width must be positive, got 0"]
assert validate(lambda u: u.castexpression(lambda r: r.value(x)), bound={"x"}) == ["a castexpression needs a type"]
assert validate(SV.binary("<<=", x, 1), bound={"x"}, core=True) == ["'<<=' is not a core operation"]
assert validate(SV.call("f", x), bound={"f", "x"}, core=True) == [] and validate(SV.identifier("x")) == ["nameexpression 'x' is not bound"]
assert validate(SV.literal(True)) == ["a SystemVerilog literal cannot hold a bool"] and validate(SV.literal(b"\x00")) == ["a SystemVerilog literal cannot hold a bytes"]
assert validate(lambda u: u.literal(lambda r: r)) == ["a literal needs a value"]
assert validate(lambda u: u.arraymethodwithexpression(lambda r: r.value(SV.method(x, "sum"))), {"x"}) == [
    "an arraymethodwithexpression needs a body"]  # its iterator is `item`: bound within its body
assert validate(SV.iterate(x, "sum", "p", SV.identifier("q")), {"x"}) == ["body: nameexpression 'q' is not bound"]
with raises(TypeError, match="expected a term, a writer or a native value, got True"):
    SV.DIALECT.spec(True)  # a bool is no SystemVerilog constant
with raises(ValueError, match="a Patterns let is not a SystemVerilog term"):
    SVText.ToText(term(Patterns.Expressions.let_("a", 1, Patterns.Expressions.variable("a")), Patterns.Expressions.DIALECT))
# Types: integral vectors of a width, a signedness and 2 or 4 states, and reals, each named type read from the domain
# SystemVerilog's define stores (its kind's properties), a sized vector's computed.
T = {name: SVD.named(store, name) for name in ["bit", "logic", "reg", "byte", "shortint", "int", "longint", "integer", "time",
                                               "real", "shortreal"]}
assert [T[n].width for n in ["bit", "logic", "reg", "byte", "shortint", "int", "longint", "integer", "time"]] == [1, 1, 1, 8, 16, 32, 64, 32, 64]
assert T["int"].signed and T["int"].states == 2 and T["integer"].states == 4 and not T["time"].signed and T["logic"] == T["reg"]
assert T["int"].name() == "int" and repr(T["integer"]) == "integer" and SVD.vector(8).name() == "logic [7:0]"
assert SVD.vector(4, True, 2).name() == "bit signed [3:0]" and SVD.vector(32, True, 2) == T["int"] and T["shortreal"].format == "binary32"
assert SVD.common([T["byte"], T["int"]]) == T["int"] and SVD.common([T["byte"], SVD.vector(16)]) == SVD.vector(16)
assert SVD.common([T["shortreal"], T["int"]]) == T["real"] and SVD.common([T["shortreal"], T["shortreal"]]) == T["shortreal"]
assert SVD.common([T["bit"], T["logic"]]) == T["logic"]
assert [SVD.of(value) for value in [1, -2 ** 31, 2 ** 31, 2 ** 63, 1.5, "a", True]] == ["integer", "integer", "longint", None, "real",
                                                                                         None, None]  # a constant's type, by name
assert T["int"].domain is store.Instances.Expressions.SystemVerilog.Expressions.domains({"name": "int"}) and SVD.vector(8).domain is None
# A named type is a stored domain: a store where SystemVerilog's define has not run defines none, and a type no stored
# domain names is refused by name; a lone term reaches no store where it reads one.
bare = Builders.Store()
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, SystemVerilog.install, SVTypes.install]:
    bare.transaction(install)
with bare.transaction():
    bare_cast = F.term(bare.Expressions.Framework.Expression(lambda e: e.root(SV.DIALECT.spec(SV.cast(atom("int"), 1)))))
assert SV.DIALECT.validate(bare_cast) == ["'int' is not a type SystemVerilog defines"]
with raises(ValueError, match="'int' is not a type SystemVerilog defines"):
    SVD.named(bare, "int")
with store.transaction():
    lone = store.Expressions.Framework.Term(SV.DIALECT.spec(SV.binary("+", 1, SV.vector("10"))))  # held by no expression
with raises(TypeError, match="the value does not belong to a store"):
    SV.DIALECT.infer(lone)
assert SV.DIALECT.validate(lone) == [] and SV.DIALECT.infer(F.form(lone).arguments[1]) == SVD.vector(2)  # it reads no type
# Values: a Logic is a type and its bits, encoded as VPI does.
a = SVD.Logic.of("10xz")
assert (a.aval, a.bval, a.bits, a.known, repr(a)) == (0b1010, 0b0011, "10xz", False, "4'b10xz")
assert SVD.Logic.of("10xz", True, 2).bits == "1000" and repr(SVD.Logic.of("1", True)) == "1'sb1"
assert SVD.Logic.of("1110", True).integer() == -2 and SVD.Logic.of("1110").integer() == 14 and SVD.Logic.number(T["byte"], -1).bits == "11111111"
with raises(ValueError, match="4'b10xz has unknown bits"):
    a.integer()
assert T["int"].contains(SVD.Logic.number(T["int"], 5)) and not T["int"].contains(SVD.Logic.number(T["integer"], 5)) and not T["int"].contains(5)
assert T["real"].contains(1.5) and not T["real"].contains(1) and T["int"].includes(SVD.vector(32, True, 2)) and not T["int"].includes(T["integer"])
assert T["int"].includes(D.OfUnion(T["int"])) and not T["int"].includes(D.Anything)
# Inference sizes an expression as IEEE 1800 does.
assert infer(SV.binary("+", v("1010"), 1)) == SVD.vector(32) and infer(SV.binary("+", 1, 2)) == T["integer"]
assert infer(SV.binary("+", SV.cast(atom("byte"), 1), SV.cast(atom("int"), 1))) == T["int"] and infer(SV.binary("*", 1, 1.5)) == T["real"]
assert infer(SV.binary("**", v("1010"), 2)) == SVD.vector(4) and infer(SV.binary("**", 2, 0.5)) == T["real"]
assert infer(SV.binary("<<", v("1010"), 40)) == SVD.vector(4) and infer(SV.binary("==", v("1010"), 1)) == SVD.vector(1)
assert infer(SV.binary("===", v("1010"), 1)) == T["bit"] and infer(SV.binary("==?", v("1010"), 1)) == SVD.vector(1)
assert infer(SV.binary("&&", x, 1), {"x": D.Anything}) == SVD.vector(1) and infer(SV.binary("+", x, 1), {"x": D.Anything}) is D.Anything
assert infer(SV.binary("&", v("10"), v("1000"))) == SVD.vector(4) and infer(SV.unary("~", SV.cast(atom("byte"), 1))) == T["byte"]
assert infer(SV.unary("-", 1.5)) == T["real"] and infer(SV.unary("!", 1.5)) == SVD.vector(1) and infer(SV.unary("^", v("10"))) == SVD.vector(1)
assert infer(SV.binary("%", 7, 2)) == T["integer"] and infer(SV.cast(atom("int"), x), {"x": D.Anything}) == T["int"]
assert infer(SV.cast(8, x), {"x": D.Anything}) is D.Anything and infer(SV.cast(signing("signed"), 1)) is D.Anything  # sized by the operand
assert infer(SV.inside(x, 1), {"x": D.Anything}) == SVD.vector(1) and infer(SV.conditional(x, 1, 2), {"x": D.Anything}) is D.Anything
assert SV.DIALECT.kinds()["constant"].typed(F.kind_of(sv(v("12")))[1]) is None
assert infer(v("10x1", True)) == SVD.vector(4, True) and infer(SV.literal(1.5)) == T["real"]
assert infer(SV.binary("+", "a", 1)) is D.Anything  # a string has no type of its own
with raises(TypeError, match="+ cannot take (int | real, integer); it takes (T, T) -> their common type"):
    infer(SV.binary("+", x, 1), {"x": D.OfUnion(T["int"], T["real"])})
assert SVD.BINARY["+"].describe() == "(T, T) -> their common type" and SVD.UNARY["!"].arity() == 1
for expression, message in [(SV.binary("%", 1.5, 1), "% cannot take (real, integer); it takes (integral, integral) -> their common type"),
                            (SV.binary("&", 1.5, 1), "& cannot take (real, integer)"), (SV.binary("<<", 1.5, 1), "<< cannot take (real, integer)"),
                            (SV.binary("===", 1.5, 1), "=== cannot take (real, integer)"), (SV.binary("==?", 1.5, 1), "==? cannot take (real, integer)"),
                            (SV.unary("~", 1.5), "~ cannot take (real); it takes (integral) -> its type"), (SV.unary("&", 1.5), "& cannot take (real)")]:
    with raises(TypeError, match=message):
        infer(expression)
#### [28] markdown
## DIA-14 · SystemVerilog evaluates by IEEE 1800's rules: sizing by context, 4-state logic, casts and system functions
#### [29] code

<a id="DIA-14"></a>
## DIA-14 · SystemVerilog evaluates by IEEE 1800's rules: sizing by context, 4-state logic, casts and system functions

In [ ]:
def ev(expression, **variables):
    return SVV.OfAny(sv(expression), variables)


def c(type_, operand):
    """A cast: to a width (an int, or any term), a signing, or a type SystemVerilog's keywords name, its type written
    through the builders."""
    if type(type_) is not str:
        return SV.cast(type_, operand)
    if type_ in ("signed", "unsigned"):
        return SV.cast(signing(type_), operand)
    if type_ in ("real", "shortreal", "realtime"):
        return SV.cast(lambda t: t.nonintegertype(lambda n: n.keyword(type_)), operand)
    if type_ in ("bit", "logic", "reg"):
        return SV.cast(lambda t: t.integervectortype(lambda n: n.keyword(type_)), operand)
    return SV.cast(atom(type_), operand)


v, b, u, call = SV.vector, SV.binary, SV.unary, SV.call
x = SV.identifier("x")
# Each expression and its value: a Logic by its repr, or a real.
cases = [
 (b("+", v("1010"), v("0110")), "4'b0000"), (b("+", v("1111"), 1), "32'b" + "0"*27 + "10000"),
 (b("/", -7, 2), "32'sb" + "1"*30 + "01"), (b("%", -7, 2), "32'sb" + "1"*32), (b("/", 1, 0), "32'sb" + "x"*32),
 (b(">>>", v("11111110", True), 1), "8'sb11111111"), (b(">>>", v("1000"), 1), "4'b0100"), (b(">>", v("1000", True), 1), "4'sb0100"),
 (b("&", v("10x1"), v("0011")), "4'b00x1"), (b("|", v("10z1"), v("1100")), "4'b11x1"), (b("^", v("10x1"), v("0011")), "4'b10x0"),
 (b("~^", v("1001"), v("0011")), "4'b0101"), (b("+", v("10x1"), 1), "32'b" + "x"*32),
 (b("==", v("10x1"), v("10x1")), "1'bx"), (b("===", v("10x1"), v("10x1")), "1'b1"), (b("!==", v("10z1"), v("10x1")), "1'b1"),
 (b("==?", v("1001"), v("1xx1")), "1'b1"), (b("!=?", v("1001"), v("1xx0")), "1'b1"), (b("==?", v("1x01"), v("1001")), "1'bx"),
 (u("&", v("1111")), "1'b1"), (u("|", v("000x")), "1'bx"), (u("^", v("1011")), "1'b1"), (u("~&", v("1101")), "1'b1"),
 (u("~|", v("0000")), "1'b1"), (u("&", v("1x11")), "1'bx"), (u("|", v("0100")), "1'b1"), (u("~^", v("1x11")), "1'bx"), (u("~^", v("1011")), "1'b0"), (u("^", v("1x11")), "1'bx"), (u("&", v("0x11")), "1'b0"),
 (u("-", v("0001")), "4'b1111"), (u("~", v("10x1")), "4'b01x0"), (u("-", v("0x01")), "4'bxxxx"), (u("+", v("0x01")), "4'b0x01"),
 (b("**", 3, -1), "32'sb" + "0"*32), (b("**", 0, -1), "32'sb" + "x"*32), (b("**", -1, -1), "32'sb" + "1"*32),
 (b("**", -1, -2), "32'sb" + "0"*31 + "1"), (b("**", 2, 10), "32'sb" + "0"*21 + "10000000000"), (b("**", 1, -3), "32'sb" + "0"*31 + "1"),
 (b("**", 2.0, 0.5), 2 ** 0.5), (b("/", 1.0, 0), math.inf), (b("**", 2, 0.5), 2 ** 0.5), (b("+", 1, 0.5), 1.5),
 (SV.conditional(v("x"), v("1100"), v("1010")), "4'b1xx0"), (SV.conditional(v("1"), v("1100"), v("1010")), "4'b1100"),
 (SV.conditional(0, v("1100"), v("1010")), "4'b1010"), (SV.conditional(v("z"), 1.5, 1.5), 1.5), (SV.conditional(v("z"), 1.5, 2.5), 0.0),
 (SV.inside(3, 1, SV.span(2, 5)), "1'b1"), (SV.inside(7, 1, SV.span(2, 5)), "1'b0"), (SV.inside(v("1x00"), v("1100")), "1'bx"),
 (SV.inside(v("1x00"), v("1100"), v("1x00")), "1'b1"), (SV.inside(v("1x00"), SV.span(0, 3)), "1'bx"), (SV.inside(v("1x00"), SV.span(8, 15)), "1'bx"),
 (SV.inside(v("1100"), v("1x00")), "1'b1"),
 (c("byte", 300), "8'sb00101100"), (c("byte", 200), "8'sb11001000"), (c("unsigned", -1), "32'b" + "1"*32),
 (c(4, v("11111111")), "4'b1111"), (c("int", 2.5), "32'sb" + "0"*30 + "11"), (c("int", -2.5), "32'sb" + "1"*30 + "01"),
 (c("real", 3), 3.0), (c("shortreal", 0.1), 0.10000000149011612), (c("int", v("10x1")), "32'sb" + "0"*28 + "1001"),
 (c("bit", v("x")), "1'b0"), (c("signed", v("1111")), "4'sb1111"), (c(8, v("1111", True)), "8'sb11111111"), (c(8, v("z1", True)), "8'sbzzzzzzz1"), (c(4, v("x0", True)), "4'sbxxx0"),
 (call("$clog2", 8), "32'sb" + "0"*30 + "11"), (call("$clog2", 9), "32'sb" + "0"*29 + "100"), (call("$clog2", 0), "32'sb" + "0"*32),
 (call("$clog2", v("1x")), "32'sb" + "x"*32), (call("$bits", v("00000000")), "32'sb" + "0"*28 + "1000"), (call("$bits", 1.5), "32'sb" + "0"*25 + "1000000"),
 (call("$countones", v("1011")), "32'sb" + "0"*30 + "11"), (call("$onehot", v("0100")), "1'b1"), (call("$onehot", v("0000")), "1'b0"),
 (call("$onehot0", v("0000")), "1'b1"), (call("$isunknown", v("10x1")), "1'b1"), (call("$isunknown", v("1001")), "1'b0"),
 (call("$signed", v("1111")), "4'sb1111"), (call("$unsigned", v("1111", True)), "4'b1111"),
 (SV.concatenation(v("1010"), v("11")), "6'b101011"), (SV.replication(2, v("10")), "4'b1010"), (SV.concatenation(v("x"), v("1")), "2'bx1"),
 (SV.select(v("1010"), 1), "1'b1"), (SV.select(v("1010"), 4), "1'bx"), (SV.select(v("1010"), v("x")), "1'bx"), (SV.range_(v("1010"), 3, 2), "2'b10"),
 (SV.range_(v("1010"), 5, 2), "4'bxx10"), (SV.select(c("int", v("1010")), 40), "1'b0"),
 (b("+", v("1111", True), v("00000000", True)), "8'sb11111111"), (b("+", v("1111", True), v("00000000")), "8'b00001111"),
 (b("<", -1, v("1")), "1'b0"), (b("<", -1, 1), "1'b1"), (b("==", b("+", v("1111"), v("0001")), v("10000")), "1'b1"),
 (b("==", b("<<", v("1000"), 1), v("10000")), "1'b1"), (b("<<", v("0001"), v("10")), "4'b0100"), (b("<<", v("0001"), v("x")), "4'bxxxx"),
 (b("+", b("<", 2, 1), v("0001")), "4'b0001"), (b("+", b("<", 1, 2), v("0001")), "4'b0010"),
 (u("!", v("0000")), "1'b1"), (u("!", v("00x0")), "1'bx"), (u("!", 1.5), "1'b0"), (u("!", 0.0), "1'b1"), (b(">>>", v("x000", True), 1), "4'sbxx00"), (b("&&", v("x"), 0), "1'b0"), (b("||", v("x"), 1), "1'b1"),
 (b("&&", v("x"), 1), "1'bx"), (b("&&", 1, 1), "1'b1"), (b("||", 0, 0), "1'b0"), (b("&&", 0, v("x")), "1'b0"), (b("||", 1, v("x")), "1'b1"),
 (b("->", 0, v("x")), "1'b1"), (b("->", 1, v("x")), "1'bx"), (b("->", 1, 0), "1'b0"), (b("->", v("x"), 1), "1'b1"),
 (b("<->", 1, 1), "1'b1"), (b("<->", 1, 0), "1'b0"), (b("<->", v("x"), 0), "1'bx"),
 (b(">", 1.5, 1), "1'b1"), (b("==", math.nan, math.nan), "1'b0"), (b("!=", math.nan, 1.0), "1'b1"), (b("<", v("x"), 1), "1'bx"),
 (u("-", 1.5), -1.5), (u("+", 1.5), 1.5), (b("-", c("shortreal", 1.5), c("shortreal", 0.25)), 1.25),
 (b("*", 3, 4), "32'sb" + "0"*28 + "1100"), (b("-", 3, 4), "32'sb" + "1"*32), (b("%", 7, -2), "32'sb" + "0"*31 + "1"),
]
for expression, value in cases:
    result = ev(expression)
    assert (repr(result) == value if type(value) is str else type(result) is float and result == value), (svtext(expression), result)
assert math.isnan(ev(b("**", -8.0, 1 / 3))) and ev(b("**", 0.0, -1.0)) == math.inf and ev(b("**", 10.0, 400.0)) == math.inf
assert ev(b("**", -10.0, 401.0)) == -math.inf and ev(b("**", -10.0, 400.0)) == math.inf and ev(b("**", c("shortreal", 2), c("shortreal", 0.5))) == 1.4142135381698608
assert ev(b("**", -0.0, -1.0)) == -math.inf and ev(b("**", -0.0, -2.0)) == math.inf and ev(b("**", 0.0, -3.0)) == math.inf
assert ev(b("**", 1.0, math.nan)) == 1.0 and ev(b("**", -1.0, math.inf)) == 1.0 and ev(b("**", math.nan, 0.0)) == 1.0  # as C's pow
assert repr(ev(b("/", c("int", 1), c("int", 0)))) == "32'sb" + "0" * 32 and repr(ev(b("&", c("int", 5), v("x")))) == "32'b" + "0" * 31 + "x"
# Variables are typed by their values, Patterns' typed values among them.
assert repr(ev(x, x=True)) == "1'b1" and ev(x, x=5).type == SVD.named(store, "integer") and ev(x, x=2 ** 40).type.name() == "longint"
assert repr(ev(b("<<", 1, v("1" * 64)))) == "32'sb" + "0" * 32 and repr(ev(b(">>>", -4, v("1" * 64)))) == "32'sb" + "1" * 32
assert repr(ev(call("$bits", c("shortreal", 1.5)))) == "32'sb" + "0" * 26 + "100000" and repr(ev(x, x="é")) == "16'b1100001110101001"
assert repr(ev(x, x="AB")) == "16'b0100000101000010" and repr(ev(x, x="")) == "8'b00000000" and ev(x, x=1.5) == 1.5
assert repr(ev(x, x=SVD.Logic.of("1x"))) == "2'b1x" and ev(x, x={"a": 1}) == {"a": 1}  # a value carries no domain
assert repr(ev(SV.member(x, "age"), x=ann)) == "32'sb" + "0" * 26 + "011110" and repr(ev(SV.member(x, "a"), x={"a": True})) == "1'b1"
assert repr(ev(SV.select(x, 1), x=[1, 2])) == "32'sb" + "0" * 30 + "10" and repr(ev(SV.select(SV.member(x, "a"), 0), x={"a": [False]})) == "1'b0"
assert repr(ev(SV.select(SV.member(x, "ports"), 1), x=bus)) == "32'sb" + "0" * 29 + "101"  # data objects in the store
assert SVV.OfAny(sv(SV.call("f", x, 1)), SVV.Scope({"x": 2}, functions={"f": lambda a, b: a.integer() + b.integer()})).type.name() == "integer"
assert SVV.OfAny(sv(b("+", 1, 1)), SVV.Scope()).integer() == 2 and SVV.OfAny(sv(SV.binary("+", 1, 2))).integer() == 3
with store.transaction():  # an expression, evaluated from its root
    expression = store.Expressions.Framework.Expression(lambda e: e.root(SV.binary("+", x, 1)))
assert SVV.OfAny(expression, {"x": 2}).integer() == 3 and SVText.ToText(expression) == "x + 1"
# Its types are the domains SystemVerilog defines in the store of the term's holder: a lone term is refused where it
# needs one, and evaluates where it needs none.
with store.transaction():
    lone, sized = (store.Expressions.Framework.Term(SV.DIALECT.spec(spec)) for spec in (b("+", x, 1), b("&", v("1010"), v("0110"))))
with raises(TypeError, match="the value does not belong to a store"):
    SVV.OfAny(lone, {"x": 2})
assert repr(SVV.OfAny(sized)) == "4'b0010"
for (expression, variables), error, message in [
    ((x, {}), FE.NameError, "identifier 'x' is not declared"), ((SV.call("f", 1), {}), FE.NameError, "function 'f' is not declared"),
    ((v("12"), {}), ValueError, "cannot evaluate an invalid expression: a constant's bits must be 0, 1, x and z, at least one, got '12'"),
    ((x, {"x": 2 ** 70}), OverflowError, "integer 1180591620717411303424 is too large for any type"),
    ((u("~", 1.5), {}), TypeError, "~ expects an integral operand, got real"), ((u("&", 1.5), {}), TypeError, "the reduction & expects an integral operand, got real"),
    ((b("%", 1.5, 1), {}), TypeError, "% expects integral operands, got real"), ((b("&", 1.5, 1), {}), TypeError, "& expects integral operands, got real"),
    ((b("<<", 1.5, 1), {}), TypeError, "<< expects an integral operand, got real"), ((b("<<", 1, 1.5), {}), TypeError, "<< expects integral operands, got real"),
    ((b("+", x, 1), {"x": {}}), TypeError, "+ expects a number, got dict"), ((c("int", x), {"x": {}}), TypeError, "a cast expects a number, got dict"),
    ((SV.conditional(1, x, 1), {"x": {}}), TypeError, "?: expects a number, got dict"), ((u("!", x), {"x": {}}), TypeError, "a dict has no truth value"),
    ((c("signed", 1.5), {}), TypeError, "signed' expects an integral operand, got real"), ((c("int", math.nan), {}), OverflowError, "nan has no integer value"),
    ((c("int", -math.inf), {}), OverflowError, "-inf has no integer value"),
    ((SV.member(x, "b"), {"x": {"a": 1}}), KeyError, "no member named 'b'"), ((SV.member(x, "b"), {"x": 1}), TypeError, "a Logic has no members"),
    ((SV.select(x, 2), {"x": [1, 2]}), ValueError, "index 2 is out of bounds of an array of 2"),
    ((SV.select(x, v("x")), {"x": [1, 2]}), ValueError, "an array's index must be known, got 1'bx"),
    ((SV.select(x, 0), {"x": {"a": 1}}), TypeError, "a dict cannot be selected from"), ((SV.range_(x, 1, 0), {"x": {}}), TypeError, "a dict cannot be selected from"),
    ((SV.range_(v("1010"), 0, 1), {}), ValueError, "a range's msb must be at least its lsb, got [0:1]"),
    ((SV.range_(v("1010"), v("x"), 0), {}), ValueError, "a range's msb must be a known integer, got 1'bx"),
    ((SV.range_(v("1010"), 1, 0.5), {}), ValueError, "a range's lsb must be a known integer, got 0.5"),
    ((SV.replication(0, v("1")), {}), ValueError, "a replication's count must be positive, got 0"),
    ((SV.concatenation(v("1"), 1.5), {}), TypeError, "a concatenation's parts must be integral, got float"),
    ((call("$clog2", 1, 2), {}), TypeError, "$clog2 takes 1 argument, got 2"), ((call("$signed", 1.5), {}), TypeError, "$signed expects an integral argument, got real"),
    ((call("$bits", x), {"x": {}}), TypeError, "$bits expects a number, got dict"), ((call("$countones", 1.5), {}), TypeError, "$countones expects an integral argument, got real"),
]:
    with raises(error, match=message):
        ev(expression, **variables)
# The complete dialect's literals: an unsized based integer is unsigned (unless written signed) of 32 bits, an unsized
# vector extended to 32 bits by its leftmost bit where it is x or z, a fill fills its context's width; parentheses
# written in the term pass their context through.
hexadecimal = lambda value, signed=False: lambda u: u.literal(lambda r: r.value(lambda p: p.as_scalar(value)).base("h").signed(signed))  # noqa: E731
unsized = lambda bits: lambda u: u.constant(lambda r: r.bits(bits).base("b"))  # noqa: E731: an unsized vector, 'bx01
fill = lambda bit: lambda u: u.constant(lambda r: r.fill(bit))  # noqa: E731: '0, '1, 'x, 'z
parenthesized = lambda inner: lambda u: u.parenthesizedexpression(lambda r: r.expression(inner))  # noqa: E731
assert repr(ev(hexadecimal(255))) == "32'b" + "0" * 24 + "1" * 8 and repr(ev(hexadecimal(2 ** 40))) == "41'b1" + "0" * 40
assert repr(ev(hexadecimal(5, True))) == "32'sb" + "0" * 29 + "101" and repr(ev(b(">", hexadecimal(1), -1))) == "1'b0"  # unsigned
assert repr(ev(unsized("x01"))) == "32'b" + "x" * 30 + "01" and repr(ev(unsized("101"))) == "32'b" + "0" * 29 + "101"
assert repr(ev(b("&", fill("1"), v("1010")))) == "4'b1010" and repr(ev(fill("z"))) == "1'bz"
assert repr(ev(b("+", fill("1"), v("00000001")))) == "8'b00000000" and ev(b("+", fill("1"), 0.5)) == 1.5  # a real context
assert repr(ev(b("*", parenthesized(b("+", v("0001"), v("0001"))), v("0011")))) == "4'b0110"
assert repr(ev(c("bit", 3))) == "1'b1" and ev(c("real", 3)) == 3.0  # a cast to a type its keyword names
assert repr(ev(call("$bits", lambda t: t.integeratomtype(lambda a: a.keyword("longint"))))) == "32'sb" + "0" * 25 + "1000000"
assert repr(ev(c(b("+", 2, 2), v("11111111")))) == "4'b1111"  # a width, of any known value
assert repr(ev(SV.method(x, "size"), x=[1, 2])) == "32'sb" + "0" * 30 + "10"  # an array's method, a call of its member
# Every other kind is refused by name, and so is what an evaluated kind holds beyond what it evaluated before.
scoped = lambda u: u.nameexpression(lambda r: r.name(lambda n: n.scopedname(lambda s: s.scope(  # noqa: E731: p::x
    lambda i: i.identifier(lambda d: d.spelling("p"))).name(lambda i: i.identifier(lambda d: d.spelling("x"))))))
for expression, message in [
    (lambda u: u.incrementexpression(lambda r: r.operator("++").operand(x)), "SystemVerilog incrementexpression is not evaluated"),
    (lambda u: u.assignmentpattern(lambda r: r.items([SV.literal(1)])), "SystemVerilog assignmentpattern is not evaluated"),
    (lambda u: u.macrousage(lambda r: r.name(lambda i: i.spelling("W"))), "SystemVerilog Preprocessor macrousage is not evaluated"),
    (lambda u: u.valuerange(lambda r: r.left(SV.literal(1)).right(SV.literal(2))), "SystemVerilog valuerange is not evaluated"),
    (lambda u: u.constant(lambda r: r.singleton("null")), "SystemVerilog constant null is not evaluated"),
    (lambda u: u.literal(lambda r: r.value(lambda p: p.as_scalar(10.0)).unit("ns")), "SystemVerilog time literal is not evaluated"),
    (scoped, "SystemVerilog nameexpression of a name that is not an identifier is not evaluated"),
    (lambda u: u.rangeselect(lambda r: r.value(v("1010")).left(SV.literal(0)).operator("+:").right(SV.literal(2))), "SystemVerilog rangeselect '+:' is not evaluated"),
    (SV.inside(1, lambda u: u.valuerange(lambda r: r.left(SV.literal(1)).operator("+/-").right(SV.literal(2)))), "SystemVerilog valuerange '+/-' is not evaluated"),
    (call("$display", 1), "SystemVerilog systemcall '$display' is not evaluated"),
    (lambda u: u.callexpression(lambda r: r.callee(scoped)), "SystemVerilog callexpression of a nameexpression is not evaluated"),
    (SV.method(x, "sum", 1), "SystemVerilog callexpression of a memberexpression is not evaluated"),
    (lambda u: u.arraymethodwithexpression(lambda r: r.value(SV.method(x, "unique")).body(SV.literal(1))),
     "SystemVerilog arraymethodwithexpression of a call that is not a reduction's is not evaluated"),
    (lambda u: u.arraymethodwithexpression(lambda r: r.value(x).body(SV.literal(1))),
     "SystemVerilog arraymethodwithexpression of a call that is not a reduction's is not evaluated"),
    (SV.cast(lambda t: t.structtype(lambda s: s.keyword("struct")), 1), "SystemVerilog Types structtype is not evaluated"),
    (c("realtime", 1), "SystemVerilog Types nonintegertype is not evaluated"),
    (SV.cast(lambda t: t.integervectortype(lambda n: n.keyword("logic").dimensions([
        lambda d: d.rangedimension(lambda r: r.left(SV.literal(7)).right(SV.literal(0)))])), 1), "SystemVerilog Types integervectortype is not evaluated"),
]:
    with raises(NotImplementedError, match=message):
        ev(expression, x=[1, 2])
with raises(ValueError, match="a cast's width must be positive, got 0"):  # a width known only as it is evaluated
    ev(c(b("-", 1, 1), 1))

<a id="DIA-15"></a>
## DIA-15 · SystemVerilog arrays: methods, reductions with a `with` clause, and `inside` over an array's elements

In [ ]:
xs, p, q = SV.identifier("xs"), SV.identifier("p"), SV.identifier("q")
assert svtext(SV.iterate(SV.member(xs, "pins"), "and", "p", SV.binary(">", p, 0))) == "xs.pins.and(p) with (p > 0)"
assert svtext(SV.select(SV.method(xs, "min"), 0)) == "xs.min()[0]" and svtext(SV.method(SV.binary("+", xs, 1), "size")) == "(xs + 1).size()"
assert svtext(SV.inside(1, xs, SV.span(2, 3))) == "1 inside {xs, [2:3]}"
assert validate(SV.method(xs, "pop"), {"xs"}) == [] and validate(SV.iterate(xs, "min", "p", p), {"xs"}) == []  # any method's
with raises(NotImplementedError, match="SystemVerilog callexpression of a memberexpression is not evaluated"):  # but these
    ev(SV.method(xs, "pop"), xs=[1])
with raises(NotImplementedError, match="SystemVerilog arraymethodwithexpression of a call that is not a reduction's is not evaluated"):
    ev(SV.iterate(xs, "min", "p", p), xs=[1])
assert validate(SV.iterate(xs, "and", "p", p), {"xs"}) == [] and SY.free(sv(SV.iterate(xs, "and", "p", SV.binary("&", p, q))), SV.DIALECT) == {"xs", "q"}
assert infer(SV.iterate(xs, "and", "p", p), {"xs": D.Anything}) is D.Anything
assert infer(SV.method(xs, "size"), {"xs": D.Anything}) is D.Anything

cases = [  # each over [1, 2, 3, 2], and its value
    (SV.method(xs, "size"), "32'sb" + "0" * 29 + "100"), (SV.method(xs, "sum"), "32'sb" + "0" * 28 + "1000"),
    (SV.method(xs, "product"), "32'sb" + "0" * 28 + "1100"), (SV.method(xs, "and"), "32'sb" + "0" * 32),
    (SV.method(xs, "or"), "32'sb" + "0" * 30 + "11"), (SV.method(xs, "xor"), "32'sb" + "0" * 30 + "10"),
    (SV.select(SV.method(xs, "min"), 0), "32'sb" + "0" * 31 + "1"), (SV.select(SV.method(xs, "max"), 0), "32'sb" + "0" * 30 + "11"),
    (SV.method(SV.method(xs, "unique"), "size"), "32'sb" + "0" * 30 + "11"),
    (SV.iterate(xs, "and", "p", SV.binary(">", p, 0)), "1'b1"), (SV.iterate(xs, "and", "p", SV.binary(">", p, 1)), "1'b0"),
    (SV.iterate(xs, "or", "p", SV.binary(">", p, 2)), "1'b1"), (SV.iterate(xs, "xor", "p", SV.binary("==", p, 2)), "1'b0"),
    (SV.iterate(xs, "sum", "p", SV.cast(atom("int"), SV.binary(">=", p, 2))), "32'sb" + "0" * 30 + "11"),
    (SV.iterate(xs, "sum", "p", SV.binary("*", p, p)), "32'sb" + "0" * 27 + "10010"),
    (SV.iterate(xs, "product", "p", SV.binary("+", p, 1)), "32'sb" + "0" * 25 + "1001000"),
    (SV.inside(3, xs), "1'b1"), (SV.inside(5, xs, 7), "1'b0"), (SV.inside(7, xs, 7), "1'b1"), (SV.inside(2, SV.span(0, 1), xs), "1'b1"),
]
for expression, value in cases:
    assert repr(ev(expression, xs=[1, 2, 3, 2])) == value, (svtext(expression), ev(expression, xs=[1, 2, 3, 2]))
# x and z reduce as the operators take them: and() is 0 if any value is, x if any is unknown and none 0.
assert repr(ev(SV.iterate(xs, "and", "p", p), xs=[SVD.Logic.of("1"), SVD.Logic.of("x")])) == "1'bx"
assert repr(ev(SV.iterate(xs, "and", "p", p), xs=[SVD.Logic.of("0"), SVD.Logic.of("x")])) == "1'b0"
assert repr(ev(SV.inside(SV.vector("1x"), xs), xs=[SVD.Logic.of("10")])) == "1'bx"
assert repr(ev(SV.method(SV.method(xs, "unique"), "size"), xs=[SVD.Logic.of("1x"), SVD.Logic.of("1x"), 1.0, 1, 1.0])) == "32'sb" + "0" * 30 + "11"
# Of no items, the reductions give their identities, and the locators no item.
assert [repr(ev(SV.method(xs, m), xs=[])) for m in ("sum", "product", "and", "or", "xor")] == [
    "32'sb" + "0" * 32, "32'sb" + "0" * 31 + "1", "1'b1", "1'b0", "1'b0"]
assert ev(SV.method(xs, "min"), xs=[]) == [] and repr(ev(SV.iterate(xs, "and", "p", SV.binary(">", p, 0)), xs=[])) == "1'b1"
assert ev(SV.method(xs, "sum"), xs=[1.5, 2]) == 3.5 and repr(ev(SV.select(SV.method(xs, "max"), 0), xs=[1.5, 2])) == "32'sb" + "0" * 30 + "10"
for (expression, variables), error, message in [
    ((SV.method(xs, "size"), {"xs": 5}), TypeError, "size() is a method of arrays, not of Logic"),
    ((SV.iterate(xs, "and", "p", p), {"xs": {"a": 1}}), TypeError, "and() is a method of arrays, not of dict"),
    ((SV.method(xs, "sum"), {"xs": [{"a": 1}]}), TypeError, "an array's elements must be numbers, got dict"),
    ((SV.inside(1, xs), {"xs": [{"a": 1}]}), TypeError, "an array's elements must be numbers, got dict"),
    ((SV.inside(1, xs), {"xs": {"a": 1}}), TypeError, "inside expects a number, got dict"),
    ((SV.method(xs, "and"), {"xs": [1.5, 2.5]}), TypeError, "& expects integral operands, got real"),
]:
    with raises(error, match=message):
        ev(expression, **variables)

<a id="DIA-16"></a>
## DIA-16 · MATLAB arrays: arrayfun, indexing from 1, and the functions of arrays

In [ ]:
xs, p = M.identifier("xs"), M.identifier("p")
assert mtext(M.call("all", M.arrayfun("p", M.field(xs, "pins"), M.binary(">", p, 0)))) == "all(arrayfun(@(p) p > 0, xs.pins))"
assert mtext(M.index(xs, M.binary("+", p, 1))) == "xs(p + 1)" and mtext(M.index(M.field(xs, "a"), 2)) == "xs.a(2)"
assert M.DIALECT.validate(mat(M.arrayfun("_p", xs, 1)), {"xs"}) == ["an arrayfun's parameter must be an identifier, got '_p'"]
assert M.DIALECT.validate(mat(M.arrayfun("p", xs, p)), {"xs"}) == [] and SY.free(mat(M.arrayfun("p", xs, M.binary("+", p, M.identifier("q")))), M.DIALECT) == {"xs", "q"}
assert M.DIALECT.validate(mat(M.call("numel", xs, 1)), {"xs"}) == ["numel takes 1 arguments, got 2"]
assert M.DIALECT.infer(mat(M.call("all", xs)), {"xs": D.Anything}).name() == "logical"
assert M.DIALECT.infer(mat(M.arrayfun("p", xs, p)), {"xs": D.Anything}) is D.Anything

assert mtext(M.array(1, xs, M.array())) == "[1, xs, []]" and M.DIALECT.validate(mat(M.array(1, xs)), {"xs"}) == []
assert MV.OfAny(mat(M.array(1, xs)), {"xs": [True, 2]}) == [1.0, 1.0, 2.0] and MV.OfAny(mat(M.array(True, False))) == [True, False]
assert MV.OfAny(mat(M.array("a", xs)), {"xs": ["b"]}) == ["a", "b"]  # a string array
with raises(TypeError, match="Concatenation takes numbers and logicals, or strings."):
    MV.OfAny(mat(M.array(1, "a")))

def alike(a, b):
    if type(b) is list:
        return type(a) is list and len(a) == len(b) and all(map(alike, a, b))
    return type(a) is type(b) and (a == b or (a != a and b != b))


VALUES = [
    (M.call("all", M.arrayfun("p", xs, M.binary(">", p, 0))), [1, 2, 3, 2], True), (M.call("any", M.arrayfun("p", xs, M.binary(">", p, 2))), [1, 2, 3, 2], True),
    (M.call("nnz", M.arrayfun("p", xs, M.binary(">=", p, 2))), [1, 2, 3, 2], 3.0), (M.call("numel", xs), [1, 2, 3, 2], 4.0),
    (M.index(xs, 2), [1, 2, 3, 2], 2.0), (M.index(xs, True), [1, 2, 3, 2], 1.0), (M.index(xs, False), [1, 2, 3, 2], []),
    (M.call("ismember", 3, xs), [1, 2, 3, 2], True), (M.call("ismember", "3", xs), [1, 2, 3, 2], True), (M.call("ismember", 5, xs), [1, 2, 3, 2], False),
    (M.call("sum", xs), [1, 2, 3, 2], 8.0), (M.call("min", xs), [1, 2, 3, 2], 1.0), (M.call("max", xs), [1, 2, 3, 2], 3.0),
    (M.call("numel", M.call("unique", xs)), [1, 2, 3, 2], 3.0), (M.arrayfun("p", xs, M.binary(".*", p, 2)), [1, 2, 3, 2], [2.0, 4.0, 6.0, 4.0]),
    (M.arrayfun("p", xs, M.binary(">", p, 1)), [1, 2, 3, 2], [False, True, True, True]), (M.arrayfun("p", xs, p), [True, 2], [1.0, 2.0]),
    (M.call("all", xs), [], True), (M.call("any", xs), [], False), (M.call("sum", xs), [], 0.0), (M.call("min", xs), [], []),
    (M.call("max", xs), [math.nan, 2], 2.0), (M.call("any", xs), [math.nan], False), (M.call("all", xs), [math.nan], True),
    (M.call("nnz", xs), [math.nan, 0], 1.0), (M.call("min", xs), [True, False], False), (M.call("sum", xs), [True, 2], 3.0),
    (M.call("unique", xs), ["b", "a", "b"], ["a", "b"]), (M.call("unique", xs), [True, True], [True]), (M.call("unique", xs), [3, 1, 3], [1.0, 3.0]),
    (M.call("numel", xs), 5, 1.0), (M.index(xs, 1), 5, 5.0), (M.arrayfun("p", xs, M.binary("+", p, 1)), 5, [6.0]),
    (M.call("ismember", 1, xs), [True], True), (M.call("numel", xs), "abc", 1.0), (M.call("numel", xs), {"a": 1}, 1.0),
]
ERRORS = [
    (M.index(xs, 0), [1, 2], ValueError, "Array indices must be positive integers or logical values."),
    (M.index(xs, 1.5), [1, 2], ValueError, "Array indices must be positive integers or logical values."),
    (M.index(xs, "a"), [1, 2], ValueError, "Array indices must be positive integers or logical values."),
    (M.index(xs, 5), [1, 2, 3, 2], ValueError, "Index exceeds the number of array elements. Index must not exceed 4."),
    (M.arrayfun("p", xs, "a"), [1, 2], ValueError, "Non-scalar in Uniform output, at index 1, output 1. Set 'UniformOutput' to false."),
    (M.arrayfun("p", xs, M.binary("+", xs, 0)), [1, 2], TypeError, "Operator '+' is not supported for operands of type 'array' and 'double'."),
    (M.arrayfun("p", xs, xs), [1, 2], ValueError, "Non-scalar in Uniform output, at index 1, output 1. Set 'UniformOutput' to false."),
    (M.call("sum", xs), ["a"], TypeError, "Invalid data type. First argument must be numeric or logical."),
    (M.call("all", xs), {"a": 1}, TypeError, "Invalid data type. First argument must be numeric or logical."),
    (M.call("unique", xs), [{"a": 1}], TypeError, "Invalid data type. First argument must be numeric or logical."),
    (M.call("ismember", xs, xs), [1], TypeError, "ismember takes a scalar and an array of scalars."),
    (M.binary("==", xs, 1), [1], TypeError, "Operator '==' is not supported for operands of type 'array' and 'double'."),
    (M.binary("<", xs, 1), {"a": 1}, TypeError, "Operator '<' is not supported for operands of type 'struct' and 'double'."),
]
for expression, value, expected in VALUES:
    result = MV.OfAny(mat(expression), {"xs": value})
    assert alike(result, expected), (mtext(expression), value, result)
for expression, value, error, message in ERRORS:
    with raises(error, match=message):
        MV.OfAny(mat(expression), {"xs": value})
assert math.isnan(MV.OfAny(mat(M.call("max", xs)), {"xs": [math.nan]})) and alike(MV.OfAny(mat(M.call("unique", xs)), {"xs": [math.nan, 1, math.nan]}), [1.0, math.nan, math.nan])
assert alike(MV.OfAny(mat(M.field(xs, "ranks")), {"xs": squad}), [1.0, 2.0, 3.0])  # a list property is an array of doubles
assert MV.OfAny(mat(M.call("sum", M.field(xs, "ranks"))), {"xs": squad}) == 6.0
with raises(ValueError, match="an arrayfun needs a value"):
    MV.OfAny(mat(M.arrayfun("p", None, 1)))

<a id="DIA-17"></a>
## DIA-17 · Excel arrays: MAP and LAMBDA, AND and OR of any number of arguments, and the functions of arrays

In [ ]:
xs, p = X.name("xs"), X.name("p")
assert xtext(X.function("AND", X.map_("p", X.field(xs, "pins"), X.infix(">", p, 0)))) == "=AND(MAP(xs.pins, LAMBDA(p, p > 0)))"
assert xtext(X.function("INDEX", xs, X.infix("+", p, 1))) == "=INDEX(xs, p + 1)"
assert X.DIALECT.validate(xl(X.map_("p", xs, p)), {"xs"}) == [] and SY.free(xl(X.map_("p", xs, X.infix("+", p, X.name("q")))), X.DIALECT) == {"xs", "q"}
assert X.DIALECT.validate(xl(X.function("AND", 1, 2, 3))) == [] and X.DIALECT.validate(xl(X.function("ROWS", xs, 1)), {"xs"}) == ["ROWS takes 1 arguments, got 2"]
assert X.DIALECT.infer(xl(X.function("AND", xs)), {"xs": D.Anything}).name() == "logical"
assert X.DIALECT.infer(xl(X.map_("p", xs, p)), {"xs": D.Anything}) is D.Anything
assert XV.OfAny(xl(X.function("AND", True, True, False))) is False and XV.OfAny(xl(X.function("OR", False, False, True))) is True
# IF takes 2 or 3 arguments, and AND, OR, SUM, MIN and MAX 1 to 255, as Excel's do.
assert X.DIALECT.validate(xl(X.function("IF", True, 1))) == [] and XV.OfAny(xl(X.function("IF", False, 1))) is False and XV.OfAny(xl(X.function("IF", True, 1))) == 1.0
assert X.DIALECT.validate(xl(X.function("IF", True))) == ["IF takes 2 or 3 arguments, got 1"] and X.DIALECT.validate(xl(X.function("SUM"))) == ["SUM takes 1 to 255 arguments, got 0"]
assert XV.OfAny(xl(X.function("SUM", 1, xs, True)), {"xs": [2, "a", True]}) == 4.0 and XV.OfAny(xl(X.function("MAX", xs, 7)), {"xs": [1, 2]}) == 7.0
assert XV.OfAny(xl(X.function("MIN", 3, X.name("e"))), {"e": XV.Error("#N/A")}) == XV.Error("#N/A")
for expression, message in [(X.function("IF", True, 1, 2, 3), "IF takes 2 or 3 arguments, got 4"), (X.function("AND"), "AND takes 1 to 255 arguments, got 0"),
                            (X.function("MIN"), "MIN takes 1 to 255 arguments, got 0")]:
    with raises(TypeError, match=message):
        XV.OfAny(xl(expression))
# VSTACK stacks its arguments into one array, an array's elements in turn, errors among them.
assert xtext(X.function("VSTACK", 1, xs)) == "=VSTACK(1, xs)" and X.DIALECT.validate(xl(X.function("VSTACK")), {"xs"}) == [
    "VSTACK takes 1 to 254 arguments, got 0"]
assert XV.OfAny(xl(X.function("VSTACK", 1, xs, X.name("e"))), {"xs": [2, "a"], "e": XV.Error("#N/A")}) == [1.0, 2, "a", XV.Error("#N/A")]
with raises(TypeError, match="VSTACK takes 1 to 254 arguments, got 0"):
    XV.OfAny(xl(X.function("VSTACK")))
VALUES = [
    (X.function("AND", X.map_("p", xs, X.infix(">", p, 0))), [1, 2, 3, 2], True), (X.function("OR", X.map_("p", xs, X.infix(">", p, 2))), [1, 2, 3, 2], True),
    (X.function("SUM", X.map_("p", xs, X.function("IF", X.infix(">=", p, 2), 1, 0))), [1, 2, 3, 2], 3.0), (X.function("ROWS", xs), [1, 2, 3, 2], 4.0),
    (X.function("INDEX", xs, 2), [1, 2, 3, 2], 2.0), (X.function("INDEX", xs, 2.9), [1, 2, 3, 2], 2.0), (X.function("INDEX", xs, 5), [1, 2], XV.Error("#REF!")),
    (X.function("INDEX", xs, 0), [1, 2], XV.Error("#REF!")), (X.function("INDEX", xs, "x"), [1, 2], XV.Error("#VALUE!")),
    (X.function("MATCH", 3, xs, 0), [1, 2, 3, 2], 3.0), (X.function("MATCH", "B", xs, 0), ["a", "b"], 2.0), (X.function("MATCH", 5, xs, 0), [1, 2], XV.Error("#N/A")),
    (X.function("MATCH", "1", xs, 0), [1, 2], XV.Error("#N/A")), (X.function("MATCH", 1, xs, 1), [1, 2], XV.Error("#VALUE!")),
    (X.function("MATCH", True, xs, 0), [1, True], 2.0), (X.function("ISNUMBER", X.function("MATCH", 2, xs, 0)), [1, 2], True),
    (X.function("ISNUMBER", X.function("MATCH", 7, xs, 0)), [1, 2], False), (X.function("ISNUMBER", "1"), [], False),
    (X.function("SUM", xs), [1, 2, 3, 2], 8.0), (X.function("SUM", xs), [1, "a", True, 2], 3.0), (X.function("SUM", xs), [1, XV.Error("#N/A")], XV.Error("#N/A")),
    (X.function("SUM", True), [], 1.0), (X.function("SUM", "2"), [], 2.0), (X.function("SUM", "a"), [], XV.Error("#VALUE!")),
    (X.function("MIN", xs), [3, 1, 2], 1.0), (X.function("MAX", xs), [3, 1, 2], 3.0), (X.function("MAX", xs), ["a"], 0.0), (X.function("MIN", xs), [], 0.0),
    (X.function("ROWS", X.function("UNIQUE", xs)), [1, 2, 3, 2], 3.0), (X.function("ROWS", X.function("UNIQUE", xs)), ["a", "A", 1, True, "1"], 4.0),
    (X.function("ROWS", X.function("UNIQUE", xs)), [XV.Error("#N/A"), XV.Error("#N/A"), {"a": 1}, {"a": 1}], 3.0),
    (X.function("ROWS", xs), 5, 1.0), (X.function("INDEX", xs, 1), 5, 5.0), (X.function("ROWS", X.map_("p", xs, X.infix("*", p, 2))), 5, 1.0),
    (X.function("AND", xs), [True, "a", 1], True), (X.function("AND", xs), ["a"], XV.Error("#VALUE!")), (X.function("OR", xs), [], XV.Error("#VALUE!")),
    (X.function("AND", xs, False), [True], False), (X.function("OR", xs), [0, XV.Error("#DIV/0!")], XV.Error("#DIV/0!")),
    (X.function("INDEX", X.map_("p", xs, X.infix("-", p, "a")), 1), [1], XV.Error("#VALUE!")), (X.function("ROWS", X.map_("p", xs, p)), XV.Error("#REF!"), XV.Error("#REF!")),
    (X.infix("+", xs, 1), [1], XV.Error("#VALUE!")), (X.infix("=", xs, 1), [1], XV.Error("#VALUE!")),
    (X.function("MATCH", 1, xs, "a"), [1], XV.Error("#VALUE!")), (X.function("UNIQUE", xs), XV.Error("#REF!"), XV.Error("#REF!")),
]
assert XV.OfAny(xl(X.function("INDEX", xs, X.name("n"))), {"xs": [1], "n": math.inf}) == XV.Error("#NUM!")
for expression, value, expected in VALUES:
    result = XV.OfAny(xl(expression), {"xs": value})
    assert result == expected and type(result) is type(expected), (xtext(expression), value, result)
assert XV.OfAny(xl(X.field(xs, "ranks")), {"xs": squad}) == [1.0, 2.0, 3.0]  # a list property is an array of numbers
assert XV.OfAny(xl(X.function("SUM", X.field(xs, "ranks"))), {"xs": squad}) == 6.0
assert XV.OfAny(xl(X.map_("p", xs, X.infix(">", p, 1))), {"xs": [1, 2]}) == [False, True]

<a id="DIA-18"></a>
## DIA-18 · A literal is a domain representation: printers write its spelling where it has one, else a default spelling; its value is what evaluates

In [ ]:
def spelled(dialect, value, text):
    """A literal of `dialect` holding `value`, spelled `text`, as a parser would write it."""
    return term(lambda u: u.literal(lambda r: (r.value(lambda v: v.as_scalar(value)) if value is not None else r).text(text)), dialect)


for dialect, printer, text, value, default in [
    (P.DIALECT, PText.ToText, "0x10", 16, "16"),
    (M.DIALECT, MText.ToText, "1e1", 10.0, "10.0"),
    (X.DIALECT, XText.ToText, "1E+1", 10.0, "=10.0"),
    (L.DIALECT, LText.ToText, "10.00", 10.0, "10.0"),
    (SV.DIALECT, SVText.ToText, "'d16", 16, "16"),
    (TS.DIALECT, TSText.ToText, "0x10", 16.0, "16"),
    (C.DIALECT, CText.ToText, "0x10", 16, "16"),
    (Cpp.Expressions.DIALECT, Cpp.Text.ToText, "0x10", 16, "16"),
]:
    literal = spelled(dialect, value, text)
    assert printer(literal) == ("=" + text if dialect is X.DIALECT else text) and dialect.validate(literal) == []
    assert printer(term(value, dialect)) == default  # no spelling: the value's default one
assert PText.ToText(term(P.binop("**", spelled(P.DIALECT, -2, "-2"), 2), P.DIALECT)) == "(-2) ** 2"  # a spelling's precedence
assert PText.ToText(term(P.unaryop("-", spelled(P.DIALECT, -1, "-1")), P.DIALECT)) == "--1"  # as its value's would be
assert MText.ToText(term(M.unary("-", spelled(M.DIALECT, -1, "-1")), M.DIALECT)) == "--1"
assert LText.ToText(term(L.binary("+", 1, spelled(L.DIALECT, -1, "-1")), L.DIALECT)) == "1 + -1"
assert PV.OfAny(spelled(P.DIALECT, 16, "0x10")) == 16 and MV.OfAny(spelled(M.DIALECT, 10.0, "1e1")) == 10.0  # its value
invalid = spelled(P.DIALECT, None, "08")
assert PText.ToText(invalid) == "08" and P.DIALECT.validate(invalid) == ["a literal needs a value"]  # kept as written

<a id="DIA-19"></a>
## DIA-19 · Python3's kinds beyond the narrower ones: literals keeping what their text encodes, f-strings as written, each kind's own checks and what it binds, stored as any term

In [ ]:
x, y = P.name("x"), P.name("y")  # Python3's names, variables


def checked(spec, bound=()):
    """The problems validation reports for a Python3 term."""
    return P.DIALECT.validate(py(spec), bound)


def shown(spec):
    """A Python3 term's source."""
    return PText.ToText(py(spec))


# A Python literal keeps what its text encodes beside its text, its problems and its value: a string's prefix and
# quotes, an int's base, digit separators; Python3's own constant holds None, ... and complex numbers, a complex number
# its `complex`, a value of `Expressions.Python.Python3.Complex`, whose `real` and `imag` are IEEE 754's binary64s.
assert F.form(parsed("0x_FF")).attributes == {"value": 255, "text": "0x_FF", "base": 16, "separators": True}
assert F.form(parsed("rb'\\d'")).attributes == {"value": b"\\d", "text": "rb'\\d'", "prefix": "rb", "quote": "'"}
assert F.form(parsed('"""x"""')).attributes == {"value": "x", "text": '"""x"""', "quote": '"""'}
assert F.form(parsed("1_0.5")).attributes == {"value": 10.5, "text": "1_0.5", "separators": True}
assert F.form(parsed("True")).attributes == {"value": True, "text": "True"}
assert F.form(parsed("1_0j")).attributes == {
    "complex": F.Part("complex", {"real": 0.0, "imag": 10.0}), "text": "1_0j", "separators": True}
written = F.form(py(P.constant(10j, "1_0j"))).attributes  # written with its spelling
assert written["text"] == "1_0j" and written["complex"] == F.Part("complex", {"real": 0.0, "imag": 10.0}, 0)
assert all(store.registered("Expressions.Python.Python3.Complex").properties[part].schema is store.registered(
    "Expressions.IEEE754.Expressions.binary64") for part in ("real", "imag"))
assert F.form(parsed("None")).attributes == {"singleton": "None", "text": "None"} and F.kind_of(parsed("..."))[0] == P.CONSTANT
assert shown(P.constant(16, "0x10")) == "0x10" and F.form(py(P.constant(None, "None"))).attributes == {
    "singleton": "None", "text": "None"}  # written with its spelling
concatenated = parsed("'a' 'b' f'{c}'")  # adjacent strings, each its own literal
assert [F.kind_of(value)[0] for value in F.form(concatenated).arguments] == ["literal", "literal", "joinedstr"]
assert PText.ToText(parsed("f'{f\"a\"}' 'b'")) == "f'{f\"a\"}' 'b'"  # an f-string within an f-string, among adjacent strings
fstring = F.form(parsed("f'a{x!r:>{w}}b{y = }'"))  # its text and fields as written
assert fstring.attributes["prefix"] == "f" and [part.kind for part in fstring.attributes["values"]] == [
    "stringtext", "formattedvalue", "stringtext", "formattedvalue"]
field = fstring.attributes["values"][1].attributes
assert field["text"] == "x" and field["conversion"] == "r" and [p.kind for p in field["format_spec"].attributes["values"]] == [
    "stringtext", "formattedvalue"]
assert fstring.attributes["values"][3].attributes == {"text": "y = ", "debug": True}
assert shown(P.joinedstr(("a", P.formattedvalue(x, debug=True), P.formattedvalue(P.set_(1))))) == "f'a{x=}{ {1}}'"  # unspelled
assert shown(P.joinedstr((P.formattedvalue(x, text="x = "),))) == "f'{x = }'"  # a field's expression as written
assert shown(P.templatestr((P.interpolation(x, "s", ("0", P.formattedvalue(y))),), quote='"')) == 't"{x!s:0{y}}"'  # a format spec's fields are formatted values, as `ast` has them
for spec, problems in [  # each new kind's own checks
    (lambda u: getattr(u, P.CONSTANT)(lambda r: r), ["a constant needs a singleton or a complex number"]),
    (lambda u: getattr(u, P.CONSTANT)(lambda r: r.singleton("None").complex(lambda c: c.imag(1.0))),
     ["a constant holds a singleton or a complex number, not both"]),
    (lambda u: getattr(u, P.CONSTANT)(lambda r: r.singleton("Ellipsis")), ["a constant's singleton must be None or ..., got 'Ellipsis'"]),
    (lambda u: getattr(u, P.CONSTANT)(lambda r: r.complex(lambda c: c.real(1.0))), ["a complex constant needs an imag"]),
    (lambda u: getattr(u, P.CONSTANT)(lambda r: r.text("1__0j").problems(["invalid decimal literal"])), ["invalid decimal literal"]),
    (lambda u: getattr(u, P.LITERAL)(lambda r: r.value(lambda v: v.as_scalar("x")).prefix("q").quote("`").base(3)),
     ["a literal's prefix cannot be 'q'", "a literal's quote cannot be '`'", "a literal's base cannot be 3"]),
    (P.joinedstr(("x",), prefix="b"), ["a joinedstr's prefix cannot be 'b'"]),
    (P.templatestr(("x",), prefix="Rt", quote="`"), ["a templatestr's quote cannot be '`'"]),
    (P.joinedstr((P.formattedvalue(1, "q", ("0", P.formattedvalue(2, "z"))),)),
     ["a formattedvalue's conversion cannot be 'q'", "a formattedvalue's conversion cannot be 'z'"]),  # its spec's fields too
    (P.concatenatedstring("a"), ["a concatenatedstring needs two strings"]),
    (P.set_(), ["a set needs an element"]),
    (P.boolop("and", 1), ["a boolop needs two values"]),
    (P.listcomp(1), ["a listcomp needs a for clause"]),
    (P.call(P.name("f"), P.keyword("a b", 1)), ["a keyword's arg must be an identifier, got 'a b'"]),
    (lambda u: getattr(u, P.COMPARE)(lambda r: r.left(P.constant(1)).comparisons([
        lambda a: a.as_object(lambda o: o.comparator(P.constant(2)))])), ["a comparison needs an op"]),
    (P.namedexpr(P.attribute(P.name("o"), "a"), 1), ["a namedexpr's target must be a name"]),
    (P.lambda_(1, (P.arg("a", annotation=P.name("int")),)), ["a lambda's parameters have no annotations"]),
    (P.lambda_(1, ("a b",)), ["a lambda's parameter must be an identifier, got 'a b'"]),
    (lambda u: getattr(u, P.LAMBDA)(lambda r: r.body(P.constant(1)).args(lambda a: a.args([
        lambda e: e.as_object(lambda o: o.default_value(P.constant(1)))]))), ["a lambda's parameter needs an arg"]),
    (lambda u: getattr(u, P.LISTCOMP)(lambda r: r.body(P.constant(1)).generators([
        lambda a: a.as_object(lambda o: o.iter(P.name("xs")))])), ["a listcomp needs a generators 0 target"]),
    (P.lambda_(1), []), (P.yield_(), []), (P.dictcomp(1, None, P.comprehension("x", P.name("xs"))), []),  # optional slots
    (P.lambda_(P.binop("+", P.name("a"), P.name("k")), ("a",), vararg="v", kwonlyargs=("k",), kwarg="w"), []),
]:
    assert checked(spec, {"f", "xs", "o"}) == problems, (spec, checked(spec, {"f", "xs", "o"}))
# What each new kind binds: a comprehension its targets, a lambda its parameters, a named expression its target alone
for spec, free in [
    (P.listcomp(P.name("q"), P.comprehension(P.tuple_(P.name("p"), P.starred(P.name("q"))), P.name("xs"), (P.name("p"),))), {"xs"}),
    (P.listcomp(1, P.comprehension(P.attribute(P.name("o"), "a"), P.name("xs"))), {"o", "xs"}),  # an attribute binds nothing
    (P.dictcomp(P.name("k"), P.name("v"), P.comprehension(P.parenthesized(P.list_(P.name("k"), P.name("v"))), P.name("d"))), {"d"}),
    (P.setcomp(P.name("q"), P.comprehension("p", P.name("ps")), P.comprehension("q", P.name("p"))), {"ps"}),  # later clauses see earlier
    (P.lambda_(P.binop("+", P.name("a"), P.name("b")), ("a", P.arg("b", P.name("a")))), {"a"}),  # a default: the scope's
    (P.boolop("and", P.namedexpr("y", 1), P.name("y")), {"y"}),  # beyond itself, the caller binds a named expression's target
    (lambda u: getattr(u, P.LISTCOMP)(lambda r: r.body(P.name("e")).generators([
        lambda a: a.as_object(lambda o: o.iter(P.name("xs")))])), {"e", "xs"}),  # no target: nothing bound
]:
    assert SY.free(py(spec), P.DIALECT) == free, (spec, SY.free(py(spec), P.DIALECT))
# Printed as held, what the writers would not write: a complex number of no real part, a parameter of no name
assert shown(lambda u: getattr(u, P.CONSTANT)(lambda r: r.complex(lambda c: c.imag(2.0)))) == "2j"
assert shown(lambda u: getattr(u, P.LAMBDA)(lambda r: r.body(P.constant(1)).args(lambda a: a.args([
    lambda e: e.as_object(lambda o: o.default_value(P.constant(1)))])))) == "lambda =1: 1"
spelled = py(lambda u: getattr(u, P.LITERAL)(lambda r: r.value(lambda v: v.as_scalar(255)).text("0x_FF").base(16).separators(True)))
assert F.form(spelled).attributes == {"value": 255, "text": "0x_FF", "base": 16, "separators": True} and PText.ToText(spelled) == "0x_FF"
# A union of helpers holding none (an f-string's value with no branch written) holds nothing, read or printed
empty = py(lambda u: getattr(u, P.JOINEDSTR)(lambda r: r.values([lambda a: a.as_union(lambda v: v)])))
assert "values" not in F.form(empty).attributes and PText.ToText(empty) == "f''"
# Each new kind is stored as any term, its helpers nested in it: every kind of FRM's sample round-trips; here a few
for spec in [P.lambda_(1, (P.arg("a", 2),), posonlyargs=("p",)), P.joinedstr(("a", P.formattedvalue(x, "r", ("0",)))),
             P.call("f", P.keyword(None, x)), P.compare("<", 1, 2, ("in", x))]:
    assert round_trips(py(spec)) and F.same(py(P.DIALECT.make(F.form(py(spec)))), py(spec))
# Its patterns: a binding target, a comprehension's or a named expression's, holds `Expressions.Python.Python3.Pattern`, the
# union of the terms a target may be, each a schema of a term too, and invalid syntax ("(ii)"); a form reads a target as
# a part, its terms after the clause's own
Pattern = store.registered("Expressions.Python.Python3.Pattern")
assert [b.name for b in Pattern.branches] == ["variable", "tuple", "list", "starred", "attribute", "subscript", "parenthesized", "error", "missing"]
assert {b.name: b.schema for b in Pattern.branches}["tuple"] is store.registered("Expressions.Python.Python3.Tuple")  # a term's schema
assert all(store.registered(f"Expressions.Python.Python3.{name}").properties["target"].schema is Pattern for name in ("Comprehension", "NamedExpr"))
generator = py(P.generatorexp(P.name("p"), P.comprehension(P.tuple_(P.name("p"), P.starred(P.name("q"))), P.name("ps"))))
clause = F.form(generator).attributes["generators"][0]
assert clause.attributes["target"].kind == "tuple" and clause.attributes["target"].size == 2 and clause.size == 3
assert P.target_names(clause.attributes["target"], F.form(generator).arguments[3:]) == {"p", "q"} and P.target_names(None) == set()
assert round_trips(generator) and F.same(py(P.DIALECT.make(F.form(generator))), generator) and PText.ToText(generator) == "(p for p, *q in ps)"
with raises(Exception, match="'binop' is not a member of a union value"):  # a kind no target is
    py(P.generatorexp(1, P.comprehension(P.binop("+", P.name("a"), 1), P.name("ps"))))
for spec, problems in [  # a target's own checks, as its kind's
    (lambda u: getattr(u, P.NAMEDEXPR)(lambda r: r.value(P.constant(1))), ["a namedexpr needs a target"]),
    (P.generatorexp(1, P.comprehension(P.attribute(P.name("o"), "a b"), P.name("ps"))),
     ["generators 0 target: an attribute must be an identifier, got 'a b'"]),
    (lambda u: getattr(u, P.GENERATOREXP)(lambda r: r.body(P.constant(1)).generators([lambda a: a.as_object(
        lambda o: o.target(lambda t: t.variable(lambda v: v)).iter(P.name("ps")))])), ["generators 0 target: a variable needs a name"]),
]:
    assert checked(spec, {"ps", "o"}) == problems, checked(spec, {"ps", "o"})
assert shown(lambda u: getattr(u, P.LISTCOMP)(lambda r: r.body(P.constant(1)).generators([
    lambda a: a.as_object(lambda o: o.iter(P.name("xs")))]))) == "[1 for  in xs]"  # no target, as held
assert shown(P.listcomp(1, P.comprehension(P.starred(P.name("q")), P.name("xs")))) == "[1 for (*q) in xs]"  # by its precedence

<a id="DIA-20"></a>
## DIA-20 · Python3 is complete, and its sublanguages' dialects: each node of `ast` an expression is or holds, 3.0 to 3.15, and each type parameter and pattern node, kept as data, is a kind, a helper or an operator of a kind, or is listed as not an expression with its reason

In [ ]:
import ast
import json
import sys
from pathlib import Path

grammar = json.loads(Path("../../grammars/python3-ast.json").read_text(encoding="utf-8"))  # `ast`'s nodes, as data
PT, PP = PythonTypes.Expressions.DIALECT, PythonPatterns.Expressions.DIALECT
DIALECT_OF = {"type_param": PT, "pattern": PP}  # a sublanguage's nodes, its dialect's kinds; every other node, Python3's
kinds, mapped = P.DIALECT.kinds(), {P.DIALECT: set(), PT: set(), PP: set()}
for node in grammar["nodes"]:  # each a kind, a helper, an operator of a kind, or not an expression, with its reason
    ways = [way for way in ("kinds", "helper", "op", "not") if way in node]
    assert len(ways) == 1, node
    if "kinds" in node:
        dialect = DIALECT_OF.get(node["class"], P.DIALECT)
        assert node["kinds"] and all(kind in dialect.kinds() for kind in node["kinds"]), node
        mapped[dialect] |= set(node["kinds"])
    elif "helper" in node:  # a value object a kind holds, not a term
        helper = store.registered(f"Expressions.Python.Python3.{node['helper']}")
        assert helper is not None and not helper.ref and node["helper"].lower() not in kinds, node
    elif "op" in node:
        assert node["op"] in kinds[node["kind"]].vocabulary, node
    else:
        assert type(node["not"]) is str and node["not"], node
concrete = {entry["kind"] for entry in grammar["concrete"]}  # what `ast` does not keep: parentheses
assert set(kinds) == mapped[P.DIALECT] | concrete | P.EXTENSIONS | {"error", "missing"}  # every kind is `ast`'s, but these
assert all(set(dialect.kinds()) == mapped[dialect] | {"error", "missing"} for dialect in (PT, PP))  # each sublanguage's, exactly
for kind in ("boolop", "binop", "unaryop", "compare"):  # every operator of the vocabularies is `ast`'s, and the other way
    assert set(kinds[kind].vocabulary) == {node["op"] for node in grammar["nodes"] if node.get("kind") == kind}, kind
# The running Python's `ast` against the data (the Python implementation only: `ast` is Python's): every node of an
# expression, a type parameter or a pattern it defines is listed, for its version, and every one listed it defines.
version = sys.version_info[:2]


def listed_in(node):
    """Whether this Python's `ast` has the node: from its `since`, to its `until` where it was removed."""
    since = tuple(map(int, node["since"].split(".")))
    until = tuple(map(int, node["until"].split("."))) if "until" in node else None
    return since <= version and (until is None or version <= until)


BASES = ("expr", "expr_context", "boolop", "operator", "unaryop", "cmpop", "type_param", "pattern")
live = {(c.__name__, base) for base in BASES for c in getattr(ast, base).__subclasses__()}
live |= {(name, "helper") for name in ("comprehension", "arguments", "arg", "keyword", "alias", "match_case")}
listed = {(node["node"], node["class"]) for node in grammar["nodes"] if listed_in(node)}
assert live <= listed, live - listed
for base, count in [("expr", 29), ("type_param", 3), ("pattern", 8)]:  # 3.14 and 3.15: the same
    assert {n for n in listed if n[1] == base} == {n for n in live if n[1] == base} and len({n for n in live if n[1] == base}) == count, base

<a id="DIA-21"></a>
## DIA-21 · TypeScript5: typescript-estree's expressions in mbse-programs' shapes, its types and patterns each a union of its own; literals keeping what their text encodes; each kind's checks and bindings; evaluation of literals alone

In [ ]:
def ts(spec):
    """A TypeScript5 term, written through the builders (TypeScript5 has no writers), built in a transaction."""
    return term(spec, TS.DIALECT)


def ts_checked(spec, bound=()):
    """The problems validation reports for a TypeScript5 term."""
    return TS.DIALECT.validate(ts(spec), bound)


# Its union, and its sublanguages, each a dialect of its own, `TypeScript Types` and `TypeScript Patterns`, whose `Term`s
# were its `Type` and `Pattern` unions, their branches unchanged; a schema is a branch of every union whose positions
# hold it ("(b) as you recommended"), named alike in each, and TypeScript5's `Term` lists three patterns.
unions = {name: store.registered(f"Expressions.TypeScript.{path}.Term") for name, path in (
    ("Term", "TypeScript5"), ("Pattern", "Patterns"), ("Type", "Types"))}
branches = {name: {b.name: b.schema for b in union.branches} for name, union in unions.items()}
assert [tag for tag in branches["Term"] if tag not in TSP.TAGS] == list(TS.DIALECT.kinds()) and len(branches["Term"]) == 46
assert list(branches["Pattern"]) == ["arraypattern", "objectpattern", "assignmentpattern", "restelement", "identifier",
                                     "memberexpression", "parenthesizedexpression", "tsasexpression",
                                     "tssatisfiesexpression", "tstypeassertion", "tsnonnullexpression", "property",
                                     "elision", "tsparameterproperty", "error", "missing"]
assert len([tag for tag in branches["Type"] if tag.startswith("ts")]) == 37  # mbse-programs' 37: typescript-estree's 36, and parentheses
assert list(branches["Type"])[37:] == ["identifier", "thisexpression", "literal", "unaryexpression", "templateliteral",
                                       "objectexpression", "error", "missing"]  # the terms a type holds, and invalid syntax
assert branches["Pattern"]["identifier"] is branches["Term"]["identifier"] is branches["Type"]["identifier"]
assert branches["Term"]["arraypattern"] is branches["Pattern"]["arraypattern"] is store.registered(
    "Expressions.TypeScript.Patterns.ArrayPattern")  # a moved schema, named by its dialect
assert branches["Term"]["arrayexpression"] is store.registered("Expressions.Framework.List")  # the shape matches exactly
assert branches["Term"]["literal"].refines is store.registered("Expressions.Framework.Literal")
assert store.registered("Expressions.TypeScript.TypeScript5.Expression").properties["root"].schema is unions["Term"]
assert branches["Term"]["assignmentexpression"].properties["left"].schema is unions["Pattern"]  # what holds a pattern
assert branches["Term"]["tsasexpression"].properties["typeAnnotation"].schema is unions["Type"]  # and a type
assert store.registered("Expressions.TypeScript.Types.TSQualifiedName").properties["right"].schema is branches["Term"]["identifier"]
# A sublanguage's value is a term of its dialect, read as an argument, its patterns and types among an arrow's:
# x = ([a, b = 1]: number[]) => a as string
arrow = ts(lambda u: u.assignmentexpression(lambda r: r.left(lambda t: t.identifier(lambda i: i.name("x"))).operator("=").right(
    lambda v: v.arrowfunctionexpression(lambda a: a.params([lambda p: p.arraypattern(lambda ap: ap.elements([
        lambda e: e.identifier(lambda i: i.name("a")),
        lambda e: e.assignmentpattern(lambda s: s.left(lambda t: t.identifier(lambda i: i.name("b"))).right(
            lambda t: t.literal(lambda li: li.value(lambda n: n.as_scalar(1.0)))))]).typeAnnotation(
        lambda ta: ta.typeAnnotation(lambda t: t.tsarraytype(lambda at: at.elementType(lambda y: y.tsnumberkeyword(lambda k: k))))))])
        .body(lambda b: b.tsasexpression(lambda s: s.expression(lambda e: e.identifier(lambda i: i.name("a"))).typeAnnotation(
            lambda t: t.tsstringkeyword(lambda k: k))))))))
form = F.form(arrow)
assert form.kind == "assignmentexpression" and form.attributes == {"operator": "="}  # its target, a pattern, an argument
assert F.dialect_of(form.arguments[0]) == "Expressions.TypeScript.Patterns.Term" and F.form(form.arguments[0]).attributes == {"name": "x"}
lambda_ = F.form(form.arguments[1])  # Let's `value`, unset, its body, and its parameter
assert lambda_.kind == "arrowfunctionexpression" and lambda_.arguments[0] is None and len(lambda_.arguments) == 3
pattern = F.form(lambda_.arguments[2])
assert pattern.kind == "arraypattern" and pattern.attributes == {  # its annotation a helper; its decorators and elements
    "typeAnnotation": F.Part("tstypeannotation", {}, 1), "decorators": F.Part("decorators", {}, 0),  # containers of
    "elements": F.Part("elements", {}, 2)}  # terms, each its size
assert [F.form(e).kind for e in pattern.arguments[:2]] == ["identifier", "assignmentpattern"]
assert F.dialect_of(pattern.arguments[2]) == "Expressions.TypeScript.Types.Term" and F.form(pattern.arguments[2]).kind == "tsarraytype"
node = TS.tree(lambda_.arguments[2])  # its helpers nested, as the printer and the checks read it
assert node.kind == "arraypattern" and node.lists == {"decorators": (), "elements": pattern.arguments[:2]}
assert TS.tree(node.helpers["typeAnnotation"].slots["typeAnnotation"]).kind == "tsarraytype"
assert TS.bound_names(node) == {"a", "b"} and TS.bound_names(None) == frozenset() and TSP.bound_names is TS.bound_names
assert TS.DIALECT.validate(arrow) == ["left: identifier 'x' is not bound"]  # an assignment's target, a reference
assert TS.DIALECT.validate(arrow, {"x"}) == [] and round_trips(arrow) and F.same(ts(TS.DIALECT.make(form)), arrow)
assert SY.free(arrow, TS.DIALECT) == {"x"}  # its parameters bind its body
# A literal keeps what its text encodes beside its text, its problems and its value: a number is a float, a bigint an
# int; TypeScript5's own constant holds null and a regular expression, which no schema holds.
for spec, attributes, domain in [
    (lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(255.0)).text("0x_FF").base(16).separators(True)),
     {"value": 255.0, "text": "0x_FF", "base": 16, "separators": True}, "number"),
    (lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(1000)).text("1_000n").separators(True)),
     {"value": 1000, "text": "1_000n", "separators": True}, "bigint"),
    (lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar("a")).text("'a'").quote("'")),
     {"value": "a", "text": "'a'", "quote": "'"}, "string"),
    (lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(True)).text("true")), {"value": True, "text": "true"}, "boolean"),
]:
    spelled = ts(spec)
    assert F.form(spelled).attributes == attributes and TS.DIALECT.validate(spelled) == []
    assert TS.DIALECT.infer(spelled).name() == domain and TV.OfAny(spelled) == attributes["value"]  # its value
null = ts(lambda u: u.constant(lambda r: r.singleton("null").text("null")))
regex = ts(lambda u: u.constant(lambda r: r.pattern("a+").flags("gu")))
assert F.form(regex).attributes == {"pattern": "a+", "flags": "gu"} and TS.DIALECT.validate(regex) == TS.DIALECT.validate(null) == []
assert TS.DIALECT.infer(null).name() == "null" and TS.DIALECT.infer(regex).name() == "RegExp"
assert TS.DIALECT.kinds()["literal"].spelling == ("text", "problems", "quote", "base", "separators")
with raises(TypeError, match="expected a term, a writer or a native value, got b'x'"):  # no literal holds bytes
    TS.DIALECT.spec(b"x")
assert F.form(ts(1)).attributes == {"value": 1} and F.form(ts(1.5)).attributes == {"value": 1.5}  # a bigint, a number
# Evaluation is shallow: a literal's value; every other kind refused by name, a constant among them
for spec, message in [(lambda u: u.thisexpression(lambda r: r), "TypeScript5 thisexpression is not evaluated"),
                      (lambda u: u.constant(lambda r: r.singleton("null")), "TypeScript5 constant is not evaluated")]:
    with raises(NotImplementedError, match=message):
        TV.OfAny(ts(spec))
with raises(NotImplementedError, match="TypeScript5 assignmentexpression is not evaluated"):
    TV.OfAny(arrow)
assert TV.EVALUATED == {"literal"}
# Each kind's own checks, mbse-programs', and those of the types, patterns and JSX it holds
x = lambda e: e.identifier(lambda i: i.name("x"))  # noqa: E731: a spec of `x`, in any union that has identifiers
for spec, problems in [
    (lambda u: u.identifier(lambda r: r.name("a b")), ["an identifier's name cannot be 'a b'"]),
    (lambda u: u.identifier(lambda r: r.name("$a")), []),  # `$` is a letter
    (lambda u: u.privateidentifier(lambda r: r.name("1")), ["a privateidentifier's name cannot be '1'"]),
    (lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar("s")).quote("`").base(3)),
     ["a literal's quote cannot be '`'", "a literal's base cannot be 3"]),
    (lambda u: u.literal(lambda r: r.text("08n").problems(["a bigint has no leading zeros"])), ["a bigint has no leading zeros"]),
    (lambda u: u.constant(lambda r: r), ["a constant needs a singleton or a regular expression"]),
    (lambda u: u.constant(lambda r: r.text("/a").problems(["unterminated regular expression"])), ["unterminated regular expression"]),
    (lambda u: u.constant(lambda r: r.singleton("null").pattern("a")), ["a constant holds a singleton or a regular expression, not both"]),
    (lambda u: u.constant(lambda r: r.singleton("undefined")), ["a constant's singleton must be null, got 'undefined'"]),
    (lambda u: u.constant(lambda r: r.flags("g")), ["a regular expression needs a pattern"]),
    (lambda u: u.constant(lambda r: r.pattern("a")), []),  # its flags, none
    (lambda u: u.constant(lambda r: r.pattern("a").flags("gq")), ["a regular expression's flags cannot be 'gq'"]),
    (lambda u: u.constant(lambda r: r.pattern("a").flags("gg")), ["a regular expression's flags cannot be 'gg'"]),
    (lambda u: u.templateliteral(lambda r: r.quasis([lambda q: q.raw("a")]).expressions([x])),
     ["a templateliteral needs one more quasi than expressions"]),
    (lambda u: u.sequenceexpression(lambda r: r.expressions([x])), ["a sequenceexpression needs two expressions"]),
    (lambda u: u.tsasexpression(lambda r: r.expression(x).typeAnnotation(lambda t: t.tsuniontype(lambda y: y.types([
        lambda z: z.tsintersectiontype(lambda w: w.types([lambda v: v.tsanykeyword(lambda k: k)]))])))),
     ["typeAnnotation: a tsuniontype needs two types", "typeAnnotation: argument 0: a tsintersectiontype needs two types"]),  # a type's, a term
    (lambda u: u.tsasexpression(lambda r: r.expression(x).typeAnnotation(lambda t: t.tstemplateliteraltype(lambda y: y.quasis([
        lambda q: q.raw("a")]).types([lambda v: v.tsstringkeyword(lambda k: k)])))),
     ["typeAnnotation: a tstemplateliteraltype needs one more quasi than types"]),
    (lambda u: u.arrowfunctionexpression(lambda r: r.typeParameters(lambda d: d.params([]))), ["a tstypeparameterdeclaration needs a type parameter"]),
    (lambda u: u.arrowfunctionexpression(lambda r: r.params([lambda p: p.identifier(lambda i: i.name("a b"))])),
     ["argument 0: an identifier's name cannot be 'a b'"]),  # a pattern's, a term
    (lambda u: u.binaryexpression(lambda r: r.left(x).operator("<>").right(x)), []),  # an operator outside its vocabulary
    (lambda u: u.binaryexpression(lambda r: r.left(x).right(x)), ["a binaryexpression needs an operator"]),
    (lambda u: u.tsasexpression(lambda r: r.expression(x)), ["a tsasexpression needs a typeAnnotation"]),  # a type it needs
    (lambda u: u.assignmentexpression(lambda r: r.right(x)), ["an assignmentexpression needs an operator", "an assignmentexpression needs a left"]),
    (lambda u: u.unaryexpression(lambda r: r.operator("-")), ["an unaryexpression needs an argument"]),
    (lambda u: u.yieldexpression(lambda r: r), []), (lambda u: u.importexpression(lambda r: r.source(x)), []),  # optional slots
    (lambda u: u.arrowfunctionexpression(lambda r: r), []),  # a body of statements, not part of the term
    (lambda u: u.classexpression(lambda r: r), []),
    (lambda u: u.assignmentexpression(lambda r: r.left(lambda t: t.assignmentpattern(lambda a: a.left(x))).operator("=").right(x)),
     ["left: an assignmentpattern needs a right"]),  # a pattern's term, labelled by its slot
]:
    assert ts_checked(spec, {"x", "$a"}) == problems, (ts_checked(spec, {"x", "$a"}), problems)
assert TS.DIALECT.validate(ts(lambda u: u.binaryexpression(lambda r: r.left(x).operator("<>").right(x))), {"x"}, core=True) == [
    "'<>' is not a core operation"]
# What each kind binds: a function's and an arrow's parameters; a member's property and a key, not computed, are names
# rather than references
for spec, free in [
    (lambda u: u.memberexpression(lambda r: r.object(x).property(lambda p: p.identifier(lambda i: i.name("y")))), {"x"}),
    (lambda u: u.memberexpression(lambda r: r.object(x).computed(True).property(lambda p: p.identifier(lambda i: i.name("y")))), {"x", "y"}),
    (lambda u: u.objectexpression(lambda r: r.properties([lambda p: p.property(lambda q: q.key(lambda k: k.identifier(lambda i: i.name("k")))
        .value(lambda v: v.literal(lambda li: li.value(lambda n: n.as_scalar("s"))))), lambda p: p.property(lambda q: q.computed(True)
        .key(x).value(x)), lambda p: p.property(lambda q: q.key(lambda k: k.literal(lambda li: li.value(lambda n: n.as_scalar("s")))).value(x)),
        lambda p: p.property(lambda q: q.value(x))])), {"x"}),  # a key that is a literal, or none, names nothing
    (lambda u: u.tsasexpression(lambda r: r.expression(x).typeAnnotation(lambda t: t.tstypeliteral(lambda y: y.members([
        lambda m: m.tspropertysignature(lambda s: s.key(lambda k: k.identifier(lambda i: i.name("k")))),
        lambda m: m.tsmethodsignature(lambda s: s.computed(True).key(lambda k: k.identifier(lambda i: i.name("z")))),
        lambda m: m.comment(lambda c: c.text(" c"))])))), {"x", "z"}),  # a type literal's keys, within a type
    (lambda u: u.functionexpression(lambda r: r.params([lambda p: p.objectpattern(lambda o: o.properties([
        lambda q: q.property(lambda s: s.key(lambda k: k.identifier(lambda i: i.name("a"))).value(lambda v: v.identifier(lambda i: i.name("b")))),
        lambda q: q.restelement(lambda s: s.argument(lambda a: a.identifier(lambda i: i.name("c"))))])),
        lambda p: p.tsparameterproperty(lambda s: s.parameter(lambda a: a.assignmentpattern(lambda t: t.left(lambda i: i.identifier(lambda j: j.name("d"))).right(
            lambda e: e.identifier(lambda i: i.name("b")))))),
        lambda p: p.memberexpression(lambda m: m.object(x).property(lambda y: y.identifier(lambda i: i.name("y"))))])), {"x"}),
    (lambda u: u.arrowfunctionexpression(lambda r: r.params([lambda p: p.identifier(lambda i: i)]).body(x)), {"x"}),  # a name not written binds none
]:
    assert SY.free(ts(spec), TS.DIALECT) == free, (SY.free(ts(spec), TS.DIALECT), free)
# Empty value objects are held and read back present (mbse-schemas 0.26.9): a fragment's opening and closing, an empty
# template, an empty quasi, each printed as written
for spec, text, held in [
    (lambda u: u.jsxfragment(lambda f: f.openingFragment(lambda h: h).closingFragment(lambda h: h)), "<></>", ["openingFragment", "closingFragment"]),
    (lambda u: u.taggedtemplateexpression(lambda r: r.tag(x).quasi(lambda q: q.quasis([lambda e: e]))), "x``", ["quasi"]),
    (lambda u: u.templateliteral(lambda r: r.quasis([lambda e: e])), "``", ["quasis"]),
]:
    value = ts(spec)
    assert round_trips(value) and TSText.ToText(value) == text and list(F.form(value).attributes) == held, text

<a id="DIA-22"></a>
## DIA-22 · TypeScript5 printed as TypeScript writes it: every kind, its types, patterns and JSX, by precedence, as an expression statement holds it, a literal in its spelling or its value's

In [ ]:
def shown(spec):
    """A TypeScript5 term's source."""
    return TSText.ToText(ts(spec))


# Specs of terms, types and patterns, written through the builders, each in any union whose branches it names
x, y, z = (lambda e, n=n: e.identifier(lambda i: i.name(n)) for n in "xyz")
one = lambda e: e.literal(lambda r: r.value(lambda n: n.as_scalar(1.0)))  # noqa: E731
number = lambda t: t.tsnumberkeyword(lambda k: k)  # noqa: E731
string = lambda t: t.tsstringkeyword(lambda k: k)  # noqa: E731
T_, U_ = (lambda t, n=n: t.tstypereference(lambda r: r.typeName(lambda m: m.identifier(lambda i: i.name(n)))) for n in "TU")
empty = lambda e: e.objectexpression(lambda r: r)  # noqa: E731: `{}`
# A number as JavaScript's `String(number)` writes it, a string in its quote with its escapes
assert [TSText.number(v) for v in (math.nan, math.inf, -math.inf, 0.0, -0.0, 100.0, 1.5, 0.001, 1e21, 1.5e-7, 5e-7, -2.5,
                                   123456789012345680000.0, 1e-6)] == [
    "NaN", "Infinity", "-Infinity", "0", "0", "100", "1.5", "0.001", "1e+21", "1.5e-7", "5e-7", "-2.5",
    "123456789012345680000", "0.000001"]
assert TSText.string("it's \"x\"\\\n\x01\x7f é", '"') == '"it\'s \\"x\\"\\\\\\n\\x01\\x7f\\u2028é"'
assert TSText.string("it's", "'") == "'it\\'s'"
for spec, text in [
    # literals: a value's default spelling, else its text
    (lambda u: u.literal(lambda r: r.value(lambda n: n.as_scalar(True))), "true"),
    (lambda u: u.literal(lambda r: r.value(lambda n: n.as_scalar(False))), "false"),
    (lambda u: u.literal(lambda r: r.value(lambda n: n.as_scalar(5))), "5n"),
    (lambda u: u.literal(lambda r: r.value(lambda n: n.as_scalar(1.5))), "1.5"),
    (lambda u: u.literal(lambda r: r.value(lambda n: n.as_scalar("a\n"))), '"a\\n"'),
    (lambda u: u.literal(lambda r: r.value(lambda n: n.as_scalar("a")).quote("'")), "'a'"),
    (lambda u: u.literal(lambda r: r.value(lambda n: n.as_scalar(16.0)).text("0x10")), "0x10"),
    (lambda u: u.binaryexpression(lambda r: r.left(lambda e: e.literal(lambda li: li.value(lambda n: n.as_scalar(-5))))
        .operator("**").right(one)), "(-5n) ** 1"),  # a negative value's precedence, a unary operator's
    (lambda u: u.constant(lambda r: r.singleton("null").text("null")), "null"),
    (lambda u: u.constant(lambda r: r.singleton("null")), "null"),
    (lambda u: u.constant(lambda r: r.pattern("a+").flags("g")), "/a+/g"),
    (lambda u: u.constant(lambda r: r.pattern("a")), "/a/"),
    # names and templates
    (lambda u: u.arrowfunctionexpression(lambda r: r.params([lambda p: p.identifier(lambda i: i.decorators([lambda d: d.expression(x)])
        .name("y").optional(True).typeAnnotation(lambda a: a.typeAnnotation(number))), lambda p: p.identifier(lambda i: i)])),
     "(@x y?: number, ) => {}"),  # a parameter's decorators, `?` and type; one of no name, as held
    (lambda u: u.memberexpression(lambda r: r.object(x).property(lambda p: p.privateidentifier(lambda i: i.name("p")))), "x.#p"),
    (lambda u: u.templateliteral(lambda r: r), "``"),
    (lambda u: u.templateliteral(lambda r: r.quasis([lambda q: q.tail(True)])), "``"),
    (lambda u: u.templateliteral(lambda r: r.quasis([lambda q: q.raw("a"), lambda q: q.tail(True)]).expressions([x])), "`a${x}`"),
    (lambda u: u.taggedtemplateexpression(lambda r: r.tag(x).typeArguments(lambda a: a.params([number])).quasi(
        lambda q: q.quasis([lambda e: e.raw("a").tail(True)]))), "x<number>`a`"),
    (lambda u: u.parenthesizedexpression(lambda r: r.expression(lambda e: e.sequenceexpression(lambda s: s.expressions([x, y])))), "(x, y)"),
    # displays
    (lambda u: u.arrayexpression(lambda r: r), "[]"),
    (lambda u: u.arrayexpression(lambda r: r.elements([lambda e: e.typescript5(lambda t: t.elision(lambda h: h)),
                                                       lambda e: e.typescript5(x), lambda e: e.typescript5(lambda t: t.elision(lambda h: h))])),
     "[, x,,]"),  # a hole as nothing, a last hole kept
    (empty, "({})"),  # an object, as a statement holds it
    (lambda u: u.objectexpression(lambda r: r.properties([
        lambda p: p.property(lambda q: q.key(x).shorthand(True).value(x)),
        lambda p: p.property(lambda q: q.key(y).shorthand(True).value(lambda v: v.assignmentpattern(lambda a: a.left(y).right(one)))),
        lambda p: p.property(lambda q: q.propertyKind("get").key(z).value(lambda v: v.functionexpression(lambda f: f))),
        lambda p: p.property(lambda q: q.propertyKind("set").key(z).value(lambda v: v.functionexpression(lambda f: f.params([x])))),
        lambda p: p.property(lambda q: q.method(True).key(x).value(lambda v: v.functionexpression(lambda f: f.isAsync(True).generator(True)))),
        lambda p: p.property(lambda q: q.method(True).optional(True).key(x).value(lambda v: v.functionexpression(lambda f: f))),
        lambda p: p.property(lambda q: q.method(True).key(x).value(lambda v: v.tsemptybodyfunctionexpression(
            lambda f: f.returnType(lambda a: a.typeAnnotation(number))))),
        lambda p: p.property(lambda q: q.computed(True).key(x).value(lambda v: v.sequenceexpression(lambda s: s.expressions([x, y])))),
        lambda p: p.spreadelement(lambda s: s.argument(x))])),
     "({ x, y = 1, get z() {}, set z(x) {}, async *x() {}, x?() {}, x(): number, [x]: (x, y), ...x })"),
    # functions and classes: a body of statements is written `{}`
    (lambda u: u.functionexpression(lambda r: r), "(function() {})"),
    (lambda u: u.sequenceexpression(lambda s: s.expressions([x, lambda e: e.functionexpression(lambda r: r.isAsync(True).generator(True)
        .id(lambda i: i.name("f")).typeParameters(lambda d: d.params([lambda p: p.name(lambda i: i.name("T"))])).params([
            lambda p: p.identifier(lambda i: i.name("a").typeAnnotation(lambda a: a.typeAnnotation(T_)))]).returnType(
            lambda a: a.typeAnnotation(T_)))])), "x, async function* f<T>(a: T): T {}"),
    (lambda u: u.tsemptybodyfunctionexpression(lambda r: r.isAsync(True).generator(True)), "async *()"),
    (lambda u: u.tsemptybodyfunctionexpression(lambda r: r), "()"),
    (lambda u: u.arrowfunctionexpression(lambda r: r.typeParameters(lambda d: d.params([lambda p: p.name(lambda i: i.name("T"))]))
        .params([x])), "<T,>(x) => {}"),  # a lone type parameter, as a .tsx file reads it
    (lambda u: u.arrowfunctionexpression(lambda r: r.isAsync(True).typeParameters(lambda d: d.params([
        lambda p: p.name(lambda i: i.name("T")).constraint(U_)])).body(x)), "async <T extends U>() => x"),
    (lambda u: u.arrowfunctionexpression(lambda r: r.typeParameters(lambda d: d.params([
        lambda p: p.name(lambda i: i.name("T")), lambda p: p.name(lambda i: i.name("U"))])).body(x)), "<T, U>() => x"),
    (lambda u: u.arrowfunctionexpression(lambda r: r.body(empty)), "() => ({})"),  # an object body, parenthesized
    (lambda u: u.arrowfunctionexpression(lambda r: r.body(lambda b: b.memberexpression(lambda m: m.object(empty).property(x)))),
     "() => ({}.x)"),
    (lambda u: u.arrowfunctionexpression(lambda r: r.body(lambda b: b.sequenceexpression(lambda s: s.expressions([x, y])))), "() => (x, y)"),
    (lambda u: u.classexpression(lambda r: r), "(class {})"),
    (lambda u: u.sequenceexpression(lambda s: s.expressions([x, lambda e: e.classexpression(lambda r: r.decorators([lambda d: d.expression(x)])
        .id(lambda i: i.name("K")).typeParameters(lambda d: d.params([lambda p: p.name(lambda i: i.name("T"))])).superClass(x)
        .superTypeArguments(lambda a: a.params([number])).implements([lambda c: c.expression(y).typeArguments(lambda a: a.params([T_])),
                                                                       lambda c: c.expression(z)]))])),
     "x, @x class K<T> extends x<number> implements y<T>, z {}"),
    (lambda u: u.sequenceexpression(lambda s: s.expressions([x, lambda e: e.classexpression(lambda r: r.superClass(lambda c: c.callexpression(
        lambda f: f.callee(x))))])), "x, class extends x() {}"),
    (lambda u: u.metaproperty(lambda r: r.meta(lambda i: i.name("new")).property(lambda i: i.name("target"))), "new.target"),
    # operations
    (lambda u: u.memberexpression(lambda r: r.object(one).property(x)), "(1).x"),  # `1.x` would be a number
    (lambda u: u.memberexpression(lambda r: r.object(lambda e: e.literal(lambda li: li.value(lambda n: n.as_scalar(1.5)))).property(x)), "1.5.x"),
    (lambda u: u.memberexpression(lambda r: r.object(lambda e: e.literal(lambda li: li.value(lambda n: n.as_scalar(1000.0)).text("1_000"))).property(x)),
     "(1_000).x"),
    (lambda u: u.memberexpression(lambda r: r.object(lambda e: e.thisexpression(lambda h: h)).computed(True).optional(True).property(y)), "this?.[y]"),
    (lambda u: u.memberexpression(lambda r: r.object(x).computed(True).property(lambda p: p.sequenceexpression(lambda s: s.expressions([x, y])))),
     "x[x, y]"),
    (lambda u: u.chainexpression(lambda r: r.expression(lambda e: e.memberexpression(lambda m: m.object(x).optional(True).property(y)))), "x?.y"),
    (lambda u: u.callexpression(lambda r: r.callee(x).optional(True).typeArguments(lambda a: a.params([number])).arguments([
        y, lambda e: e.sequenceexpression(lambda s: s.expressions([x, y]))])), "x?.<number>(y, (x, y))"),
    (lambda u: u.newexpression(lambda r: r.callee(lambda c: c.callexpression(lambda f: f.callee(x)))), "new (x())()"),
    (lambda u: u.newexpression(lambda r: r.callee(lambda c: c.memberexpression(lambda m: m.object(lambda o: o.tsnonnullexpression(
        lambda n: n.expression(x))).property(y)))), "new (x!.y)()"),  # a callee that calls, in parentheses
    (lambda u: u.newexpression(lambda r: r.callee(lambda c: c.memberexpression(lambda m: m.object(x).property(y))).typeArguments(
        lambda a: a.params([T_])).arguments([z])), "new x.y<T>(z)"),
    (lambda u: u.importexpression(lambda r: r.phase("defer").source(x)), "import.defer(x)"),
    (lambda u: u.importexpression(lambda r: r.source(x).options(y)), "import(x, y)"),
    (lambda u: u.updateexpression(lambda r: r.operator("++").prefix(True).argument(x)), "++x"),
    (lambda u: u.updateexpression(lambda r: r.operator("--").argument(lambda a: a.binaryexpression(lambda b: b.left(x).operator("+").right(y)))),
     "(x + y)--"),
    (lambda u: u.unaryexpression(lambda r: r.operator("void").argument(x)), "void x"),
    (lambda u: u.unaryexpression(lambda r: r.operator("-").argument(lambda a: a.unaryexpression(lambda b: b.operator("-").argument(x)))), "- -x"),
    (lambda u: u.unaryexpression(lambda r: r.operator("+").argument(lambda a: a.literal(lambda li: li.value(lambda n: n.as_scalar(1.0)).text("+1")))), "+ +1"),
    (lambda u: u.unaryexpression(lambda r: r.operator("!").argument(lambda a: a.binaryexpression(lambda b: b.left(x).operator("in").right(y)))), "!(x in y)"),
    (lambda u: u.awaitexpression(lambda r: r.argument(x)), "await x"),
    # binary and logical operators by precedence: `**`'s left, `??` against `||` and `&&`
    (lambda u: u.binaryexpression(lambda r: r.left(lambda a: a.binaryexpression(lambda b: b.left(x).operator("+").right(y))).operator("*").right(
        lambda a: a.binaryexpression(lambda b: b.left(x).operator("-").right(y)))), "(x + y) * (x - y)"),
    (lambda u: u.binaryexpression(lambda r: r.left(lambda a: a.binaryexpression(lambda b: b.left(x).operator("**").right(y))).operator("**").right(
        lambda a: a.binaryexpression(lambda b: b.left(x).operator("**").right(y)))), "(x ** y) ** x ** y"),
    (lambda u: u.binaryexpression(lambda r: r.left(lambda a: a.awaitexpression(lambda b: b.argument(x))).operator("**").right(y)), "(await x) ** y"),
    (lambda u: u.binaryexpression(lambda r: r.left(lambda a: a.tstypeassertion(lambda b: b.typeAnnotation(number).expression(x))).operator("**").right(y)),
     "(<number>x) ** y"),
    (lambda u: u.logicalexpression(lambda r: r.left(lambda a: a.logicalexpression(lambda b: b.left(x).operator("??").right(y))).operator("||").right(
        lambda a: a.logicalexpression(lambda b: b.left(x).operator("&&").right(y)))), "(x ?? y) || x && y"),
    (lambda u: u.logicalexpression(lambda r: r.left(x).operator("??").right(lambda a: a.logicalexpression(lambda b: b.left(y).operator("||").right(z)))),
     "x ?? (y || z)"),
    (lambda u: u.logicalexpression(lambda r: r.left(lambda a: a.logicalexpression(lambda b: b.left(x).operator("??").right(y))).operator("??").right(z)),
     "x ?? y ?? z"),
    (lambda u: u.conditionalexpression(lambda r: r.test(lambda t: t.conditionalexpression(lambda c: c.test(x).consequent(y).alternate(z)))
        .consequent(lambda c: c.assignmentexpression(lambda a: a.left(x).operator("=").right(y))).alternate(z)), "(x ? y : z) ? x = y : z"),
    # an assignment's target, a pattern: a term it may be, or a destructuring
    (lambda u: u.assignmentexpression(lambda r: r.left(lambda t: t.memberexpression(lambda m: m.object(x).property(y))).operator("+=").right(one)),
     "x.y += 1"),
    (lambda u: u.assignmentexpression(lambda r: r.left(lambda t: t.arraypattern(lambda a: a.elements([x, lambda e: e.elision(lambda h: h),
        lambda e: e.restelement(lambda s: s.argument(y))]))).operator("=").right(z)), "[x,, ...y] = z"),
    (lambda u: u.assignmentexpression(lambda r: r.left(lambda t: t.objectpattern(lambda o: o)).operator("=").right(z)), "({} = z)"),
    (lambda u: u.assignmentexpression(lambda r: r.left(lambda t: t.tsasexpression(lambda a: a.expression(x).typeAnnotation(
        lambda k: k.tsanykeyword(lambda h: h)))).operator("=").right(z)), "(x as any) = z"),
    (lambda u: u.assignmentexpression(lambda r: r.left(lambda t: t.parenthesizedexpression(lambda a: a.expression(x))).operator("??=").right(z)),
     "(x) ??= z"),
    (lambda u: u.yieldexpression(lambda r: r), "yield"),
    (lambda u: u.yieldexpression(lambda r: r.argument(lambda a: a.sequenceexpression(lambda s: s.expressions([x, y])))), "yield (x, y)"),
    (lambda u: u.tsasexpression(lambda r: r.expression(lambda e: e.tsasexpression(lambda a: a.expression(x).typeAnnotation(T_))).typeAnnotation(U_)),
     "x as T as U"),
    (lambda u: u.tssatisfiesexpression(lambda r: r.expression(lambda e: e.binaryexpression(lambda b: b.left(x).operator("==").right(y)))
        .typeAnnotation(lambda t: t.tsuniontype(lambda n: n.types([number, string])))), "(x == y) satisfies number | string"),
    (lambda u: u.tsinstantiationexpression(lambda r: r.expression(lambda e: e.tsnonnullexpression(lambda n: n.expression(x))).typeArguments(
        lambda a: a.params([T_]))), "x!<T>"),
    # patterns, as parameters
    (lambda u: u.arrowfunctionexpression(lambda r: r.params([
        lambda p: p.arraypattern(lambda a: a.decorators([lambda d: d.expression(x)]).elements([x, lambda e: e.elision(lambda h: h)]).optional(True)
            .typeAnnotation(lambda t: t.typeAnnotation(lambda k: k.tsarraytype(lambda n: n.elementType(number))))),
        lambda p: p.objectpattern(lambda o: o.properties([lambda q: q.property(lambda s: s.key(x).shorthand(True).value(x)),
            lambda q: q.property(lambda s: s.key(y).value(lambda v: v.assignmentpattern(lambda a: a.left(z).right(one)))),
            lambda q: q.restelement(lambda s: s.argument(z))])),
        lambda p: p.objectpattern(lambda o: o.optional(True)),
        lambda p: p.assignmentpattern(lambda a: a.left(lambda t: t.identifier(lambda i: i.name("x").typeAnnotation(lambda b: b.typeAnnotation(number))))
            .right(one)),
        lambda p: p.restelement(lambda s: s.argument(y).typeAnnotation(lambda t: t.typeAnnotation(lambda k: k.tsarraytype(lambda n: n.elementType(T_))))),
        lambda p: p.tsparameterproperty(lambda s: s.decorators([lambda d: d.expression(x)]).accessibility("private").static(True).override(True)
            .readonly(True).parameter(z))])),
     "(@x [x,,]?: number[], { x, y: z = 1, ...z }, {}?, x: number = 1, ...y: T[], @x private static override readonly z) => {}"),
    # types, as `x as <type>`
    (lambda u: u.tsasexpression(lambda r: r.expression(x).typeAnnotation(lambda t: t.tstupletype(lambda n: n.elementTypes([
        lambda k, keyword=keyword: getattr(k, keyword)(lambda h: h) for keyword in (
            "tsanykeyword", "tsunknownkeyword", "tsnumberkeyword", "tsbigintkeyword", "tsbooleankeyword", "tsstringkeyword",
            "tssymbolkeyword", "tsobjectkeyword", "tsneverkeyword", "tsvoidkeyword", "tsundefinedkeyword", "tsnullkeyword",
            "tsintrinsickeyword", "tsthistype")])))),
     "x as [any, unknown, number, bigint, boolean, string, symbol, object, never, void, undefined, null, intrinsic, this]"),
    (lambda u: u.tsasexpression(lambda r: r.expression(x).typeAnnotation(lambda t: t.tsuniontype(lambda n: n.types([
        lambda k: k.tstypereference(lambda q: q.typeName(lambda m: m.tsqualifiedname(lambda a: a.left(lambda l: l.tsqualifiedname(
            lambda b: b.left(x).right(lambda i: i.name("y")))).right(lambda i: i.name("z")))).typeArguments(lambda a: a.params([T_]))),
        lambda k: k.tstypereference(lambda q: q.typeName(lambda m: m.thisexpression(lambda h: h))),
        lambda k: k.tsparenthesizedtype(lambda q: q.typeAnnotation(number)),
        lambda k: k.tsliteraltype(lambda q: q.literal(lambda li: li.literal(lambda v: v.value(lambda n: n.as_scalar("s"))))),
        lambda k: k.tsliteraltype(lambda q: q.literal(lambda li: li.unaryexpression(lambda v: v.operator("-").argument(one)))),
        lambda k: k.tsliteraltype(lambda q: q.literal(lambda li: li.templateliteral(lambda v: v.quasis([lambda e: e.raw("a").tail(True)])))),
        lambda k: k.tstemplateliteraltype(lambda q: q.quasis([lambda e: e.raw("a"), lambda e: e.tail(True)]).types([number])),
        lambda k: k.tstemplateliteraltype(lambda q: q), lambda k: k.tstemplateliteraltype(lambda q: q.quasis([lambda e: e.tail(True)]))])))),
     'x as x.y.z<T> | this | (number) | "s" | -1 | `a` | `a${number}` | `` | ``'),
    (lambda u: u.tsasexpression(lambda r: r.expression(x).typeAnnotation(lambda t: t.tsintersectiontype(lambda n: n.types([
        lambda k: k.tsarraytype(lambda a: a.elementType(lambda e: e.tsuniontype(lambda q: q.types([number, string])))),
        lambda k: k.tsuniontype(lambda q: q.types([number, lambda e: e.tsfunctiontype(lambda f: f.returnType(lambda a: a.typeAnnotation(
            lambda v: v.tsvoidkeyword(lambda h: h))))])),
        lambda k: k.tstupletype(lambda q: q.elementTypes([
            lambda e: e.tsnamedtuplemember(lambda m: m.label(lambda i: i.name("a")).elementType(number)),
            lambda e: e.tsnamedtuplemember(lambda m: m.label(lambda i: i.name("b")).optional(True).elementType(string)),
            lambda e: e.tsoptionaltype(lambda o: o.typeAnnotation(number)),
            lambda e: e.tsresttype(lambda o: o.typeAnnotation(lambda a: a.tsarraytype(lambda b: b.elementType(T_))))])),
        lambda k: k.tsfunctiontype(lambda f: f.typeParameters(lambda d: d.params([lambda p: p.name(lambda i: i.name("T"))])).params([
            lambda p: p.identifier(lambda i: i.name("a").typeAnnotation(lambda a: a.typeAnnotation(T_)))]).returnType(
            lambda a: a.typeAnnotation(T_))),
        lambda k: k.tsconstructortype(lambda f: f.abstract(True).returnType(lambda a: a.typeAnnotation(T_))),
        lambda k: k.tsconstructortype(lambda f: f)])))),
     "x as (number | string)[] & (number | (() => void)) & [a: number, b?: string, number?, ...T[]] & (<T>(a: T) => T) & (abstract new () => T) & (new ())"),
    (lambda u: u.tsasexpression(lambda r: r.expression(x).typeAnnotation(lambda t: t.tsuniontype(lambda n: n.types([
        lambda k: k.tstypeliteral(lambda q: q),
        lambda k: k.tstypeliteral(lambda q: q.members([
            lambda m: m.tspropertysignature(lambda s: s.readonly(True).key(x).optional(True).typeAnnotation(lambda a: a.typeAnnotation(T_))),
            lambda m: m.tsmethodsignature(lambda s: s.computed(True).key(y).optional(True).returnType(lambda a: a.typeAnnotation(
                lambda v: v.tsvoidkeyword(lambda h: h)))),
            lambda m: m.tsmethodsignature(lambda s: s.methodKind("get").key(z).returnType(lambda a: a.typeAnnotation(T_))),
            lambda m: m.tsmethodsignature(lambda s: s.methodKind("method").key(z)),
            lambda m: m.tscallsignaturedeclaration(lambda s: s.returnType(lambda a: a.typeAnnotation(T_))),
            lambda m: m.tsconstructsignaturedeclaration(lambda s: s.params([x])),
            lambda m: m.tsindexsignature(lambda s: s.accessibility("public").static(True).readonly(True).parameters([
                lambda p: p.identifier(lambda i: i.name("k").typeAnnotation(lambda a: a.typeAnnotation(string)))]).typeAnnotation(
                lambda a: a.typeAnnotation(T_))),
            lambda m: m.tspropertysignature(lambda s: s.key(lambda k: k.literal(lambda li: li.value(lambda n: n.as_scalar("k"))))),
            lambda m: m.tsindexsignature(lambda s: s.parameters([x]))]))])))),
     'x as {} | { readonly x?: T; [y]?(): void; get z(): T; z(); (): T; new (x); public static readonly [k: string]: T; "k"; [x]; }'),
    (lambda u: u.tsasexpression(lambda r: r.expression(x).typeAnnotation(lambda t: t.tstypeliteral(lambda q: q.members([
        lambda m: m.comment(lambda c: c.text(" first").trailing(True)),
        lambda m: m.tspropertysignature(lambda s: s.key(x)),
        lambda m: m.comment(lambda c: c.text(" of x").trailing(True)),
        lambda m: m.comment(lambda c: c.block(True).text(" y ")),
        lambda m: m.comment(lambda c: c)])))),
     "x as {\n    // first\n    x; // of x\n    /* y */\n    //\n}"),  # with comments, its members' lines
    (lambda u: u.tsasexpression(lambda r: r.expression(x).typeAnnotation(lambda t: t.tsuniontype(lambda n: n.types([
        lambda k: k.tsmappedtype(lambda m: m.readonly("+readonly").key(lambda i: i.name("K")).value(T_).nameType(U_).optional("?").body(T_)),
        lambda k: k.tsmappedtype(lambda m: m.key(lambda i: i.name("K")).value(T_)),
        lambda k: k.tsindexedaccesstype(lambda a: a.objectType(lambda o: o.tsuniontype(lambda q: q.types([T_, U_]))).indexType(
            lambda i: i.tstypeoperator(lambda o: o.operator("keyof").typeAnnotation(T_)))),
        lambda k: k.tstypeoperator(lambda o: o.operator("unique").typeAnnotation(lambda s: s.tssymbolkeyword(lambda h: h))),
        lambda k: k.tstypeoperator(lambda o: o.operator("readonly").typeAnnotation(lambda a: a.tsuniontype(lambda q: q.types([T_, U_])))),
        lambda k: k.tstypeoperator(lambda o: o.operator("keyof")),
        lambda k: k.tstypequery(lambda q: q.exprName(x).typeArguments(lambda a: a.params([T_]))),
        lambda k: k.tstypequery(lambda q: q.exprName(lambda e: e.tsimporttype(lambda i: i.source(lambda s: s.literal(
            lambda li: li.value(lambda n: n.as_scalar("m"))))))),
        lambda k: k.tsimporttype(lambda i: i.source(lambda s: s.literal(lambda li: li.value(lambda n: n.as_scalar("m")).quote("'"))).options(
            lambda o: o.objectexpression(lambda e: e.properties([lambda p: p.property(lambda q: q.key(lambda w: w.identifier(lambda j: j.name("with")))
                .value(empty))]))).qualifier(lambda q: q.identifier(lambda j: j.name("A"))).typeArguments(lambda a: a.params([T_])))])))),
     "x as { +readonly [K in T as U]?: T } | { [K in T] } | (T | U)[keyof T] | unique symbol | readonly (T | U) | keyof | typeof x<T> | "
     "typeof import(\"m\") | import('m', { with: {} }).A<T>"),
    (lambda u: u.tsasexpression(lambda r: r.expression(x).typeAnnotation(lambda t: t.tsconditionaltype(lambda c: c
        .checkType(lambda k: k.tsuniontype(lambda q: q.types([T_, U_])))
        .extendsType(lambda k: k.tsarraytype(lambda a: a.elementType(lambda e: e.tsinfertype(lambda i: i.typeParameter(
            lambda p: p.name(lambda n: n.name("V")).constraint(string)))))).trueType(lambda k: k.tsconditionaltype(lambda d: d
                .checkType(T_).extendsType(lambda e: e.tsconditionaltype(lambda f: f.checkType(T_).extendsType(U_).trueType(T_).falseType(U_)))
                .trueType(T_).falseType(U_))).falseType(lambda k: k.tsconditionaltype(lambda d: d.checkType(T_)
                    .extendsType(lambda e: e.tsfunctiontype(lambda f: f.returnType(lambda a: a.typeAnnotation(
                        lambda g: g.tsconditionaltype(lambda h: h.checkType(T_).extendsType(U_).trueType(T_).falseType(U_))))))
                    .trueType(lambda e: e.tsfunctiontype(lambda f: f.returnType(lambda a: a.typeAnnotation(T_)))).falseType(
                    lambda e: e.tsconditionaltype(lambda h: h.checkType(T_).extendsType(lambda g: g.tsfunctiontype(lambda f: f)).trueType(T_)
                        .falseType(U_)))))))),
     "x as T | U extends (infer V extends string)[] ? T extends (T extends U ? T : U) ? T : U : "
     "T extends (() => T extends U ? T : U) ? () => T : T extends () ? T : U"),
    (lambda u: u.tsasexpression(lambda r: r.expression(x).typeAnnotation(lambda t: t.tsuniontype(lambda n: n.types([
        lambda k: k.tsfunctiontype(lambda f: f.params([x]).returnType(lambda a: a.typeAnnotation(lambda p: p.tstypepredicate(lambda q: q
            .asserts(True).parameterName(x).typeAnnotation(lambda b: b.typeAnnotation(T_)))))),
        lambda k: k.tsfunctiontype(lambda f: f.returnType(lambda a: a.typeAnnotation(lambda p: p.tstypepredicate(lambda q: q.parameterName(
            lambda m: m.tsthistype(lambda h: h)))))),
        lambda k: k.tsfunctiontype(lambda f: f.typeParameters(lambda d: d.params([lambda p: p.const(True).isIn(True).isOut(True).name(
            lambda i: i.name("T")).default(U_)])))])))),
     "x as ((x) => asserts x is T) | (() => this) | (<const in out T = U>())"),
    # JSX, as written
    (lambda u: u.jsxelement(lambda r: r.openingElement(lambda o: o.name(lambda n: n.jsxmemberexpression(lambda m: m.object(
        lambda b: b.jsxidentifier(lambda i: i.name("a"))).property(lambda i: i.name("b")))).typeArguments(lambda a: a.params([T_])).attributes([
            lambda a: a.jsxattribute(lambda t: t.name(lambda n: n.jsxnamespacedname(lambda s: s.namespace(lambda i: i.name("x")).name(
                lambda i: i.name("y"))))),
            lambda a: a.jsxattribute(lambda t: t.name(lambda n: n.jsxidentifier(lambda i: i.name("z"))).value(lambda v: v.jsxexpressioncontainer(
                lambda c: c.expression(one)))),
            lambda a: a.jsxattribute(lambda t: t.name(lambda n: n.jsxidentifier(lambda i: i.name("e"))).value(lambda v: v.jsxelement(
                lambda c: c.openingElement(lambda o: o.name(lambda n: n.jsxidentifier(lambda i: i.name("b"))).selfClosing(True))))),
            lambda a: a.jsxattribute(lambda t: t.name(lambda n: n.jsxidentifier(lambda i: i.name("f"))).value(lambda v: v.jsxfragment(lambda c: c))),
            lambda a: a.jsxspreadattribute(lambda s: s.argument(lambda e: e.sequenceexpression(lambda q: q.expressions([x, y]))))]))
        .children([lambda c: c.jsxtext(lambda t: t.raw("hi ")), lambda c: c.jsxtext(lambda t: t), lambda c: c.jsxspreadchild(lambda s: s.expression(x)),
                   lambda c: c.jsxelement(lambda e: e.openingElement(lambda o: o.name(lambda n: n.jsxidentifier(lambda i: i)))),
                   lambda c: c.jsxexpressioncontainer(lambda e: e.expression(lambda v: v.jsxemptyexpression(lambda h: h)))])
        .closingElement(lambda c: c.name(lambda n: n.jsxmemberexpression(lambda m: m.object(lambda b: b.jsxidentifier(lambda i: i.name("a")))
            .property(lambda i: i.name("b")))))),
     "<a.b<T> x:y z={1} e=<b /> f=<></> {...(x, y)}>hi {...x}<></>{}</a.b>"),  # an element without its closing one closes its name
    # as an expression statement holds it: parenthesized where it would start with `{`, `function`, `class` or `let [`
    (lambda u: u.memberexpression(lambda r: r.object(empty).property(x)), "({}.x)"),
    (lambda u: u.binaryexpression(lambda r: r.left(empty).operator("+").right(x)), "({} + x)"),
    (lambda u: u.callexpression(lambda r: r.callee(lambda c: c.functionexpression(lambda f: f))), "(function() {}())"),
    (lambda u: u.taggedtemplateexpression(lambda r: r.tag(lambda c: c.classexpression(lambda f: f)).quasi(
        lambda q: q.quasis([lambda e: e.tail(True)]))), "(class {}``)"),
    (lambda u: u.conditionalexpression(lambda r: r.test(empty).consequent(x).alternate(y)), "({} ? x : y)"),
    (lambda u: u.tssatisfiesexpression(lambda r: r.expression(empty).typeAnnotation(T_)), "({} satisfies T)"),
    (lambda u: u.updateexpression(lambda r: r.operator("++").argument(empty)), "({}++)"),
    (lambda u: u.updateexpression(lambda r: r.operator("++").prefix(True).argument(empty)), "++{}"),
    (lambda u: u.memberexpression(lambda r: r.object(lambda e: e.identifier(lambda i: i.name("let"))).computed(True).property(one)), "(let[1])"),
    (lambda u: u.memberexpression(lambda r: r.object(lambda e: e.identifier(lambda i: i.name("let"))).property(x)), "let.x"),
]:
    assert shown(spec) == text, (shown(spec), text)
with raises(ValueError, match="TypeScript5 binaryexpression has no operator '<>'"):  # an operator outside its vocabulary
    shown(lambda u: u.binaryexpression(lambda r: r.left(x).operator("<>").right(y)))

<a id="DIA-23"></a>
## DIA-23 · TypeScript5 is complete: each node type of typescript-estree, TypeScript 5.0 to 5.9, kept as data, is a kind, a type or a pattern, or held by one, or not an expression

In [ ]:
import json
from pathlib import Path

estree = json.loads(Path("../../grammars/typescript5-estree.json").read_text(encoding="utf-8"))  # typescript-estree's nodes, as data
unions = {name: {b.name for b in store.registered(f"Expressions.TypeScript.{path}").branches} for name, path in (
    ("Term", "TypeScript5.Term"), ("Type", "Types.Term"), ("Pattern", "Patterns.Term"), ("TypeElement", "Types.TypeElement"))}
mapped = {name: set() for name in unions}
for node in estree["nodes"]:  # each a kind, a type, a pattern, any of them at once, a helper, or not an expression, with its reason
    ways = {way for way in ("kinds", "types", "patterns", "helper", "not") if way in node}
    assert ways and (ways <= {"kinds", "types", "patterns"} or len(ways) == 1), node
    for way, union in (("kinds", "Term"), ("types", "Type"), ("patterns", "Pattern")):
        if way in node:
            assert node[way] and set(node[way]) <= unions[union], node
            mapped[union] |= set(node[way])
    if "helper" in node:  # a value object a kind holds, not a term
        owner = "Types" if node["helper"] in TST.HELPERS else "TypeScript5"  # an object type's members, `TypeScript Types`'
        helper = store.registered(f"Expressions.TypeScript.{owner}.{node['helper']}")
        assert helper is not None and not helper.ref and node["helper"].lower() not in unions["Term"], node
        mapped["TypeElement"] |= {node["helper"].lower()} & unions["TypeElement"]
    if "not" in node:
        assert type(node["not"]) is str and node["not"], node
assert len(estree["nodes"]) == len({node["node"] for node in estree["nodes"]}) == 168  # every node type, once
assert [node["node"] for node in estree["nodes"] if "kinds" in node and node.get("class") != "Expression"] == [
    "AssignmentPattern", "JSXEmptyExpression", "PrivateIdentifier", "Property", "SpreadElement",
    "TSEmptyBodyFunctionExpression"]  # an expression's kinds, and what an expression's place holds beside them
concrete = {(entry["union"], entry["kind"]) for entry in estree["concrete"]}  # what typescript-estree does not keep
for name, branches in unions.items():  # every branch of every union is typescript-estree's, but these
    assert branches == mapped[name] | {kind for union, kind in concrete if union == name} | (
        {"error", "missing"} if name in ("Term", "Type", "Pattern") else set()), (name, branches ^ mapped[name])
assert set(TS.DIALECT.kinds()) | {"arraypattern", "objectpattern", "assignmentpattern"} == unions["Term"]  # the patterns,
assert TS.EXTENSIONS <= mapped["Term"]  # `TypeScript Patterns`' kinds; JSX, in typescript-estree, an extension here
# Each dialect's own kinds are those of its `Term` that are its schemas, the rest its family's terms, each dialect's
# `types` or `patterns` entries; its helpers, too
for dialect, union in ((TST, "Type"), (TSP, "Pattern")):
    path = f"Expressions.TypeScript.{dialect.DIALECT.path.split('.')[1]}."
    own = [b.name for b in store.registered(path + "Term").branches if b.schema.name.startswith(path)]
    assert own == list(dialect.TAGS) and set(own) <= mapped[union] | {k for u, k in concrete if u == union}, dialect
    assert all(store.registered(path + helper) is not None for helper in dialect.HELPERS)
assert set(TST.DIALECT.kinds()) == set(TST.TAGS) | {"identifier", "error", "missing"}  # an identifier read as a type's name
assert set(TSP.DIALECT.kinds()) == set(TSP.TAGS) | {"error", "missing"}

<a id="DIA-24"></a>
## DIA-24 · C and C++: mbse-programs' kinds each language has, in its shapes, its type names a dialect of their own and its binding targets a union of its own; literals keeping what their text encodes, names, what each kind binds, the slots that may be empty, and evaluation refusing new kinds by name

In [ ]:
from mbse.Expressions.Cpp.Expressions import Evaluators as CPV
CPPE = Cpp.Expressions
CPP = CPPE.DIALECT


def cpp(spec):
    return term(spec, CPP)


def branches(name):
    """A union's branches, by name."""
    return [b.name for b in store.registered(name).branches]


def schemas(name):
    return {b.name: b.schema for b in store.registered(name).branches}


# Each dialect's terms, its type names a dialect of its own (C Types, Cpp Types: suite CTY and XTY), its binding targets
# (C++'s an init capture's name, an identifier; none in a C expression, so invalid syntax alone), and the helpers a type
# name and a lambda hold, each union named after mbse-programs' category: C's, which no expression holds, C Types'; C++'s,
# which a lambda's declarator holds too, C++'s
DIRECTIVES = ["includedirective", "definedirective", "ifdirective", "ifdefdirective", "elifdirective", "elifdefdirective",
              "elsedirective", "otherdirective", "comment"]  # an enumeration's items, which C Preprocessor's install adds
assert branches("Expressions.C.Types.Term") == branches("Expressions.Cpp.Types.Term") == ["typeid", "enumerator", "error", "missing", *DIRECTIVES]
assert branches("Expressions.C.Expressions.Pattern") == ["error", "missing"] and branches("Expressions.Cpp.Expressions.Pattern") == ["identifier", "error", "missing"]
assert branches("Expressions.C.Types.Specifier") == [
    "declspecifier", "cvqualifier", "primitivetypespecifier", "namedtypespecifier", "classspecifier", "enumspecifier",
    "placeholdertypespecifier", "typeofspecifier", "atomictypespecifier", "bitintspecifier", "standardattributespecifier",
    "alignasspecifier", "gnuattributespecifier", "declspecspecifier"]
assert branches("Expressions.Cpp.Expressions.Specifier") == [
    "declspecifier", "cvqualifier", "primitivetypespecifier", "namedtypespecifier", "typenamespecifier", "classspecifier",
    "enumspecifier", "decltypespecifier", "placeholdertypespecifier", "standardattributespecifier", "alignasspecifier",
    "gnuattributespecifier", "declspecspecifier"]
assert branches("Expressions.C.Types.Declarator") == ["iddeclarator", "pointerdeclarator", "arraydeclarator", "functiondeclarator",
                                                "parenthesizeddeclarator"]
assert branches("Expressions.Cpp.Expressions.Declarator") == ["iddeclarator", "packdeclarator", "pointerdeclarator", "referencedeclarator",
                                                  "arraydeclarator", "functiondeclarator", "parenthesizeddeclarator"]
assert branches("Expressions.C.Types.Parameter") == branches("Expressions.Cpp.Expressions.Parameter") == ["parameterdeclaration", "ellipsisparameter"]
assert branches("Expressions.C.Types.AttributeSpecifier") == branches("Expressions.Cpp.Expressions.AttributeSpecifier") == [
    "standardattributespecifier", "alignasspecifier", "gnuattributespecifier", "declspecspecifier"]
assert branches("Expressions.C.Expressions.Designator") == ["fielddesignator", "indexdesignator"]  # C++20's designators name fields alone
assert branches("Expressions.Cpp.Expressions.Designator") == ["fielddesignator"]
assert branches("Expressions.C.Expressions.Initializer") == ["equalinitializer", "initializerlist"]
assert branches("Expressions.Cpp.Expressions.Initializer") == ["equalinitializer", "parenthesizedinitializer", "initializerlist"]
assert branches("Expressions.C.Expressions.Name") == ["identifier"]  # C's one name, held wherever mbse-programs holds a name
assert branches("Expressions.Cpp.Expressions.Name") == ["identifier", "operatorname", "conversionname", "literaloperatorname",
                                            "destructorname", "templateid", "qualifiedname", "decltypespecifier"]
assert branches("Expressions.Cpp.Expressions.Capture") == ["defaultcapture", "simplecapture", "thiscapture", "initcapture"]
assert branches("Expressions.Cpp.Expressions.Requirement") == ["simplerequirement", "typerequirement", "compoundrequirement", "nestedrequirement"]
assert branches("Expressions.Cpp.Expressions.TemplateParameter") == ["typeparameter", "templatetemplateparameter", "parameterdeclaration"]
assert branches("Expressions.Cpp.Expressions.ExceptionSpecification") == ["noexceptspecifier", "throwspecifier"]
# As the user decided ("(b)"), one schema is a branch of every union whose positions hold it
for dialect, types, helpers in [("C.Expressions", "C.Types", "C.Types"), ("Cpp.Expressions", "Cpp.Types", "Cpp.Expressions")]:
    term_, initializer = (schemas(f"Expressions.{dialect}.{union}") for union in ("Term", "Initializer"))
    type_, specifier, attribute_ = schemas(f"Expressions.{types}.Term"), *(
        schemas(f"Expressions.{helpers}.{union}") for union in ("Specifier", "AttributeSpecifier"))
    assert term_["typeid"] is type_["typeid"] and term_["initializerlist"] is initializer["initializerlist"]
    assert type_["typeid"] is store.registered(f"Expressions.{types}.TypeId")  # the type name is its dialect's kind
    assert all(specifier[name] is schema for name, schema in attribute_.items())  # a type name's specifiers hold them
    assert term_["error"] is type_["error"] is schemas(f"Expressions.{dialect}.Pattern")["error"]  # the framework's
assert schemas("Expressions.C.Expressions.Name")["identifier"] is store.registered("Expressions.C.Expressions.Identifier")
assert store.registered("Expressions.C.Expressions.FieldDesignator").properties["name"].schema is store.registered("Expressions.C.Expressions.Identifier")
assert store.registered("Expressions.C.Expressions.MemberExpression").properties["member"].schema is store.registered("Expressions.C.Expressions.Name")
assert store.registered("Expressions.Cpp.Expressions.SimpleCapture").properties["name"].schema is store.registered("Expressions.Cpp.Expressions.Identifier")
# C++'s init capture holds its name as a Pattern, whose identifier is C++'s; C's Pattern is held by nothing, as C
# expressions bind no names
assert store.registered("Expressions.Cpp.Expressions.InitCapture").properties["name"].schema is store.registered("Expressions.Cpp.Expressions.Pattern")
assert schemas("Expressions.Cpp.Expressions.Pattern")["identifier"] is store.registered("Expressions.Cpp.Expressions.Identifier")
C_PATTERN = store.registered("Expressions.C.Expressions.Pattern")
assert not any(prop.schema is C_PATTERN or getattr(prop.schema, "element", None) is C_PATTERN
               for name in store.names() if name.startswith("Expressions.C.Expressions.")
               for prop in getattr(store.registered(name), "properties", {}).values())
assert schemas("Expressions.Cpp.Expressions.Name")["decltypespecifier"] is schemas("Expressions.Cpp.Expressions.Specifier")["decltypespecifier"]
assert schemas("Expressions.Cpp.Expressions.TemplateParameter")["parameterdeclaration"] is schemas("Expressions.Cpp.Expressions.Parameter")["parameterdeclaration"]
# Literals: C's and C++'s refine the framework's, adding what their text encodes
for dialect, encoded in [("C.Expressions", ["prefix", "quote", "base", "separators", "suffix"]),
                         ("Cpp.Expressions", ["prefix", "quote", "raw", "delimiter", "base", "separators", "suffix"])]:
    Literal = store.registered(f"Expressions.{dialect}.Literal")
    assert Literal.refines is store.registered("Expressions.Framework.Literal") and list(Literal.properties)[-len(encoded):] == encoded
encoded = cc(lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(1)).prefix("Q").quote("`").base(3)))
assert C.DIALECT.validate(encoded) == ["a literal's prefix cannot be 'Q'", "a literal's quote cannot be '`'", "a literal's base cannot be 3"]
assert C.DIALECT.validate(cc(C.literal(5, "z"))) == ["a literal's suffix cannot be 'z'"] and CPP.validate(cpp(CPPE.literal(5, "zu"))) == []
assert CPP.validate(cpp(CPPE.literal(5, "wb"))) == ["a literal's suffix cannot be 'wb'"] and CPP.validate(cpp(CPPE.literal(1.5, "bf16"))) == []
assert C.DIALECT.validate(cc(C.literal(True, "u"))) == ["a literal's suffix cannot be 'u'"]  # a bool has none
assert CPP.validate(cpp(CPPE.literal(2 ** 70, "z"))) == ["integer literal 1180591620717411303424 does not fit 'long'"]
# A character's quote is C's spelling, `'a'` an int as 97 is, and not C++'s, where `'a'` is a char
assert "quote" in C.DIALECT.kinds()["literal"].spelling and "quote" not in CPP.kinds()["literal"].spelling
problem = cc(lambda u: u.literal(lambda r: r.text("0x").problems(["a hexadecimal literal needs digits"])))
assert C.DIALECT.validate(problem) == ["a hexadecimal literal needs digits"] and CText.ToText(problem) == "0x"  # kept as written
# Constants: C23's and C++'s nullptr
assert C.DIALECT.validate(cc(lambda u: u.constant(lambda r: r.singleton("null")))) == ["a constant's singleton must be nullptr, got 'null'"]
assert C.DIALECT.validate(cc(lambda u: u.constant(lambda r: r))) == ["a constant needs a singleton"]
assert C.DIALECT.validate(cc(lambda u: u.constant(lambda r: r.text("nullpt").problems(["no such keyword"])))) == ["no such keyword"]
assert C.DIALECT.infer(cc(lambda u: u.constant(lambda r: r.singleton("nullptr")))).name() == "nullptr_t"
# Names: an identifier, as mbse-programs checks it, wherever one is held
assert C.DIALECT.validate(cc(C.identifier("1x")), {"1x"}) == ["'1x' is not an identifier"]
assert C.DIALECT.validate(cc(C.identifier("été")), {"été"}) == [] and C.DIALECT.validate(cc(C.identifier("$a")), {"$a"}) == []
assert C.DIALECT.validate(cc(C.member(C.identifier("x"), "a b")), {"x"}) == ["'a b' is not an identifier"]
assert C.DIALECT.validate(cc(lambda u: u.designatedinitializer(lambda r: r.designators([lambda d: d.fielddesignator(
    lambda f: f.name(lambda i: i.spelling("1a")))]).initializer(lambda i: i.equalinitializer(lambda e: e.value(C.literal(1))))))) == [
    "'1a' is not an identifier"]
assert CPP.validate(cpp(lambda u: u.idexpression(lambda r: r.name(lambda m: m.qualifiedname(lambda q: q.qualifiers([
    lambda n: n.identifier(lambda i: i.spelling("a-b"))]).name(lambda n: n.identifier(lambda i: i.spelling("c"))))))))  == ["'a-b' is not an identifier"]
assert C.DIALECT.validate(cc(lambda u: u.castexpression(lambda r: r.type(lambda t: t.typeid(lambda i: i.specifiers([
    lambda s: s.namedtypespecifier(lambda n: n.name(lambda m: m.identifier(lambda i: i.spelling("2t"))))]))).operand(C.literal(1))))) == ["type: '2t' is not an identifier"]  # within its type, a C Types term
# What each kind binds: a function declarator's, a lambda's or a requires expression's parameters, and a lambda's init
# captures, within it, and not beyond
x, w = CPPE.identifier("x"), CPPE.identifier("w")
lambda_ = lambda u: u.lambdaexpression(lambda r: r.captures([  # noqa: E731: [w = 1, x](int a, ...) noexcept(a + w) {}
    lambda c: c.initcapture(lambda i: i.name(lambda n: n.identifier(lambda j: j.spelling("w"))).initializer(lambda e: e.equalinitializer(lambda q: q.value(CPPE.literal(1))))),
    lambda c: c.simplecapture(lambda s: s.name(lambda n: n.spelling("x")))]).declarator(lambda d: d.parameters([
        lambda p: p.parameterdeclaration(lambda q: q.specifiers([lambda s: s.primitivetypespecifier(lambda k: k.keyword("int"))]).declarator(
            lambda e: e.iddeclarator(lambda i: i.name(lambda n: n.identifier(lambda j: j.spelling("a")))))),
        lambda p: p.ellipsisparameter(lambda e: e)]).exception(lambda e: e.noexceptspecifier(lambda n: n.condition(
            CPPE.binary("+", CPPE.identifier("a"), w))))))
assert CPP.validate(cpp(lambda_)) == [] and Cpp.Text.ToText(cpp(lambda_)) == "[w = 1, x](int a, ...) noexcept(a + w) {}"
assert CPP.validate(cpp(CPPE.binary(",", lambda_, w))) == ["right: idexpression 'w' is not bound"]
abstract = lambda u: u.requiresexpression(lambda r: r.parameters([lambda p: p.parameterdeclaration(  # noqa: E731: requires (int) { w; }
    lambda q: q.specifiers([lambda s: s.primitivetypespecifier(lambda k: k.keyword("int"))]))]).requirements([
    lambda q: q.simplerequirement(lambda s: s.expression(w))]))
assert CPP.validate(cpp(abstract)) == ["requirements 0 expression: idexpression 'w' is not bound"]  # an abstract parameter binds none
# Slots that may be empty, wherever a type name's helpers nest them; the others are needed
for dialect, spec in [
    (C.DIALECT, lambda u: u.sizeofexpression(lambda r: r.operand(lambda t: t.typeid(lambda i: i.specifiers([
        lambda s: s.primitivetypespecifier(lambda k: k.keyword("int"))]).declarator(lambda d: d.arraydeclarator(lambda a: a)))))),  # int[]
    (C.DIALECT, lambda u: u.conditionalexpression(lambda r: r.condition(C.literal(1)).alternative(C.literal(2)))),  # GNU's ?:
    (CPP, lambda u: u.foldexpression(lambda r: r.left(CPPE.literal(1)).operator("+"))), (CPP, lambda u: u.throwexpression(lambda r: r)),
]:
    assert dialect.validate(term(spec, dialect)) == []
for dialect, spec, problems in [
    (C.DIALECT, lambda u: u.designatedinitializer(lambda r: r), ["a designatedinitializer needs an initializer"]),
    (C.DIALECT, lambda u: u.compoundliteralexpression(lambda r: r), ["a compoundliteralexpression needs an initializer",
                                                                     "a compoundliteralexpression needs a type"]),
    (C.DIALECT, lambda u: u.memberexpression(lambda r: r.object(C.literal(1))), ["a memberexpression needs an operator", "a memberexpression needs a member"]),
    (C.DIALECT, lambda u: u.alignofexpression(lambda r: r.operand(C.literal(1))), ["an alignofexpression needs a keyword"]),
    (C.DIALECT, lambda u: u.genericselection(lambda r: r.controlling(C.literal(1)).associations([lambda a: a.type(lambda t: t.typeid(lambda i: i.specifiers([
        lambda s: s.primitivetypespecifier(lambda k: k.keyword("int"))])))])), [
        "a genericselection needs a value 0"]),  # an association, a pair of terms, labelled by its member
    (CPP, lambda u: u.userdefinedliteral(lambda r: r), ["an userdefinedliteral needs a literal", "an userdefinedliteral needs a suffix"]),  # the article by its letter
    (CPP, lambda u: u.idexpression(lambda r: r), ["an idexpression needs a name"]),
    (CPP, lambda u: u.namedcastexpression(lambda r: r), ["a namedcastexpression needs an operator", "a namedcastexpression needs a type",
                                                         "a namedcastexpression needs an operand"]),
    (CPP, lambda u: u.sizeofpackexpression(lambda r: r), ["a sizeofpackexpression needs a pack"]),
    (CPP, lambda u: u.newexpression(lambda r: r), ["a newexpression needs a type"]),
    (CPP, lambda u: u.namedcastexpression(lambda r: r.operator("bit_cast").type(lambda t: t.typeid(lambda i: i)).operand(CPPE.literal(1))), []),
]:
    assert dialect.validate(term(spec, dialect)) == problems, problems
assert CPP.validate(cpp(lambda u: u.namedcastexpression(lambda r: r.operator("bit_cast").type(lambda t: t.typeid(lambda i: i)).operand(CPPE.literal(1)))), core=True) == [
    "'bit_cast' is not a core operation"]
# C++ writes a name only as an idexpression, holding its Name, as C++'s grammar has it (C keeps the framework's
# variable): its writers write one, a lexical reference whose name is its identifier's, and C++'s union has no variable
assert "variable" not in CPP.kinds() and CPP.kinds()["idexpression"].role == F.REFERENCE and "variable" in C.DIALECT.kinds()
assert F.form(cpp(x)) == F.Form("idexpression", {"name": F.Part("identifier", {"spelling": "x"}, 0)}) and F.name_of(cpp(x)) == "x"
assert CPP.validate(cpp(x)) == ["idexpression 'x' is not bound"] and SY.free(cpp(CPPE.binary("+", x, w)), CPP) == {"x", "w"}
with raises(Exception, match="'variable' is not a member of a union value"):
    cpp(lambda u: u.variable(lambda r: r.name("x")))
cout = lambda u: u.idexpression(lambda r: r.name(lambda n: n.qualifiedname(lambda q: q.qualifiers([  # noqa: E731: std::cout
    lambda m: m.identifier(lambda i: i.spelling("std"))]).name(lambda m: m.identifier(lambda i: i.spelling("cout"))))))
f_w = lambda u: u.idexpression(lambda r: r.name(lambda n: n.templateid(lambda t: t.name(  # noqa: E731: f<w>
    lambda m: m.identifier(lambda i: i.spelling("f"))).arguments([w]))))
assert F.name_of(cpp(cout)) is None and CPP.validate(cpp(cout)) == [] and SY.free(cpp(cout), CPP) == set()  # the scope's alone
assert CPP.validate(cpp(f_w)) == ["name argument 0: idexpression 'w' is not bound"] and SY.free(cpp(f_w), CPP) == {"w"}  # its terms
# An init capture's name holds C++'s Pattern: an identifier, or invalid syntax, and no other kind
capture = lambda name: lambda u: u.lambdaexpression(lambda r: r.captures([lambda c: c.initcapture(  # noqa: E731: [name = 1] {}
    lambda i: i.name(name).initializer(lambda e: e.equalinitializer(lambda q: q.value(CPPE.literal(1)))))]))
invalid_capture = cpp(capture(lambda n: n.error(lambda e: e.text("1w"))))
assert CPP.validate(invalid_capture) == ["invalid syntax '1w'"] and Cpp.Text.ToText(invalid_capture) == "[1w = 1] {}"
assert C.bound_names(F.node("lambdaexpression", F.kind_of(invalid_capture)[1])) == frozenset()  # invalid syntax binds none
with raises(Exception, match="'idexpression' is not a member of a union value"):  # a term, not a pattern
    cpp(capture(w))
# A dialect's own kinds hold its terms alone: a C term is not a C++ term, nor a C++ term a C term
assert CPP.validate(term(C.binary("+", 1, 2), C.DIALECT)) == ["a C binaryexpression is not a Cpp term"]
assert C.DIALECT.validate(cpp(x), {"x"}) == ["a Cpp idexpression is not a C term"]
# Evaluation stays shallow: C++'s by C's rules, for the kinds C evaluated, a character a char; its own kinds refused by name
assert CPV.OfAny(cpp(CPPE.binary("+", CPPE.literal(1, "u"), 1))) == 2 and CPV.OfAny(cpp(CPPE.member(x, "a")), {"x": {"a": 1}}) == 1
assert CPV.OfAny(cpp(lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(97)).quote("'")))) == 97
assert CPP.infer(cpp(lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(97)).quote("'")))).name() == "char"
for spec, message in [
    (lambda_, "Cpp lambdaexpression is not evaluated"),
    (lambda u: u.memberexpression(lambda r: r.object(x).operator(".").member(lambda m: m.operatorname(lambda o: o.operator("+")))),
     "Cpp memberexpression of a member that is not an identifier is not evaluated"),
    (lambda u: u.memberexpression(lambda r: r.object(x).operator(".").template_keyword(True).member(lambda m: m.identifier(lambda i: i.spelling("a")))),
     "Cpp memberexpression of a member that is not an identifier is not evaluated"),
]:
    with raises(NotImplementedError, match=message):
        CPV.OfAny(cpp(spec), {"x": {"a": 1}})
assert CPV.EVALUATED == CV.EVALUATED - {"variable"} | {"idexpression"}
for held in (cout, lambda u: u.idexpression(lambda r: r.name(lambda n: n.identifier(lambda i: i)))):  # an identifier with no spelling
    with raises(NotImplementedError, match="Cpp idexpression of a name that is not an identifier is not evaluated"):
        CPV.OfAny(cpp(held))
with raises(NotImplementedError, match="Cpp callexpression of a callee that is not a function's name is not evaluated"):
    CPV.OfAny(cpp(CPPE.call(cout, 1)))
# C++'s types: its own (`bool`, the exact-width typedefs, `std::float16_t` to `std::float128_t`, `std::bfloat16_t`), then
# its holder's ABI's, as C's are; a C++ literal typed by C++'s names
assert CPP.infer(cpp(CPPE.literal(1.5, "f16"))).name() == "std::float16_t" and CPP.infer(cpp(CPPE.literal(1.5, "BF16"))).name() == "std::bfloat16_t"
assert CPV.OfAny(cpp(CPPE.binary("+", CPPE.literal(1.5, "bf16"), 1))) == "2.5" and CPV.OfAny(cpp(CPPE.binary("/", CPPE.literal(1.0, "bf16"), 3))) == "0.334"
assert CPV.OfAny(cpp(CPPE.literal(0.1, "f16"))) == 0.0999755859375 and CPP.infer(cpp(CPPE.cast(named_type("int32_t"), 1))).name() == "int32_t"
assert CPV.OfAny(cpp(CPPE.cast(keywords("char"), 200))) == -56 and CPV.OfAny(term(CPPE.cast(keywords("char"), 200), CPP, AAPCS64_)) == 200
CPT = CD.Types(term(0, CPP, AAPCS64_), "Cpp.Expressions")
assert CPT.named("std::float128_t") == CPT.named("long double") and CPT.named("std::bfloat16_t").format == (2, 8, 127)  # binary128, bfloat16
assert CPT.named("std::bfloat16_t") != CPT.named("std::float16_t") and CPT.named("std::float16_t") != CPT.named("std::bfloat16_t")  # 16 bits each
with raises(ValueError, match="'_Float16' is not a type Cpp or AAPCS64 defines"):  # C's own, not C++'s
    CPT.named("_Float16")
with raises(ValueError, match="'int' is not a type Cpp defines, and its expression names no ABI"):
    CPV.OfAny(term(CPPE.literal(5), CPP, None))
# Inference: sizeof and alignof give a size_t, noexcept a bool, a cast to a type that is not arithmetic what is not known
assert C.DIALECT.infer(cc(lambda u: u.sizeofexpression(lambda r: r.operand(C.literal(1))))).name() == "size_t"
assert CPP.infer(cpp(lambda u: u.noexceptexpression(lambda r: r.operand(CPPE.literal(1))))).name() == "bool"
assert CPP.infer(cpp(lambda u: u.sizeofpackexpression(lambda r: r.pack(lambda i: i.spelling("p"))))).name() == "size_t"
assert C.DIALECT.infer(cc(lambda u: u.castexpression(lambda r: r.type(lambda t: t.typeid(lambda i: i.specifiers([
    lambda s: s.primitivetypespecifier(lambda k: k.keyword("void"))]))).operand(C.literal(1))))) is D.Anything
assert C.DIALECT.infer(cc(lambda u: u.castexpression(lambda r: r.type(lambda t: t.typeid(lambda i: i.specifiers([
    lambda s: s.classspecifier(lambda k: k.key("struct").name(lambda n: n.identifier(lambda i: i.spelling("S"))))]))).operand(C.literal(1))))) is D.Anything
for parts in [lambda i: i, lambda i: i.specifiers([lambda s: s.namedtypespecifier(lambda n: n)]),
              lambda i: i.specifiers([lambda s: s.primitivetypespecifier(lambda k: k)])]:  # no specifier, a named type or a keyword with no name
    assert C.DIALECT.infer(cc(lambda u: u.castexpression(lambda r: r.type(lambda t: t.typeid(parts)).operand(C.literal(1))))) is D.Anything
invalid = cc(lambda u: u.castexpression(lambda r: r.type(lambda t: t.error(lambda e: e.text("int int"))).operand(C.literal(1))))
assert C.DIALECT.validate(invalid) == ["type: invalid syntax 'int int'"]  # invalid syntax in a type name, a C Types term
with raises(ValueError, match="C Types error is not evaluated"):  # refused by its owner's name, as every dialect refuses it
    CV.OfAny(invalid)
with raises(NotImplementedError, match="Cpp memberexpression of a member that is not an identifier is not evaluated"):  # no spelling
    CPV.OfAny(cpp(lambda u: u.memberexpression(lambda r: r.object(x).operator(".").member(lambda m: m.identifier(lambda i: i)))), {"x": {"a": 1}})
unnamed = lambda u: u.requiresexpression(lambda r: r.parameters([lambda p: p.parameterdeclaration(  # noqa: E731: requires (int <no name>) { w; }
    lambda q: q.declarator(lambda d: d.iddeclarator(lambda i: i.attributes([lambda a: a.alignasspecifier(lambda s: s.keyword("alignas").operand(CPPE.literal(1)))]))))]).requirements([
    lambda q: q.simplerequirement(lambda s: s.expression(w))]))
assert CPP.validate(cpp(unnamed)) == ["requirements 0 expression: idexpression 'w' is not bound"]
# The writers: C's and C++'s write a name as an identifier, and a cast's type is a spec of its family's Types' `Term`
assert F.form(cpp(CPPE.member(x, "a"))).attributes["member"] == F.Part("identifier", {"spelling": "a"}, 0)
assert F.form(cc(C.member(C.identifier("x"), "a"))).attributes["member"] == F.Part("identifier", {"spelling": "a"}, 0)
assert Cpp.Text.ToText(cpp(CPPE.cast(lambda t: t.typeid(lambda i: i.specifiers([
    lambda s: s.primitivetypespecifier(lambda p: p.keyword("unsigned")),
    lambda s: s.namedtypespecifier(lambda n: n.name(lambda m: m.identifier(lambda d: d.spelling("size_t")))),
])), x))) == "(unsigned size_t)x"
assert F.form(cpp(CPPE.call(CPPE.identifier("f"), 1))).kind == "callexpression"  # a callee that is a term
# An empty initializer list is held and read back present (mbse-schemas 0.26.9): `(int){}`, printed as written
empty_list = cc(lambda u: u.compoundliteralexpression(lambda r: r.type(lambda t: t.typeid(lambda i: i.specifiers([
    lambda s: s.primitivetypespecifier(lambda k: k.keyword("int"))]))).initializer(lambda li: li)))
assert C.DIALECT.validate(empty_list) == [] and round_trips(empty_list) and CText.ToText(empty_list) == "(int){}"
assert F.form(empty_list).attributes["initializer"] == F.Part("initializerlist", {}, 0)

<a id="DIA-25"></a>
## DIA-25 · C and C++ printed as they write them: every kind, its names, type names, lambdas and attributes, by precedence, a literal's default spelling with what it encodes

In [ ]:
CPPE = Cpp.Expressions


def c_text(spec):
    """A C term's source."""
    return CText.ToText(term(spec, C.DIALECT))


def cpp_text(spec):
    """A C++ term's source."""
    return Cpp.Text.ToText(term(spec, CPPE.DIALECT))


# Specs of terms and their helpers, written through the builders, each in any union whose branches it names
x, y, z = (lambda e, n=n: e.variable(lambda v: v.name(n)) for n in "xyz")
one, two = (lambda e, n=n: e.literal(lambda r: r.value(lambda v: v.as_scalar(n))) for n in (1, 2))
true_ = lambda e: e.literal(lambda r: r.value(lambda v: v.as_scalar(True)))  # noqa: E731
keyword = lambda word: lambda s: s.primitivetypespecifier(lambda p: p.keyword(word))  # noqa: E731
int_, char = keyword("int"), keyword("char")
const_ = lambda s: s.cvqualifier(lambda p: p.keyword("const"))  # noqa: E731
int_type = lambda t: t.typeid(lambda i: i.specifiers([int_]))  # noqa: E731
named = lambda n: lambda m: m.identifier(lambda i: i.spelling(n))  # noqa: E731: a name that is an identifier
T_, U_ = (lambda t, n=n: t.typeid(lambda i: i.specifiers([lambda s: s.namedtypespecifier(lambda m: m.name(named(n)))])) for n in "TU")
parameter = lambda specifier, name: lambda p: p.parameterdeclaration(lambda d: d.specifiers([specifier]).declarator(  # noqa: E731
    lambda e: e.iddeclarator(lambda i: i.name(name))))
attribute = lambda name: lambda a: a.standardattributespecifier(lambda s: s.attributes([lambda t: t.name(name)]))  # noqa: E731
# C: literals in their default spellings, with what their text encodes
for spec, text in [
    (lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(255)).base(16).suffix("u")), "0xffu"),
    (lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(255)).base(8)), "0377"),
    (lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(0)).base(8)), "0"),
    (lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(5)).base(2)), "0b101"),
    (lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(255)).base(10)), "255"),
    (lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(-1)).base(16)), "-0x1"),
    (lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(97)).quote("'")), "'a'"),
    (lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(39)).quote("'").prefix("L")), "L'\\''"),
    (lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(10)).quote("'")), "'\\n'"),
    (lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar('x"\0')).prefix("u8")), 'u8"x\\"\\0"'),
    (lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(1)).text("0b1")), "0b1"),
    (lambda u: u.constant(lambda r: r.singleton("nullptr")), "nullptr"),
    (lambda u: u.constant(lambda r: r.singleton("nullptr").text("nullptr")), "nullptr"),
    (lambda u: u.concatenatedstring(lambda r: r.parts([C.literal("a"), lambda p: p.variable(lambda v: v.name("PRId64"))])), '"a" PRId64'),
    # precedence: assignments, the comma, the conditional, unary operators' spacing, postfix, sizeof
    (C.DIALECT.spec(lambda u: u.assignmentexpression(lambda r: r.left(x).operator("=").right(
        lambda e: e.assignmentexpression(lambda a: a.left(y).operator("*=").right(one))))), "x = y *= 1"),
    (lambda u: u.assignmentexpression(lambda r: r.left(lambda e: e.assignmentexpression(lambda a: a.left(x).operator("=").right(y)))
        .operator("=").right(one)), "(x = y) = 1"),
    (lambda u: u.binaryexpression(lambda r: r.left(lambda e: e.assignmentexpression(lambda a: a.left(x).operator("=").right(one)))
        .operator("+").right(y)), "(x = 1) + y"),
    (C.binary(",", x, y), "x, y"), (C.call("f", C.binary(",", x, y)), "f((x, y))"),
    (lambda u: u.conditionalexpression(lambda r: r.condition(x).alternative(y)), "x ?: y"),
    (C.conditional(x, C.binary(",", y, z), 1), "x ? y, z : 1"),
    (C.unary("&", C.unary("*", x)), "&*x"), (C.unary("*", C.unary("*", x)), "* *x"), (C.unary("&", C.unary("&", x)), "& &x"),
    (C.unary("-", C.unary("--", x)), "- --x"), (C.unary("++", x), "++x"), (C.unary("!", C.unary("-", x)), "!-x"),
    (lambda u: u.postfixexpression(lambda r: r.operand(C.binary("+", x, 1)).operator("++")), "(x + 1)++"),
    (lambda u: u.sizeofexpression(lambda r: r.operand(x)), "sizeof x"),
    (lambda u: u.sizeofexpression(lambda r: r.operand(lambda e: e.parenthesizedexpression(lambda p: p.expression(x)))), "sizeof(x)"),
    (lambda u: u.sizeofexpression(lambda r: r.operand(C.binary("+", x, 1))), "sizeof(x + 1)"),
    (lambda u: u.alignofexpression(lambda r: r.keyword("_Alignof").operand(int_type)), "_Alignof(int)"),
    (C.unary("-", lambda e: e.sizeofexpression(lambda r: r.operand(x))), "-sizeof x"),
    # type names: specifiers and abstract declarators
    (lambda u: u.castexpression(lambda r: r.type(lambda t: t.typeid(lambda i: i.specifiers([char]).declarator(
        lambda d: d.pointerdeclarator(lambda p: p)))).operand(x)), "(char*)x"),
    (lambda u: u.castexpression(lambda r: r.type(lambda t: t.typeid(lambda i: i.specifiers([const_, char]).declarator(
        lambda d: d.pointerdeclarator(lambda p: p.qualifiers([lambda q: q.keyword("const")]).declarator(
            lambda e: e.pointerdeclarator(lambda q: q)))))).operand(x)), "(const char*const *)x"),
    (lambda u: u.castexpression(lambda r: r.type(lambda t: t.typeid(lambda i: i.specifiers([int_]).declarator(
        lambda d: d.functiondeclarator(lambda f: f.declarator(lambda e: e.pointerdeclarator(lambda p: p)).parameters([
            parameter(int_, named("n")), lambda p: p.parameterdeclaration(lambda q: q.specifiers([int_]).declarator(
                lambda e: e.arraydeclarator(lambda a: a.declarator(lambda g: g.iddeclarator(lambda i: i.name(lambda n: n.identifier(lambda j: j.spelling("a"))))).size(
                    lambda s: s.variable(lambda v: v.name("n"))))))]))))).operand(x)), "(int(*)(int n, int a[n]))x"),
    (lambda u: u.castexpression(lambda r: r.type(lambda t: t.typeid(lambda i: i.specifiers([int_]).declarator(
        lambda d: d.functiondeclarator(lambda f: f.declarator(lambda e: e.parenthesizeddeclarator(lambda p: p.declarator(
            lambda g: g.pointerdeclarator(lambda q: q)))).parameters([lambda p: p.parameterdeclaration(lambda q: q.attributes([
                attribute("maybe_unused")]).specifiers([int_])), lambda p: p.ellipsisparameter(lambda e: e)]).attributes([
            attribute("a")]))))).operand(x)), "(int(*)([[maybe_unused]] int, ...) [[a]])x"),
    (lambda u: u.sizeofexpression(lambda r: r.operand(lambda t: t.typeid(lambda i: i.specifiers([int_]).declarator(
        lambda d: d.arraydeclarator(lambda a: a.declarator(lambda e: e.pointerdeclarator(lambda p: p.attributes([attribute("a")])))
                                    .size(two).attributes([attribute("b")])))))), "sizeof(int(*[[a]])[2] [[b]])"),
    (lambda u: u.sizeofexpression(lambda r: r.operand(lambda t: t.typeid(lambda i: i.specifiers([int_]).declarator(
        lambda d: d.arraydeclarator(lambda a: a.static(True).qualifiers([lambda q: q.keyword("const")]).size(two)))))),
     "sizeof(int[static const 2])"),
    (lambda u: u.sizeofexpression(lambda r: r.operand(lambda t: t.typeid(lambda i: i.specifiers([int_]).declarator(
        lambda d: d.arraydeclarator(lambda a: a.star(True).static(True)))))), "sizeof(int[static *])"),
    (lambda u: u.sizeofexpression(lambda r: r.operand(lambda t: t.typeid(lambda i: i.specifiers([int_]).declarator(
        lambda d: d.functiondeclarator(lambda f: f.declarator(lambda e: e.iddeclarator(lambda g: g.name(lambda n: n.identifier(lambda j: j.spelling("f"))).attributes([attribute("c")])))))))),
     "sizeof(int f [[c]]())"),
    (lambda u: u.sizeofexpression(lambda r: r.operand(lambda t: t.typeid(lambda i: i.declarator(lambda d: d.pointerdeclarator(lambda p: p))))),
     "sizeof(*)"),  # a declarator without specifiers
    (lambda u: u.compoundliteralexpression(lambda r: r.type(lambda t: t.typeid(lambda i: i.specifiers([
        lambda s: s.classspecifier(lambda c: c.key("struct").attributes([attribute("deprecated")]).name(lambda n: n.identifier(lambda j: j.spelling("S"))).body(True))]))).initializer(
        lambda li: li.items([one]).trailing_comma(True))), "(struct [[deprecated]] S {}){1,}"),
    (lambda u: u.compoundliteralexpression(lambda r: r.type(lambda t: t.typeid(lambda i: i.specifiers([
        lambda s: s.declspecifier(lambda d: d.keyword("static")), lambda s: s.classspecifier(lambda c: c.key("union").body(True))]))).initializer(
        lambda li: li)), "(static union {}){}"),
    (lambda u: u.sizeofexpression(lambda r: r.operand(lambda t: t.typeid(lambda i: i.specifiers([
        lambda s: s.enumspecifier(lambda c: c.key("enum").attributes([attribute("a")]).name(lambda n: n.identifier(lambda j: j.spelling("E"))).base(int_type).body(True))])))),
     "sizeof(enum [[a]] E : int {})"),
    (lambda u: u.sizeofexpression(lambda r: r.operand(lambda t: t.typeid(lambda i: i.specifiers([
        lambda s: s.classspecifier(lambda c: c.key("struct").name(lambda n: n.identifier(lambda j: j.spelling("S")))), lambda s: s.enumspecifier(lambda c: c.key("enum"))])))),
     "sizeof(struct S enum)"),  # elaborated, without a body
    (lambda u: u.sizeofexpression(lambda r: r.operand(lambda t: t.typeid(lambda i: i.specifiers([
        lambda s: s.typeofspecifier(lambda c: c.keyword("typeof_unqual").operand(int_type)), lambda s: s.typeofspecifier(lambda c: c.keyword("typeof").operand(x)),
        lambda s: s.atomictypespecifier(lambda c: c.type(int_type)), lambda s: s.bitintspecifier(lambda c: c.width(two)),
        lambda s: s.placeholdertypespecifier(lambda c: c), lambda s: s.namedtypespecifier(lambda c: c.name(lambda n: n.identifier(lambda j: j.spelling("size_t"))))])))),
     "sizeof(typeof_unqual(int) typeof(x) _Atomic(int) _BitInt(2) auto size_t)"),
    (lambda u: u.sizeofexpression(lambda r: r.operand(lambda t: t.typeid(lambda i: i.specifiers([
        lambda s: s.standardattributespecifier(lambda a: a.attributes([lambda t2: t2.namespace("gnu").name("cold"),
                                                                       lambda t2: t2.name("deprecated").arguments([C.literal("x")])])),
        lambda s: s.alignasspecifier(lambda a: a.keyword("_Alignas").operand(int_type)), lambda s: s.alignasspecifier(lambda a: a.keyword("alignas").operand(two)),
        lambda s: s.gnuattributespecifier(lambda a: a.keyword("__attribute__").attributes([lambda t2: t2.name("packed"), lambda t2: t2.name("aligned").arguments([two])])),
        lambda s: s.declspecspecifier(lambda a: a.attributes([lambda t2: t2.name("dllexport"), lambda t2: t2.name("noinline")])), int_])))),
     'sizeof([[gnu::cold, deprecated("x")]] _Alignas(int) alignas(2) __attribute__((packed, aligned(2))) __declspec(dllexport noinline) int)'),
    # selections, designators, compound literals and GNU's extensions
    (lambda u: u.genericselection(lambda r: r.controlling(int_type).associations([lambda a: a.value(two)])), "_Generic(int, default: 2)"),
    (lambda u: u.initializerlist(lambda r: r.items([lambda i: i.designatedinitializer(lambda d: d.designators([
        lambda g: g.indexdesignator(lambda e: e.index(one).last(two)), lambda g: g.fielddesignator(lambda f: f.name(lambda j: j.spelling("a")))]).initializer(
        lambda n: n.initializerlist(lambda li: li.items([one, two]).trailing_comma(True)))),
        lambda i: i.designatedinitializer(lambda d: d.designators([lambda g: g.indexdesignator(lambda e: e.index(x))]).initializer(
            lambda n: n.equalinitializer(lambda e: e.value(C.binary(",", x, y)))))])), "{[1 ... 2].a{1, 2,}, [x] = (x, y)}"),
    (lambda u: u.extensionexpression(lambda r: r.operand(C.binary("+", x, 1))), "__extension__ (x + 1)"),
]:
    assert c_text(spec) == text, (text, c_text(spec))
assert C.DIALECT.validate(term(lambda u: u.castexpression(lambda r: r.type(lambda t: t.typeid(lambda i: i.specifiers([int_]).declarator(
    lambda d: d.functiondeclarator(lambda f: f.declarator(lambda e: e.pointerdeclarator(lambda p: p)).parameters([
        parameter(int_, named("n")), lambda p: p.parameterdeclaration(lambda q: q.specifiers([int_]).declarator(
            lambda e: e.arraydeclarator(lambda a: a.declarator(lambda g: g.iddeclarator(lambda i: i.name(lambda n: n.identifier(lambda j: j.spelling("a"))))).size(
                lambda s: s.variable(lambda v: v.name("n"))))))]))))).operand(x)), C.DIALECT), {"x"}) == []  # n, its parameter's
# C++: its own kinds, its names, lambdas, requires expressions and its declarators; a name used as an expression is an
# idexpression, holding its Name
name = lambda spelling: lambda u: u.idexpression(lambda r: r.name(named(spelling)))  # noqa: E731
x, y, z = (name(n) for n in "xyz")
qualified = lambda *parts: lambda m: m.qualifiedname(lambda q: q.qualifiers([named(p) for p in parts[:-1]]).name(named(parts[-1])))  # noqa: E731
template = lambda spelling, *arguments: lambda m: m.templateid(lambda t: t.name(named(spelling)).arguments(list(arguments)))  # noqa: E731
typename = lambda t: t.typeparameter(lambda p: p.key("typename").name(lambda j: j.spelling("T")))  # noqa: E731
for spec, text in [
    (lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar("a")).raw(True).delimiter("d").prefix("u8")), 'u8R"d(a)d"'),
    (lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(97)).quote("'")), "'a'"),
    (lambda u: u.userdefinedliteral(lambda r: r.literal(lambda li: li.value(lambda v: v.as_scalar("s"))).suffix("_s")), '"s"_s'),
    (lambda u: u.userdefinedliteral(lambda r: r.literal(lambda li: li.value(lambda v: v.as_scalar(1.5)).text("1.50")).suffix("_m")), "1.50_m"),
    # names
    (name("x"), "x"), (lambda u: u.idexpression(lambda r: r.name(qualified("std", "cout"))), "std::cout"),
    (lambda u: u.idexpression(lambda r: r.name(lambda m: m.qualifiedname(lambda q: q.global_scope(True).name(named("x"))))), "::x"),
    (lambda u: u.idexpression(lambda r: r.name(lambda m: m.operatorname(lambda o: o.operator("+")))), "operator+"),
    (lambda u: u.idexpression(lambda r: r.name(lambda m: m.operatorname(lambda o: o.operator("new[]")))), "operator new[]"),
    (lambda u: u.idexpression(lambda r: r.name(lambda m: m.conversionname(lambda o: o.type(int_type)))), "operator int"),
    (lambda u: u.idexpression(lambda r: r.name(lambda m: m.literaloperatorname(lambda o: o.suffix("_km")))), 'operator""_km'),
    (lambda u: u.memberexpression(lambda r: r.object(x).operator("->").member(lambda m: m.destructorname(lambda d: d.type(named("T"))))), "x->~T"),
    (lambda u: u.idexpression(lambda r: r.name(template("f", int_type, one, CPPE.binary(">", x, y), CPPE.binary(">>", x, one), CPPE.binary("<", x, y)))),
     "f<int, 1, (x > y), x >> 1, x < y>"),  # an argument holding `>`, parenthesized where it would end the list
    (lambda u: u.memberexpression(lambda r: r.object(x).operator(".").template_keyword(True).member(template("f", int_type))), "x.template f<int>"),
    (lambda u: u.idexpression(lambda r: r.name(lambda m: m.qualifiedname(lambda q: q.qualifiers([
        lambda n: n.decltypespecifier(lambda d: d.expression(x)), lambda n: n.templateid(lambda t: t.template_keyword(True).name(named("V")).arguments([T_]))]).name(
        named("type"))))), "decltype(x)::template V<T>::type"),
    (lambda u: u.thisexpression(lambda r: r), "this"),
    # lambdas: their bodies of statements written `{}`
    (lambda u: u.lambdaexpression(lambda r: r), "[] {}"),
    (lambda u: u.lambdaexpression(lambda r: r.captures([lambda c: c.defaultcapture(lambda d: d.mode("&")), lambda c: c.thiscapture(lambda t: t),
                                                        lambda c: c.thiscapture(lambda t: t.copy(True))]).declarator(lambda d: d)), "[&, this, *this]() {}"),  # an empty declarator, held: its parameter list
    (lambda u: u.lambdaexpression(lambda r: r.captures([
        lambda c: c.simplecapture(lambda s: s.by_reference(True).name(lambda j: j.spelling("x")).pack(True)), lambda c: c.simplecapture(lambda s: s.name(lambda j: j.spelling("y"))),
        lambda c: c.initcapture(lambda s: s.pack(True).name(named("z")).initializer(lambda i: i.equalinitializer(lambda e: e.value(one)))),
        lambda c: c.initcapture(lambda s: s.by_reference(True).name(named("w")).initializer(lambda i: i.initializerlist(lambda li: li.items([x]))))]).declarator(
        lambda d: d.parameters([lambda p: p.parameterdeclaration(lambda q: q.specifiers([lambda s: s.placeholdertypespecifier(lambda h: h)]).declarator(
            lambda e: e.packdeclarator(lambda k: k.declarator(lambda g: g.iddeclarator(lambda i: i.name(named("a")))))))]).specifiers([
            lambda s: s.declspecifier(lambda k: k.keyword("mutable")), lambda s: s.declspecifier(lambda k: k.keyword("constexpr"))]).exception(
            lambda e: e.noexceptspecifier(lambda n: n)).attributes([attribute("a")]).trailing_return(int_type).requires(true_))),
     "[&x..., y, ...z = 1, &w{x}](auto... a) mutable constexpr noexcept [[a]] -> int requires true {}"),
    (lambda u: u.lambdaexpression(lambda r: r.template_parameters([typename, lambda t: t.typeparameter(lambda p: p.key("class").pack(True).name(lambda j: j.spelling("Us"))),
        lambda t: t.typeparameter(lambda p: p.constraint(qualified("std", "integral")).name(lambda j: j.spelling("V")).default(int_type)),
        lambda t: t.typeparameter(lambda p: p),
        lambda t: t.templatetemplateparameter(lambda p: p.parameters([lambda q: q.typeparameter(lambda h: h.key("typename"))]).requires(true_).key("class").name(lambda j: j.spelling("TT"))
                                            .default(qualified("std", "vector"))),
        lambda t: t.parameterdeclaration(lambda q: q.specifiers([int_]).declarator(lambda e: e.iddeclarator(lambda i: i.name(named("N")))).default(one))])
        .template_requires(CPPE.binary("&&", CPPE.binary("||", x, y), z)).attributes([attribute("nodiscard")]).declarator(
            lambda d: d.parameters([parameter(lambda s: s.namedtypespecifier(lambda n: n.name(named("T"))), named("t"))]))),
     "[]<typename T, class... Us, std::integral V = int, typename, template <typename> requires true class TT = std::vector, int N = 1>"
     " requires (x || y) && z [[nodiscard]] (T t) {}"),
    (lambda u: u.lambdaexpression(lambda r: r.template_parameters([lambda t: t.templatetemplateparameter(
        lambda p: p.key("typename").pack(True).name(lambda j: j.spelling("TT")))])), "[]<template <> typename... TT> {}"),
    (lambda u: u.lambdaexpression(lambda r: r.declarator(lambda d: d.parameters([lambda p: p.parameterdeclaration(lambda q: q.specifiers([
        lambda s: s.placeholdertypespecifier(lambda h: h)]).declarator(lambda e: e.packdeclarator(lambda k: k)))]))), "[](auto...) {}"),
    # folds and requires expressions
    (lambda u: u.foldexpression(lambda r: r.operator("+").right(x)), "(... + x)"),
    (lambda u: u.foldexpression(lambda r: r.left(x).operator("&&").right(CPPE.binary("+", y, one))), "(x && ... && (y + 1))"),
    (lambda u: u.requiresexpression(lambda r: r), "requires {}"),
    (lambda u: u.requiresexpression(lambda r: r.parameters([parameter(lambda s: s.namedtypespecifier(lambda n: n.name(named("T"))), named("t"))]).requirements([
        lambda q: q.simplerequirement(lambda s: s.expression(CPPE.binary("+", CPPE.identifier("t"), 1))),
        lambda q: q.typerequirement(lambda s: s.name(qualified("T", "type"))),
        lambda q: q.compoundrequirement(lambda s: s.expression(CPPE.identifier("t")).noexcept(True).return_type(lambda m: m.templateid(
            lambda t: t.name(qualified("std", "same_as")).arguments([int_type])))),
        lambda q: q.compoundrequirement(lambda s: s.expression(CPPE.identifier("t"))),
        lambda q: q.nestedrequirement(lambda s: s.constraint(CPPE.binary("||", CPPE.binary("&&", true_, x), y)))])),
     "requires (T t) { t + 1; typename T::type; { t } noexcept -> std::same_as<int>; { t }; requires true && x || y; }"),
    # casts, typeid, sizeof..., alignof, noexcept, new and delete, throw, co_await and co_yield, packs
    (lambda u: u.functionalcastexpression(lambda r: r.type(lambda s: s.namedtypespecifier(lambda n: n.name(named("T")))).initializer(
        lambda i: i.initializerlist(lambda li: li.items([one])))), "T{1}"),
    (lambda u: u.namedcastexpression(lambda r: r.operator("reinterpret_cast").type(lambda t: t.typeid(lambda i: i.specifiers([char]).declarator(
        lambda d: d.pointerdeclarator(lambda p: p)))).operand(CPPE.binary(",", x, y))), "reinterpret_cast<char*>(x, y)"),  # its operand an expression, a comma among them
    (lambda u: u.typeidexpression(lambda r: r.operand(int_type)), "typeid(int)"),
    (lambda u: u.sizeofpackexpression(lambda r: r.pack(lambda j: j.spelling("Ts"))), "sizeof...(Ts)"),
    (lambda u: u.alignofexpression(lambda r: r.keyword("alignof").operand(x)), "alignof(x)"),
    (lambda u: u.newexpression(lambda r: r.global_scope(True).placement([x, one]).parenthesized_type(True).type(lambda t: t.typeid(lambda i: i.specifiers([int_]).declarator(
        lambda d: d.arraydeclarator(lambda a: a.size(two)))))), "::new (x, 1) (int[2])"),
    (lambda u: u.newexpression(lambda r: r.type(T_).initializer(lambda i: i.parenthesizedinitializer(lambda p: p.arguments([one, two])))), "new T(1, 2)"),
    (lambda u: u.newexpression(lambda r: r.type(int_type)), "new int"),
    (lambda u: u.deleteexpression(lambda r: r.global_scope(True).array(True).operand(x)), "::delete[] x"),
    (lambda u: u.deleteexpression(lambda r: r.operand(x)), "delete x"),
    (lambda u: u.throwexpression(lambda r: r.operand(CPPE.binary("+", x, 1))), "throw x + 1"),
    (lambda u: u.yieldexpression(lambda r: r.operand(lambda e: e.assignmentexpression(lambda a: a.left(x).operator("=").right(y)))), "co_yield x = y"),
    (CPPE.binary("+", lambda e: e.awaitexpression(lambda a: a.operand(x)), lambda e: e.throwexpression(lambda t: t)), "co_await x + (throw)"),
    (lambda u: u.packexpansion(lambda r: r.pattern(CPPE.binary("+", x, y))), "(x + y)..."),
    (lambda u: u.idexpression(lambda r: r.name(template("f", lambda e: e.packexpansion(lambda p: p.pattern(T_))))), "f<T...>"),
    # C++'s binary operators, several indices, designated initializers
    (CPPE.binary("<=>", x, y), "x <=> y"), (CPPE.binary("->*", CPPE.binary(".*", x, y), z), "x.*y->*z"),
    (CPPE.subscript(x, 1, 2), "x[1, 2]"),
    (lambda u: u.initializerlist(lambda r: r.items([lambda i: i.designatedinitializer(lambda d: d.designators([lambda g: g.fielddesignator(lambda f: f.name(lambda j: j.spelling("y")))]).initializer(
        lambda n: n.initializerlist(lambda li: li.items([two]))))])), "{.y{2}}"),
    # C++'s specifiers and declarators
    (lambda u: u.sizeofexpression(lambda r: r.operand(lambda t: t.typeid(lambda i: i.specifiers([
        lambda s: s.typenamespecifier(lambda n: n.name(qualified("T", "type"))), lambda s: s.placeholdertypespecifier(lambda p: p.decltype(True)),
        lambda s: s.placeholdertypespecifier(lambda p: p.constraint(qualified("std", "integral"))), lambda s: s.decltypespecifier(lambda d: d.expression(x))])))),
     "sizeof(typename T::type decltype(auto) std::integral auto decltype(x))"),
    (lambda u: u.sizeofexpression(lambda r: r.operand(lambda t: t.typeid(lambda i: i.specifiers([lambda s: s.classspecifier(lambda c: c.key("struct").name(named("S")).final(True).bases([
        lambda b: b.access("public").virtual(True).type(named("B")), lambda b: b.attributes([attribute("a")]).type(named("C")).pack(True)]).body(True)),
        lambda s: s.enumspecifier(lambda e: e.key("enum class").name(named("E")).base(int_type).body(True))])))),
     "sizeof(struct S final : virtual public B, [[a]] C... {} enum class E : int {})"),
    (lambda u: u.sizeofexpression(lambda r: r.operand(lambda t: t.typeid(lambda i: i.specifiers([int_]).declarator(lambda d: d.referencedeclarator(lambda e: e.rvalue(True).declarator(
        lambda g: g.pointerdeclarator(lambda p: p.scope(named("S")).qualifiers([lambda q: q.keyword("const")])))))))), "sizeof(int&&S::*const)"),
    (lambda u: u.sizeofexpression(lambda r: r.operand(lambda t: t.typeid(lambda i: i.specifiers([lambda s: s.placeholdertypespecifier(lambda h: h)]).declarator(
        lambda d: d.functiondeclarator(lambda f: f.declarator(lambda e: e.referencedeclarator(lambda g: g.attributes([attribute("r")]))).parameters([
            lambda p: p.parameterdeclaration(lambda q: q.attributes([attribute("a")]).this_keyword(True).specifiers([lambda s: s.placeholdertypespecifier(lambda h: h)]).declarator(
                lambda e: e.iddeclarator(lambda g: g.name(named("self"))))),
            lambda p: p.parameterdeclaration(lambda q: q.specifiers([int_]).declarator(lambda e: e.iddeclarator(lambda g: g.name(named("n")))).default(one))]).qualifiers([
            lambda q: q.keyword("const")]).ref_qualifier("&").exception(lambda e: e.throwspecifier(lambda h: h.types([int_type]))).trailing_return(int_type)))))),
     "sizeof(auto(&[[r]])([[a]] this auto self, int n = 1) const & throw(int) -> int)"),
    (lambda u: u.sizeofexpression(lambda r: r.operand(lambda t: t.typeid(lambda i: i.specifiers([
        lambda s: s.standardattributespecifier(lambda a: a.using_namespace("gnu").attributes([lambda t2: t2.name("cold"), lambda t2: t2.name("hot").pack(True)])),
        lambda s: s.alignasspecifier(lambda a: a.keyword("alignas").operand(T_).pack(True)), int_])))),
     "sizeof([[using gnu: cold, hot...]] alignas(T...) int)"),
]:
    assert cpp_text(spec) == text, (text, cpp_text(spec))
# Without a declarator, a lambda has no parameter list (`[] {}`); an empty one is held and read back (mbse-schemas 0.26.9)
empty_declarator = term(lambda u: u.lambdaexpression(lambda r: r.declarator(lambda d: d)), CPPE.DIALECT)
assert round_trips(empty_declarator) and Cpp.Text.ToText(empty_declarator) == "[]() {}"
assert F.form(empty_declarator).attributes["declarator"].kind == "lambdadeclarator"

<a id="DIA-26"></a>
## DIA-26 · C and C++ are complete: each named node type of tree-sitter-c and tree-sitter-cpp, kept as data, is a kind or a helper of its family's dialects (its expressions, its types, C's preprocessor), or listed as not an expression with its reason

In [ ]:
import json
from pathlib import Path

CPP = Cpp.Expressions.DIALECT
CT, XT, CPRE = CTypes.Expressions.DIALECT, CppTypes.Expressions.DIALECT, CPreprocessor.Expressions.DIALECT
for file, count, outside, dialects in [
        ("c-tree-sitter.json", 125, ["gnu_asm_expression"], {"Expressions": C.DIALECT, "Types": CT, "Preprocessor": CPRE}),
        ("cpp-tree-sitter.json", 206, ["generic_expression", "gnu_asm_expression", "requires_clause"],
         {"Expressions": CPP, "Types": XT, "Preprocessor": CPRE})]:  # C++ reads C's preprocessor
    grammar = json.loads(Path(f"../../grammars/{file}").read_text(encoding="utf-8"))  # the grammar's node types, as data
    mapped = {name: set() for name in dialects}
    for node in grammar["nodes"]:  # each a kind or a helper of one of its family's dialects, or both, or not an expression
        ways = {way for way in ("kinds", "helpers", "not") if way in node}
        assert ways and (ways <= {"kinds", "helpers"} or ways == {"not"}) and set(node) <= {"node", "supertypes"} | ways, node
        assert "not" not in node or (type(node["not"]) is str and node["not"]), node
        for way in ("kinds", "helpers"):
            for item in node.get(way, ()):
                name, _, tag = item.partition(".")
                assert name in dialects and tag, node
                dialect = dialects[name]
                if way == "kinds":  # a kind of its dialect's own, a branch of its `Term`
                    assert tag in dialect.kinds() and tag in branches(dialect.union), node
                    mapped[name].add(tag)
                else:  # a value object schema its dialect registers, not a term
                    schema = store.registered(f"Expressions.{dialect.path}.{tag}")
                    assert not schema.ref and tag.lower() not in branches(dialect.union), node
    assert len(grammar["nodes"]) == len({node["node"] for node in grammar["nodes"]}) == count  # every named node type, once
    for name, dialect in dialects.items():  # each of its dialects' kinds a node type's, but invalid syntax, which none is
        assert mapped[name] | {"error", "missing"} == set(dialect.kinds()), (file, name)
    assert [node["node"] for node in grammar["nodes"] if "expression" in node.get("supertypes", ()) and "not" in node] == outside


<a id="DIA-27"></a>
## DIA-27 · SystemVerilog's Expressions and Types: their kinds and unions, literals and constants, checks, names, binders, holders, and each dialect of the family reading the others' terms

In [ ]:
import functools

from mbse.Expressions.SystemVerilog.Patterns import Expressions as SVP
from mbse.Expressions.SystemVerilog.Assertions import Expressions as SVA
from mbse.Expressions.SystemVerilog.Constraints import Expressions as SVC
from mbse.Expressions.SystemVerilog.Preprocessor import Expressions as SVPP

FAMILY = [SV.DIALECT, SVT.DIALECT, SVP.DIALECT, SVA.DIALECT, SVC.DIALECT, SVPP.DIALECT]
union = lambda dialect: [b.name for b in store.registered(dialect.union).branches]  # noqa: E731: its Term's branches
spelled = lambda text: lambda i: i.spelling(text)  # noqa: E731: an identifier, of its spelling
name = lambda text: lambda u: u.nameexpression(lambda n: n.name(lambda i: i.identifier(spelled(text))))  # noqa: E731
literal = lambda value, **encoded: lambda u: u.literal(lambda r: functools.reduce(  # noqa: E731: a literal, with what its text encodes
    lambda b, item: getattr(b, item[0])(item[1]), encoded.items(), r.value(lambda p: p.as_scalar(value))))
constant = lambda **parts: lambda u: u.constant(lambda r: functools.reduce(  # noqa: E731: a constant, of its parts
    lambda b, item: getattr(b, item[0])(item[1]), parts.items(), r))
# Its kinds: mbse-programs' Verilog expressions, named and shaped as it has them, 41 branches with the Preprocessor's
# macro usage and invalid syntax; each sublanguage's Term lists them beside its own, but the Preprocessor's.
assert union(SV.DIALECT) == [*SV.TAGS, "error", "missing", "macrousage"] and len(union(SV.DIALECT)) == 41
assert union(SVT.DIALECT) == [*SV.TAGS, "error", "missing", *SVT.TAGS, "macrousage"]
assert union(SVP.DIALECT) == [*SV.TAGS, "error", "missing", *SVP.TAGS, "macrousage"]
assert union(SVA.DIALECT) == [*SV.TAGS, "error", "missing", *SVA.TAGS, "distexpression", "macrousage"]  # Constraints' too
assert union(SVC.DIALECT) == [*SV.TAGS, "error", "missing", *SVC.TAGS, "macrousage", *SVPP.DIRECTIVES, "comment"]
assert union(SVPP.DIALECT) == [*SVPP.TAGS, "error", "missing"] and list(SV.DIALECT.kinds()) == [*SV.TAGS, "error", "missing"]
kinds = {b.name: b.schema for b in store.registered(SVT.DIALECT.union).branches}  # each schema in every union that holds it
assert all(kinds[tag] is {b.name: b.schema for b in store.registered(SV.DIALECT.union).branches}[tag] for tag in SV.TAGS)
assert [b.name for b in store.registered("Expressions.SystemVerilog.Expressions.Name").branches] == [
    "identifier", "scopedname", "localname", "parameterizedname", "interfacetypename", "interfacemethodname", "unitname",
    "superexpression"]
assert [b.name for b in store.registered("Expressions.SystemVerilog.Types.Dimension").branches] == [
    "rangedimension", "sizedimension", "unsizeddimension", "associativedimension", "queuedimension"]
assert list(kinds["binaryexpression"].properties) == ["left", "operator", "attributes", "right"]  # in mbse-programs' order
assert F._parts(kinds["callexpression"]) == (("callee",), ("arguments",), ("attributes",))
assert kinds["castexpression"].properties["type"].schema is store.registered(SVT.DIALECT.union)  # a sublanguage's position
assert kinds["callexpression"].properties["arguments"].schema.element is store.registered(SVA.DIALECT.union)
assert kinds["matchesexpression"].properties["pattern"].schema is store.registered(SVP.DIALECT.union)
assert kinds["randomizewithexpression"].properties["items"].schema.element is store.registered(SVC.DIALECT.union)
assert kinds["structtype"].properties["members"].schema.element is store.registered("Expressions.SystemVerilog.Types.StructMember")
# Literals (Q3, "(b)"): `literal`, the framework's refined, holds what a scalar schema can, with what its text encodes;
# `constant` what none can: `null`, a fill, or a 4-state vector, exactly one of them.
Literal = kinds["literal"]
assert Literal.refines is store.registered("Expressions.Framework.Literal") and list(Literal.properties)[3:] == [
    "base", "signed", "separators", "unit", "triple"] and SV.SPELLING == ("text", "problems", "separators", "triple")
assert list(kinds["constant"].properties) == ["singleton", "fill", "bits", "size", "signed", "base", "text", "problems"]
for spec, problems in [
    (literal(255, base="h"), []), (literal(10.0, unit="ns"), []), (literal("s", triple=True), []),
    (literal(1, base="q"), ["a literal's base must be b, o, d or h, got 'q'"]),
    (literal(10, unit="ns"), ["a time literal's unit must be s, ms, us, ns, ps or fs, its value a float, got 'ns'"]),
    (literal(1.0, unit="ks"), ["a time literal's unit must be s, ms, us, ns, ps or fs, its value a float, got 'ks'"]),
    (constant(fill="1"), []), (constant(singleton="null"), []), (constant(bits="1x0z"), []), (constant(bits="10", size=2), []),
    (constant(), ["a constant holds a singleton, a fill or bits, exactly one of them"]),
    (constant(fill="1", bits="1"), ["a constant holds a singleton, a fill or bits, exactly one of them"]),
    (constant(text="8'hzq", problems=["'q' is not a hexadecimal digit"]), ["'q' is not a hexadecimal digit"]),
    (constant(singleton="nil"), ["a constant's singleton must be null, got 'nil'"]),
    (constant(fill="2"), ["a constant's fill must be 0, 1, x or z, got '2'"]),
    (constant(bits="10", size=3), ["a constant's size must be its number of bits, 2, got 3"]),
]:
    assert validate(spec) == problems, (problems, validate(spec))
# mbse-programs' checks, with its messages, within a kind and its helpers alike; and a choice among a few.
x = name("x")
for spec, problems in [
    (name("2x"), ["'2x' is not an identifier"]), (name("\\a+b"), []), (name("$x"), ["'$x' is not an identifier"]),
    (name("\\"), ["'\\\\' is not an identifier"]), (name(""), ["nameexpression '' is not bound"]),  # an empty one, no check
    (SV.call("$", x), ["'$' is not a system task or function name"]),
    (lambda u: u.systemcall(lambda r: r.name("")), ["a systemcall needs a name"]),
    (lambda u: u.rangeselect(lambda r: r.value(x).left(SV.literal(1)).operator("::").right(SV.literal(0))), ["RangeSelect.operator cannot be '::'"]),
    (lambda u: u.valuerange(lambda r: r.left(SV.literal(1)).operator("+-").right(SV.literal(2))), ["ValueRange.operator cannot be '+-'"]),
    (lambda u: u.streamingconcatenation(lambda r: r.operator("<>").items([lambda i: i.expression(x)])), ["StreamingConcatenation.operator cannot be '<>'"]),
    (lambda u: u.streamingconcatenation(lambda r: r.operator("<<").items([lambda i: i.expression(x).left(SV.literal(0)).operator(":")])),
     ["a StreamItem's `with` has a left, and an operator and a right, or neither"]),
    (lambda u: u.streamingconcatenation(lambda r: r.operator("<<").items([lambda i: i.expression(x).left(SV.literal(0)).operator("::").right(SV.literal(1))])),
     ["StreamItem.operator cannot be '::'"]),
    (lambda u: u.assignmentpattern(lambda r: r.count(SV.literal(2)).items([lambda i: i.patternitem(lambda p: p.value(x))])),
     ["a repeated AssignmentPattern has positional items only"]),
    (lambda u: u.predicateexpression(lambda r: r.conditions([x])), ["a PredicateExpression has two conditions or more"]),
    (lambda u: u.newexpression(lambda r: r.defaulted(True)), ["a defaulted NewExpression is super.new(default)"]),
    (lambda u: u.newexpression(lambda r: r.scope(lambda s: s.superexpression(lambda e: e)).defaulted(True).arguments([x])),
     ["a defaulted NewExpression is super.new(default)"]),
    (lambda u: u.binaryexpression(lambda r: r.left(x).operator("+").attributes([lambda a: a.specs([lambda s: s.name(spelled("9"))])]).right(x)),
     ["'9' is not an identifier"]),
    (lambda u: u.macrousage(lambda r: r.name(spelled("1W"))), ["'1W' is not an identifier"]),
    (SV.cast(lambda t: t.integervectortype(lambda v: v.keyword("int")), x), ["type: IntegerVectorType.keyword cannot be 'int'"]),
    (SV.cast(lambda t: t.integeratomtype(lambda v: v.keyword("int").signing("both")), x), ["type: IntegerAtomType.signing cannot be 'both'"]),
    (SV.cast(lambda t: t.integeratomtype(lambda v: v), x), ["type: an integeratomtype needs a keyword"]),
    (SV.cast(lambda t: t.structtype(lambda s: s.keyword("struct").qualifier("tagged")), x), ["type: a StructType with a qualifier is a union"]),
    (SV.cast(lambda t: t.structtype(lambda s: s.keyword("union").members([lambda m: m.random("maybe").type(
        lambda y: y.keywordtype(lambda k: k.keyword("string"))).declarators([lambda d: d.name(spelled("1a"))])])), x),
     ["type: StructMember.random cannot be 'maybe'", "type: '1a' is not an identifier"]),
    (SV.cast(lambda t: t.enumtype(lambda e: e.members([lambda m: m.name(spelled("A")).right(SV.literal(1))])), x), ["type: an EnumMember with a right has a left"]),
    (SV.cast(lambda t: t.integervectortype(lambda v: v.keyword("logic").dimensions([lambda d: d.rangedimension(lambda r: r.left(SV.literal(1)))])), x),
     ["type: an integervectortype needs a dimensions 0 right"]),
]:
    assert validate(spec, {"x", "\\a+b"}) == problems, (problems, validate(spec, {"x", "\\a+b"}))  # an escaped one, bound too
# Names (Q4, "(b)"): a name used as a value is a nameexpression holding its Name; an identifier's must be bound, any
# other name is resolved by the scope alone, the terms it holds (a parameterized name's parameters) checked in scope.
scoped = lambda scope, inner: lambda u: u.nameexpression(lambda n: n.name(lambda s: s.scopedname(lambda c: c.scope(scope).name(inner))))  # noqa: E731
p_x = scoped(lambda y: y.identifier(spelled("p")), lambda y: y.identifier(spelled("x")))  # p::x
c_w = scoped(lambda y: y.parameterizedname(lambda q: q.name(spelled("c")).parameters([name("w")])), lambda y: y.identifier(spelled("x")))
assert validate(p_x) == [] and F.name_of(sv(p_x)) is None and F.name_of(sv(x)) == "x" and SY.free(sv(c_w), SV.DIALECT) == {"w"}
assert validate(c_w) == ["name scope argument 0: nameexpression 'w' is not bound"] and validate(c_w, {"w"}) == []
assert validate(lambda u: u.nameexpression(lambda n: n)) == ["a nameexpression needs a name"]
assert "variable" not in SV.DIALECT.kinds()  # SystemVerilog writes a name only as a nameexpression, as C++ an idexpression
# Binders refine Let (Q8, "(a)"): an array method's `with` binds its iterator, its call's argument or `item`, within its
# call and its `with` expression; a `matches` binds its pattern's variables (Patterns' to write) in its conditional's
# consequence and the conditions after it.
summed = sv(SV.iterate(x, "sum", "q", SV.binary("+", SV.identifier("q"), name("r"))))
assert SY.free(summed, SV.DIALECT) == {"x", "r"} and validate(SV.iterate(x, "sum", "q", SV.identifier("q")), {"x"}) == []
implicit = lambda u: u.arraymethodwithexpression(lambda r: r.value(SV.method(x, "sum")).body(name("item")))  # noqa: E731: item
assert validate(implicit, {"x"}) == [] and SY.free(sv(implicit), SV.DIALECT) == {"x"} and F.is_binder(F.kind_of(summed)[1].as_schema())
assert SV.DIALECT.kinds()["arraymethodwithexpression"].role == F.BINDING and F.name_of(summed) is None  # Let's name unset
assert SV.DIALECT.kinds()["arraymethodwithexpression"].scopes(F.kind_of(sv(lambda u: u.arraymethodwithexpression(
    lambda r: r.body(SV.literal(1)))))[1]) == [frozenset(), frozenset()]  # no call: nothing bound
assert SV.DIALECT.infer(summed, {"x": D.Anything, "r": D.Anything}) is D.Anything
scoped_iterator = SV.DIALECT.kinds()["arraymethodwithexpression"].scopes(F.kind_of(sv(lambda u: u.arraymethodwithexpression(
    lambda r: r.value(SV.method(x, "sum", p_x)).body(name("item")))))[1])
assert scoped_iterator == [frozenset({"item"})] * 2  # a call's argument that is no identifier names no iterator
assert repr(SV.DIALECT.infer(sv(constant(bits="x01")))) == "logic [31:0]" and repr(SV.DIALECT.infer(sv(constant(bits="x01", size=3)))) == "logic [2:0]"
assert SV.DIALECT.infer(sv(literal(10.0, unit="ns"))) is D.Anything and infer(SV.binary("<<=", x, 1), {"x": D.Anything}) is D.Anything
matched = lambda pattern: lambda u: u.matchesexpression(lambda m: m.value(x).pattern(pattern))  # noqa: E731
choice = sv(lambda u: u.conditionalexpression(lambda c: c.condition(matched(SV.literal(1))).consequence(name("v")).alternative(SV.literal(0))))
assert SVP.bound_names(F.form(F.form(choice).arguments[0]).arguments[1]) == frozenset()  # a constant pattern binds none
assert SY.free(sv(SV.conditional(x, name("v"), 0)), SV.DIALECT) == {"x", "v"}  # a plain condition binds none
assert SY.free(choice, SV.DIALECT) == {"x", "v"} and SY.free(sv(lambda u: u.predicateexpression(lambda p: p.conditions([
    matched(SV.literal(1)), name("v")]))), SV.DIALECT) == {"x", "v"}
assert SV.DIALECT.kinds()["conditionalexpression"].scopes(F.kind_of(sv(lambda u: u.conditionalexpression(
    lambda c: c.consequence(x).alternative(x))))[1]) == [frozenset()] * 3  # no condition: nothing matched
assert SY.free(sv(lambda u: u.conditionalexpression(lambda c: c.condition(lambda p: p.predicateexpression(lambda q: q.conditions([
    matched(SV.literal(1)), x]))).consequence(name("v")).alternative(SV.literal(0)))), SV.DIALECT) == {"x", "v"}  # a `&&&`'s matches
# Each dialect of the family reads the others' terms, where its positions hold them: a cast's type is a Types term,
# a dimension's bound an Expressions term, a call's argument an Assertions term; another's is refused as before.
cast = sv(SV.cast(lambda t: t.integervectortype(lambda v: v.keyword("logic").dimensions([
    lambda d: d.rangedimension(lambda r: r.left(name("w")).right(SV.literal(0)))])), x))
typed = F.form(cast).arguments[0]
assert F.dialect_of(typed) == SVT.DIALECT.union and F.dialect_of(F.form(typed).arguments[0]) == SV.DIALECT.union
assert SV.DIALECT.validate(cast, {"x"}) == ["type: dimensions 0 left: nameexpression 'w' is not bound"]
assert SVT.DIALECT.validate(typed) == ["dimensions 0 left: nameexpression 'w' is not bound"] and SY.free(cast, SV.DIALECT) == {"x", "w"}
call_ = sv(SV.call("f", x))
assert F.dialect_of(F.form(call_).arguments[1]) == SVA.DIALECT.union and SVA.DIALECT.validate(F.form(call_).arguments[1], {"x"}) == []
assert all(d.reads() == tuple(o for o in FAMILY if o is not d) for d in FAMILY)
assert SVA.DIALECT.kind(F.form(call_).arguments[1])[2] is SV.DIALECT.kinds()["nameexpression"]  # a kind its Term lists
assert SV.DIALECT.infer(cast, {"x": D.Anything, "w": D.Anything}) is D.Anything and SVT.DIALECT.infer(typed, {"w": D.Anything}) is D.Anything
with raises(ValueError, match="a SystemVerilog Types integervectortype is not a Python3 term"):
    P.DIALECT.kind(typed)
assert P.DIALECT.validate(typed) == ["a SystemVerilog Types integervectortype is not a Python3 term"]
assert SVP.DIALECT.kind_named("structtype") is SVT.DIALECT.kinds()["structtype"]  # a kind it reads, which its Term lists not:
with raises(Exception, match="'structtype' is not a member of a union value"):  # the schema refuses it
    with store.transaction():
        store.Expressions.SystemVerilog.Patterns.Expression(lambda e: e.root(SVP.DIALECT.make(F.Form("structtype", {"keyword": "struct"}))))
with raises(ValueError, match="SystemVerilog Preprocessor has no kind 'nope'"):
    SVPP.DIALECT.kind_named("nope")
# Each dialect its holder (Q2, "(a)"), refining the framework's Expression, its root narrowed to its Term, holding its
# kinds and those its Term lists alone.
for dialect in FAMILY:
    Holder = store.registered(f"Expressions.{dialect.path}.Expression")
    assert Holder.refines is store.registered("Expressions.Framework.Expression") and Holder.properties["root"].schema is store.registered(dialect.union)
with store.transaction():
    held = store.Expressions.SystemVerilog.Types.Expression(lambda e: e.root(lambda t: t.integeratomtype(lambda a: a.keyword("int"))))
assert SVT.DIALECT.validate(held) == [] and SVText.ToText(held) == "int" and F.dialect_of(F.term(held)) == SVT.DIALECT.union
with raises(Exception):  # the schema refuses a kind its root's union lacks
    with store.transaction():
        store.Expressions.SystemVerilog.Expressions.Expression(lambda e: e.root(lambda t: t.integeratomtype(lambda a: a.keyword("int"))))

<a id="DIA-28"></a>
## DIA-28 · SystemVerilog's printing: mbse-programs' printer ported, its precedence, layout and spellings, for every kind of Expressions and Types and the Preprocessor's macro usage, from any holder of the family

In [ ]:
# Printing ports mbse-programs' printer: IEEE 1800's precedence, its layout and spellings, for every kind of Expressions
# and Types and the Preprocessor's macro usage, from any holder of the family; the samples' every kind (FRM-02).
x, y = name("x"), name("y")
attributed = lambda *names: [lambda a: a.specs([lambda s, n=n: s.name(spelled(n)) for n in names])]  # noqa: E731: (* a, b *)
vector_type = lambda keyword, *dimensions: lambda t: t.integervectortype(lambda v: v.keyword(keyword).dimensions(list(dimensions)))  # noqa: E731
packed = lambda left, right: lambda d: d.rangedimension(lambda r: r.left(SV.literal(left)).right(SV.literal(right)))  # noqa: E731
cases = [
    # precedence: an assignment, a tagged value, a `matches` and a `&&&` bind loosest, parenthesized in an operand
    (SV.binary("+", lambda u: u.assignmentexpression(lambda a: a.target(x).operator("=").value(SV.literal(1))), y), "(x = 1) + y"),
    (lambda u: u.assignmentexpression(lambda a: a.target(x).operator("<<=").value(SV.binary("+", y, 1))), "x <<= y + 1"),
    (SV.unary("-", lambda u: u.incrementexpression(lambda i: i.operator("++").operand(x))), "-(++x)"),
    (lambda u: u.incrementexpression(lambda i: i.operator("--").postfix(True).operand(SV.binary("+", x, y))), "(x + y)--"),
    (lambda u: u.incrementexpression(lambda i: i.operator("++").operand(x).attributes(attributed("a"))), "++ (* a *) x"),
    (lambda u: u.incrementexpression(lambda i: i.operator("++").postfix(True).operand(x).attributes(attributed("a"))), "x (* a *)++"),
    (lambda u: u.unaryexpression(lambda i: i.operator("!").operand(x).attributes(attributed("a", "b"))), "! (* a, b *) x"),
    (lambda u: u.binaryexpression(lambda i: i.left(x).operator("+").attributes(attributed("a")).right(y)), "x + (* a *) y"),
    (lambda u: u.conditionalexpression(lambda c: c.condition(x).attributes(attributed("a")).consequence(SV.conditional(y, 1, 2)).alternative(y)),
     "x ? (* a *) (y ? 1 : 2) : y"),
    (lambda u: u.conditionalexpression(lambda c: c.condition(lambda p: p.predicateexpression(lambda q: q.conditions([x, y]))).consequence(x).alternative(y)),
     "x &&& y ? x : y"),
    (lambda u: u.taggedexpression(lambda t: t.member(spelled("a")).value(lambda v: v.taggedexpression(lambda w: w.member(spelled("b")).value(SV.binary("+", x, 1))))),
     "tagged a tagged b (x + 1)"),
    (lambda u: u.taggedexpression(lambda t: t.member(spelled("None"))), "tagged None"),
    (lambda u: u.mintypmaxexpression(lambda m: m.min(x).typ(SV.binary("+", x, 1)).max(y)), "x:x + 1:y"),
    # selects and calls
    (lambda u: u.rangeselect(lambda r: r.value(SV.binary("+", x, y)).left(x).operator("+:").right(SV.literal(4))), "(x + y)[x +: 4]"),
    (lambda u: u.callexpression(lambda c: c.callee(x)), "x()"),
    (lambda u: u.systemcall(lambda c: c.name("$finish")), "$finish"),
    (lambda u: u.systemcall(lambda c: c.name("$finish").attributes(attributed("a"))), "$finish (* a *) ()"),
    (SV.call("$bits", vector_type("bit", packed(3, 0))), "$bits(bit [3:0])"),
    (lambda u: u.newexpression(lambda n: n), "new"), (lambda u: u.newexpression(lambda n: n.scope(lambda s: s.superexpression(lambda e: e)).arguments([x])), "super.new(x)"),
    (lambda u: u.newarrayexpression(lambda n: n.size(x)), "new[x]"),
    (lambda u: u.randomizewithexpression(lambda r: r.call(SV.method(x, "randomize")).items([])), "x.randomize() with {}"),
    (lambda u: u.randomizewithexpression(lambda r: r.call(SV.method(x, "randomize")).items([SV.binary(">", x, 0), lambda i: i.macrousage(lambda m: m.name(spelled("C")))])),
     "x.randomize() with { x > 0; `C }"),
    (lambda u: u.streamingconcatenation(lambda s: s.operator(">>").items([lambda i: i.expression(x), lambda i: i.expression(y).left(SV.literal(0))])),
     "{>>{x, y with [0]}}"),
    (lambda u: u.streamingconcatenation(lambda s: s.operator("<<").slice(SV.literal(4)).items([lambda i: i.expression(x).left(SV.literal(0)).operator(":").right(SV.literal(3))])),
     "{<< 4 {x with [0:3]}}"),
    (lambda u: u.assignmentpattern(lambda a: a.type(vector_type("logic", packed(1, 0))).count(SV.binary("+", x, 1)).items([x, y])), "logic [1:0]'{(x + 1){x, y}}"),
    (lambda u: u.assignmentpattern(lambda a: a.items([lambda i: i.patternitem(lambda p: p.key(lambda k: k.integeratomtype(lambda t: t.keyword("int"))).value(SV.literal(0)))])),
     "'{int: 0}"),
    (SV.cast(SV.binary("+", x, 1), y), "(x + 1)'(y)"), (SV.cast(lambda t: t.typereference(lambda r: r.operand(x)), y), "type(x)'(y)"),
    (lambda u: u.namedconnection(lambda c: c.name(spelled("a"))), ".a()"),
    (SV.inside(x, lambda u: u.valuerange(lambda r: r.left(SV.literal(1)).operator("+/-").right(SV.literal(2)))), "x inside {[1 +/- 2]}"),
    (lambda u: u.streamingconcatenation(lambda s: s.operator("<<").items([lambda i: i.expression(x).left(SV.literal(0)).operator("+:").right(SV.literal(4))])),
     "{<<{x with [0 +: 4]}}"),
    # literals in their spelling, or their value's default one; constants in their base where it writes their bits
    (literal(255, base="h"), "'hff"), (literal(5, base="d", signed=True), "'sd5"), (literal(-5, base="o"), "-'o5"), (literal(5, base="b"), "'b101"),
    (literal(10.0, unit="ns"), "10ns"), (literal(1.5, unit="us"), "1.5us"), (literal('a"\n', triple=True), '"""a"\n"""'),
    (literal(8, text="4'h8"), "4'h8"), (constant(fill="z"), "'z"), (constant(singleton="null"), "null"),
    (constant(bits="xxxxzzzz", base="h"), "'hxz"), (constant(bits="x1", base="h"), "'bx1"), (constant(bits="1010", size=4, base="d"), "4'd10"),
    (constant(bits="0x", size=2, base="d"), "2'b0x"), (constant(text="8'hFF", bits="11111111", size=8), "8'hFF"),
    (SV.unary("-", literal(255, base="h")), "-'hff"), (SV.unary("-", literal(-5, base="o")), "-(-'o5)"),
]
for spec, source in cases:
    assert svtext(spec) == source, (source, svtext(spec))
# Types: data types, their dimensions and members, on one line, written from a Types holder too
typetext = lambda spec: SVText.ToText(term(spec, SVT.DIALECT))  # noqa: E731
types = [
    (lambda t: t.namedtype(lambda n: n.name(lambda i: i.identifier(spelled("t")))), "t"),
    (vector_type("reg"), "reg"), (lambda t: t.integeratomtype(lambda a: a.keyword("int").signing("unsigned")), "int unsigned"),
    (lambda t: t.keywordtype(lambda k: k.keyword("chandle")), "chandle"), (lambda t: t.implicittype(lambda i: i.dimensions([packed(7, 0)])), "[7:0]"),
    (lambda t: t.implicittype(lambda i: i.signing("unsigned").dimensions([packed(7, 0)])), "unsigned [7:0]"),
    (lambda t: t.virtualinterfacetype(lambda v: v.interface(spelled("bus"))), "virtual bus"),
    (lambda t: t.virtualinterfacetype(lambda v: v.interface_keyword(True).interface(spelled("bus"))), "virtual interface bus"),
    (lambda t: t.structtype(lambda s: s.keyword("struct")), "struct {  }"),
    (lambda t: t.structtype(lambda s: s.keyword("union").qualifier("soft").members([lambda m: m.type(vector_type("bit")).declarators([
        lambda d: d.name(spelled("a")), lambda d: d.name(spelled("b")).dimensions([packed(1, 0)])])]).dimensions([packed(3, 0)])),
     "union soft { bit a, b[1:0]; } [3:0]"),
    (lambda t: t.enumtype(lambda e: e.members([lambda m: m.name(spelled("A")).value(SV.literal(1))])), "enum {A = 1}"),
    (lambda t: t.namedtype(lambda n: n.name(lambda i: i.parameterizedname(lambda q: q.name(spelled("c")).parameters([
        SV.literal(8), lambda p: p.integeratomtype(lambda a: a.keyword("int")), lambda p: p.namedconnection(lambda c: c.name(spelled("T")).implicit(True))])))),
     "c #(8, int, .T)"),
    (lambda t: t.namedtype(lambda n: n.name(lambda i: i.parameterizedname(lambda q: q.name(spelled("c"))))), "c"),  # no parameters
    (lambda t: t.literal(lambda r: r.value(lambda p: p.as_scalar(8))), "8"),  # Types lists Expressions' kinds
]
for spec, source in types:
    assert typetext(spec) == source, (source, typetext(spec))
# Invalid syntax is written as written, wherever it stands, by the dialect whose union holds it; one with no text has
# nothing to print, refused by name.
assert svtext(SV.cast(lambda t: t.error(lambda e: e.text("int int")), x)) == "int int'(x)" and typetext(lambda t: t.missing(lambda m: m)) == ""
with raises(ValueError, match="SystemVerilog Types error has no text to print"):
    svtext(SV.cast(lambda t: t.error(lambda e: e), x))
with raises(ValueError, match="SystemVerilog literal has no value to print"):
    svtext(lambda u: u.literal(lambda r: r.problems(["bad"])))
with raises(ValueError, match="SystemVerilog constant has no value to print"):
    svtext(constant(problems=["bad"]))
# A macro usage, the Preprocessor's, printed from its holder.
assert SVText.ToText(term(lambda u: u.macrousage(lambda m: m.name(spelled("W")).arguments("a, b")), SVPP.DIALECT)) == "`W(a, b)"

<a id="DIA-29"></a>
## DIA-29 · SystemVerilog is complete: each of slang's syntax kinds, kept as data, is a kind or a helper of the family's six dialects, or listed as not an expression with its reason

In [ ]:
import json
from pathlib import Path

grammar = json.loads(Path("../../grammars/systemverilog-slang.json").read_text(encoding="utf-8"))  # slang's syntax kinds, as data
DIALECTS = {"Expressions": SV, "Types": SVT, "Patterns": SVP, "Assertions": SVA, "Constraints": SVC, "Preprocessor": SVPP}
EXEMPT = {"Expressions": {"missing"}, "Preprocessor": {"disabledtext", "comment"}}  # no slang syntax kind: a missing token, trivia
assert len(grammar["nodes"]) == len({node["node"] for node in grammar["nodes"]}) == 533  # every syntax kind, once
mapped = {name: set() for name in DIALECTS}
for node in grammar["nodes"]:  # each a kind or a helper of one of the six, or both, or not an expression, with its reason
    ways = {way for way in ("kinds", "helpers", "not") if way in node}
    assert ways and (ways <= {"kinds", "helpers"} or ways == {"not"}) and set(node) == {"node"} | ways, node
    assert "not" not in node or (type(node["not"]) is str and node["not"]), node
    for way in ("kinds", "helpers"):
        for item in node.get(way, ()):
            dialect, _, tag = item.partition(".")
            assert dialect in DIALECTS and tag, node
            mapped[dialect].add(tag)
            if way == "kinds":  # a branch of its dialect's Term
                assert tag in branches(DIALECTS[dialect].DIALECT.union), node
            else:  # a value object schema its dialect registers
                schema = store.registered(f"Expressions.SystemVerilog.{dialect}.{tag}")
                assert schema is not None and not schema.ref and tag.lower() not in branches(DIALECTS[dialect].DIALECT.union), node
# slang's one kind read as either a kind or a helper, where mbse-programs' reader reads it as either
assert [node["node"] for node in grammar["nodes"] if "kinds" in node and "helpers" in node] == ["IdentifierName", "ScopedName"]
for name, module in DIALECTS.items():  # each dialect's kinds and helpers are the ones slang's kinds map to, but those
    own = {*module.TAGS, *module.HELPERS, *(("error", "missing") if name == "Expressions" else ())}  # no syntax kind has
    assert mapped[name] | EXEMPT.get(name, set()) == own, (name, own ^ (mapped[name] | EXEMPT.get(name, set())))

<a id="DIA-30"></a>
## DIA-30 · Every binder refines `Let`: Let's `name`, `value` and `body` first, renamed where the grammar named them otherwise, its role and scopes saying what it binds

In [ ]:
from mbse.Expressions.Cpp.Expressions import Expressions as CPPE
from mbse.Expressions.Patterns.Expressions import Expressions as E
from mbse.Expressions.SystemVerilog.Constraints import Expressions as SVC

# Every kind that binds names refines the framework's `Let`, Let's `name`, `value` and `body` first, renamed to them where
# the grammar named them otherwise; those it has no name or term for left unset, and optional.
LET = store.registered("Expressions.Framework.Let")
COMPREHENSION = ["name", "value", "body", "generators"]
BINDERS = {  # each binder of every dialect, its schema's properties in order: Let's, then its own
    E.DIALECT: {"quantifier": ["name", "value", "body", "quantifier"]},  # its collection Let's value
    M.DIALECT: {"arrayfun": ["name", "value", "body"], "import": ["name", "value", "body"]},  # an array Let's value
    X.DIALECT: {"map": ["name", "value", "body"]},
    P.DIALECT: {P.LISTCOMP: COMPREHENSION, P.SETCOMP: COMPREHENSION, P.GENERATOREXP: COMPREHENSION,  # an element Let's body
                P.DICTCOMP: ["name", "value", "body", "key", "generators"], P.LAMBDA: ["name", "value", "body", "args"],
                P.NAMEDEXPR: ["name", "value", "body", "target"], P.IMPORT: ["name", "value", "body", "module", "alias"],
                P.IMPORTFROM: ["name", "value", "body", "module", "alias"]},
    TS.DIALECT: {"arrowfunctionexpression": ["name", "value", "body", "isAsync", "typeParameters", "params", "returnType"],
                 "functionexpression": ["name", "value", "body", "isAsync", "generator", "id", "typeParameters", "params",
                                        "returnType"],
                 "tsemptybodyfunctionexpression": ["name", "value", "body", "isAsync", "generator", "typeParameters",
                                                   "params", "returnType"],
                 "classexpression": ["name", "value", "body", "decorators", "id", "typeParameters", "superClass",
                                     "superTypeArguments", "implements"]},
    CPPE.DIALECT: {"lambdaexpression": ["name", "value", "body", "captures", "template_parameters", "template_requires",
                                        "attributes", "declarator"],
                   "requiresexpression": ["name", "value", "body", "parameters", "requirements"]},
    SV.DIALECT: {"arraymethodwithexpression": ["name", "value", "body"]},
    SVC.DIALECT: {"foreachconstraint": ["name", "value", "body", "variables"]},
}
for dialect, binders in BINDERS.items():
    union = {b.name: b.schema for b in store.registered(dialect.union).branches}
    for tag, properties in binders.items():
        schema = union[tag]
        assert schema.refines is LET and F.is_binder(schema) and list(schema.properties) == properties, (dialect.name(), tag)
    assert union.get("let", LET) is LET  # a dialect's own let is the framework's
# A binder keeps its role, which evaluation and inference read: a quantifier and an import bind by it, not as a let.
roles = [d.kinds()[tag].role for d, tag in [(E.DIALECT, "quantifier"), (M.DIALECT, "arrayfun"), (M.DIALECT, "import"),
                                            (X.DIALECT, "map"), (P.DIALECT, P.LAMBDA), (P.DIALECT, P.IMPORT)]]
assert roles == [F.QUANTIFIER, F.QUANTIFIER, F.IMPORT, F.QUANTIFIER, F.APPLICATION, F.IMPORT]
# What each binds, as its scopes or its role say: Patterns' quantifier its name in its body, of the collection's items'
# domain, giving a bool; MATLAB's arrayfun and Excel's map their name in their body.
xs, x, y = E.variable("xs"), E.variable("x"), E.variable("y")
quantified = term(E.quantifier("any", "x", xs, x.gt(y)), E.DIALECT)
assert SY.free(quantified, E.DIALECT) == {"xs", "y"} and E.DIALECT.validate(quantified, {"xs", "y"}) == []
assert E.DIALECT.validate(quantified, {"xs"}) == ["body: argument 1: variable 'y' is not bound"]
assert E.DIALECT.infer(quantified, {"xs": D.Anything, "y": D.Anything}).name() == "bool"
mapped = term(M.arrayfun("p", M.identifier("xs"), M.binary("+", M.identifier("p"), M.identifier("q"))), M.DIALECT)
assert SY.free(mapped, M.DIALECT) == {"xs", "q"} and MText.ToText(mapped) == "arrayfun(@(p) p + q, xs)"
lambdas = term(X.map_("p", X.name("xs"), X.infix("+", X.name("p"), X.name("q"))), X.DIALECT)
assert SY.free(lambdas, X.DIALECT) == {"xs", "q"} and XText.ToText(lambdas) == "=MAP(xs, LAMBDA(p, p + q))"
# An import leaves Let's value empty, its body its last argument: printed after its line, and evaluated within it.
imported = term(M.import_("pkg.*", M.identifier("x")), M.DIALECT)
assert F.form(imported).arguments[0] is None and SY.free(imported, M.DIALECT) == {"x"} and MText.ToText(imported) == "import pkg.*\nx"
# Python3: a comprehension's element and a dict comprehension's value are its body, after Let's empty value; a lambda's
# body its parameters bind; a named expression's value; an import's body.
n = P.name
listed = term(P.listcomp(n("p"), P.comprehension("p", n("xs"))), P.DIALECT)
assert F.form(listed).arguments[0] is None and F.name_of(F.form(listed).arguments[1]) == "p"
assert SY.free(listed, P.DIALECT) == {"xs"} and PText.ToText(listed) == "[p for p in xs]"
keyed = term(P.dictcomp(n("k"), n("v"), P.comprehension(P.tuple_(n("k"), n("v")), n("d"))), P.DIALECT)
assert [F.name_of(a) for a in F.form(keyed).arguments[1:3]] == ["v", "k"]  # its value, its body, then its key
assert SY.free(keyed, P.DIALECT) == {"d"} and PText.ToText(keyed) == "{k: v for k, v in d}"
function = term(P.lambda_(P.binop("+", n("a"), n("b")), ("a",)), P.DIALECT)
assert SY.free(function, P.DIALECT) == {"b"} and PText.ToText(function) == "lambda a: a + b"
walrus = term(P.namedexpr("y", n("x")), P.DIALECT)  # it binds in the enclosing scope, which a term does not see
assert SY.free(walrus, P.DIALECT) == {"x"} and F.form(walrus).arguments[1] is None and PText.ToText(walrus) == "(y := x)"
floor = term(P.importfrom("math", "floor", P.call(n("floor"), 2.5)), P.DIALECT)
assert SY.free(floor, P.DIALECT) == set() and PText.ToText(floor) == "from math import floor\nfloor(2.5)"
assert P.DIALECT.validate(floor) == [] and P.DIALECT.validate(term(P.importfrom("math", "floor", n("ceil")), P.DIALECT)) == [
    "body: variable 'ceil' is not bound"]
# TypeScript5: an arrow's expression body is Let's body, its parameters bound within it.
ident = lambda name: lambda u: u.identifier(lambda r: r.name(name))  # noqa: E731
arrow = term(lambda u: u.arrowfunctionexpression(lambda r: r.params([ident("a")]).body(
    lambda b: b.binaryexpression(lambda e: e.left(ident("a")).operator("+").right(ident("b"))))), TS.DIALECT)
assert F.form(arrow).arguments[:2][0] is None and SY.free(arrow, TS.DIALECT) == {"b"}
assert TSText.ToText(arrow) == "(a) => a + b"
# C++: a lambda and a requires expression leave Let's name, value and body unset, a body of statements not part of the term.
captured = term(lambda u: u.lambdaexpression(lambda r: r), CPPE.DIALECT)
assert F.form(captured).arguments[:3] == (None, None, None) and CPPE.DIALECT.validate(captured) == []